# DL Candidate Generation: DSSM

Ноутбук реализует полный DSSM pipeline:

- обучение или переобучение DSSM;
- генерация DSSM candidates;
- ANN retrieval;
- merge с base retrieval;
- подготовка ranker_train / val / test.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import shutil
import time
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterable

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm, trange

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
except ImportError as e:
    raise ImportError(
        "Нужен PyTorch. Для RTX 3090 обычно подходит CUDA wheel, например: "
        "pip install torch --index-url https://download.pytorch.org/whl/cu121"
    ) from e

try:
    import hnswlib
    HAS_HNSWLIB = True
except Exception as e:
    hnswlib = None
    HAS_HNSWLIB = False
    HNSWLIB_IMPORT_ERROR = e

try:
    import faiss
    HAS_FAISS = True
except Exception as e:
    faiss = None
    HAS_FAISS = False
    FAISS_IMPORT_ERROR = e


# GPU performance knobs for RTX 30xx/40xx.
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(160)

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
print("HAS_HNSWLIB:", HAS_HNSWLIB)
print("HAS_FAISS:", HAS_FAISS)
if HAS_FAISS and hasattr(faiss, "get_num_gpus"):
    try:
        print("FAISS GPUs:", faiss.get_num_gpus())
    except Exception:
        pass

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists() or (p / "data" / "datamart").exists() or (p / "data").exists():
            return p
    raise FileNotFoundError("Не найден project root с папкой data")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DATAMART_DIR = DATA_DIR / "datamart"
AGG_DIR = DATAMART_DIR / "agg"
CANDIDATES_DIR = DATAMART_DIR / "candidates"
PROCESSED_DIR = DATA_DIR / "processed" / "dl_candidate_generators_max"
MODELS_DIR = PROJECT_ROOT / "models" / "dl_candidate_generators_max"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

for p in [PROCESSED_DIR, MODELS_DIR, REPORTS_DIR, CANDIDATES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATAMART_DIR:", DATAMART_DIR)
print("CANDIDATES_DIR:", CANDIDATES_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODELS_DIR:", MODELS_DIR)

In [ ]:
@dataclass(frozen=True)
class DLCandidateMaxConfig:
    # Даты должны совпадать с datamart/candidate/reranker ноутбуками.
    ranker_train_context_start: date = date(2025, 4, 18)
    ranker_train_context_end: date = date(2025, 6, 3)
    ranker_train_target_start: date = date(2025, 6, 4)
    ranker_train_target_end: date = date(2025, 6, 17)

    val_context_start: date = date(2025, 4, 18)
    val_context_end: date = date(2025, 6, 17)
    val_target_start: date = date(2025, 6, 18)
    val_target_end: date = date(2025, 7, 1)

    full_context_start: date = date(2025, 5, 2)
    full_context_end: date = date(2025, 7, 1)
    test_target_start: date = date(2025, 7, 2)
    test_target_end: date = date(2025, 7, 15)

    random_state: int = 42
    n_shards: int = 128
    metric_ks: tuple[int, ...] = (20, 50, 100, 200, 300, 700, 900)

    # В этом ноутбуке считаем только DL. Не-DL sources уже в enhanced generator.
    run_dssm: bool = True
    run_sasrec: bool = False  # DSSM first; enable SASRec after DSSM source is validated.

    # Для первого smoke-run можно поставить build_all_split_sources_for_enabled_models=False,
    # но для финального pipeline реранкера нужны ranker_train/val/test sources.
    build_val_sources: bool = True
    build_all_split_sources_for_enabled_models: bool = False  # сначала валидируем val; после проверки поставить True для ranker_train/test

    # Кандидатный output.
    dl_candidates_per_user: int = 300
    ann_oversample: int = 450
    merged_max_candidates_per_user: int = 900
    existing_merged_dir_name: str = "merged"
    merged_with_dl_dir_name: str = "merged_with_dl"
    merge_with_existing_candidates: bool = True
    overwrite_default_merged: bool = False

    # Reuse controls.
    rebuild_sequences: bool = False
    rebuild_dl_models: bool = False
    rebuild_dl_sources: bool = True  # важно: старые dl_dssm candidates могли быть от toy-модели; пересобираем source
    rebuild_ann_indexes: bool = True  # важно: ANN index зависит от эмбеддингов модели, не переиспользуем старый индекс
    rebuild_merged_with_dl: bool = True
    rebuild_aux: bool = False

    # Vocab / sequence controls.
    max_item_vocab: int = 1_200_000
    min_item_weight: float = 1.0
    sequence_max_len: int = 60
    min_sequence_len: int = 2
    max_train_examples: int | None = None       # None = все примеры после max_targets_per_user.
    max_targets_per_user: int = 8
    train_valid_fraction: float = 0.01
    max_valid_examples: int = 50_000

    # Events and weights.
    weight_preset: str = "current"  # current | ecup_soft | hybrid
    weak_max_cap: float = 20.0
    time_decay_enabled: bool = True
    time_decay_tau_days: float = 30.0
    time_decay_min_factor: float = 0.10

    event_weight_page_view: float = 1.0
    event_weight_view_description: float = 2.0
    event_weight_review_view: float = 3.0
    event_weight_favorite: float = 20.0
    event_weight_to_cart: float = 100.0
    order_weight: float = 100.0

    # DSSM на максималках.
    dssm_epochs: int = 6
    dssm_batch_size: int = 8192
    dssm_embedding_dim: int = 192
    dssm_item_id_dim: int = 192
    dssm_catalog_dim: int = 48
    dssm_cat_l0_dim: int = 24
    dssm_user_id_dim: int = 64
    dssm_hidden_dims: tuple[int, ...] = (768, 384, 192)
    dssm_dropout: float = 0.12
    dssm_lr: float = 3e-4
    dssm_weight_decay: float = 1e-5
    dssm_hard_negative_weight: float = 0.0  # default: in-batch negatives only; much faster and usually stronger with large batches.
    dssm_temperature_init: float = 0.07
    dssm_use_user_id_embedding: bool = True
    dssm_use_fclip_projection: bool = True
    # User history tower is intentionally lighter than item tower: otherwise every batch
    # runs the full item MLP for batch_size * sequence_len tokens, which is the main bottleneck.
    dssm_light_history_tower: bool = True
    fclip_project_dim: int = 64
    fclip_random_projection_seed: int = 42

    # SASRec.
    sasrec_epochs: int = 6
    sasrec_batch_size: int = 4096
    sasrec_embedding_dim: int = 256
    sasrec_num_heads: int = 8
    sasrec_num_layers: int = 4
    sasrec_dropout: float = 0.15
    sasrec_lr: float = 5e-4
    sasrec_weight_decay: float = 1e-5
    sasrec_hard_negative_weight: float = 0.35
    sasrec_temperature_init: float = 0.07

    # Hard negatives from existing enhanced candidate pool.
    hard_negatives_per_example: int = 0  # set 8-16 only for a separate hard-negative fine-tuning run.
    hard_negative_pool_per_user: int = 300
    popularity_negative_pool_size: int = 100_000

    # Training mechanics.
    use_amp: bool = True
    compile_model: bool = False  # Windows/notebook часто нестабилен; можно включить на Linux.
    grad_clip_norm: float = 5.0
    num_workers: int = 0          # Для Windows/Jupyter безопаснее 0.
    pin_memory: bool = True
    log_every_steps: int = 50
    eval_every_epoch: bool = False  # full candidate metrics are more important than local sampled validation loss.
    scheduler_pct_start: float = 0.05

    # ANN retrieval.
    ann_backend: str = "hnswlib"  # hnswlib | faiss | torch | auto
    hnsw_m: int = 24
    hnsw_ef_construction: int = 120
    hnsw_ef_search: int = 200
    hnsw_num_threads: int = -1
    hnsw_add_batch_size: int = 50_000  # chunked add_items gives visible progress and avoids long silent HNSW build
    faiss_use_gpu: bool = False
    faiss_index_type: str = "ivf_flat"  # flat | ivf_flat
    faiss_nlist: int = 4096
    faiss_nprobe: int = 64
    faiss_train_sample_size: int = 250_000
    retrieval_user_batch_size: int = 8192
    retrieval_flush_rows: int = 2_000_000
    filter_seen_context_items: bool = False  # e-commerce допускает repeat purchase; не фильтруем историю на этапе retrieval


CFG = DLCandidateMaxConfig()
CFG

In [ ]:
# Minimal path helpers are defined here because this override cell runs before the common helpers cell below.
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def rel(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def quick_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


from dataclasses import replace

# Отдельные директории, чтобы не переиспользовать слабые DSSM checkpoint/source.
PROCESSED_DIR = DATA_DIR / "processed" / "dl_candidate_generators_dssm_final"
MODELS_DIR = PROJECT_ROOT / "models" / "dl_candidate_generators_dssm_final"
ensure_dir(PROCESSED_DIR)
ensure_dir(MODELS_DIR)
ensure_dir(REPORTS_DIR)

# Новая версия: multi-positive DSSM + safe calibrated retrieval.
DL_SUPERVISED_SOURCE = "dl_dssm_final"

# ==============================================================================================
# MAIN RUN CONTROLS
# ==============================================================================================
# Default mode is safe: reuse existing checkpoint and only train if the checkpoint is missing.
# Set FORCE_RETRAIN_DSSM=True only when you intentionally want a full 18+ hour retrain.
FORCE_RETRAIN_DSSM = False
TRAIN_DSSM_IF_MISSING = True
REGENERATE_VAL_DSSM_SOURCE_IF_MISSING = True

TRAINING_CHECKPOINT_SPLIT = "val"
TRAINING_CHECKPOINT_PATH = MODELS_DIR / DL_SUPERVISED_SOURCE / TRAINING_CHECKPOINT_SPLIT / "model.pt"
VAL_DSSM_SOURCE_DIR = CANDIDATES_DIR / DL_SUPERVISED_SOURCE / "val"

RUN_DSSM_TRAINING_STAGE = bool(FORCE_RETRAIN_DSSM or (TRAIN_DSSM_IF_MISSING and not TRAINING_CHECKPOINT_PATH.exists()))
RUN_VAL_DSSM_SOURCE_STAGE = bool(FORCE_RETRAIN_DSSM or RUN_DSSM_TRAINING_STAGE or (REGENERATE_VAL_DSSM_SOURCE_IF_MISSING and not quick_has_parquet(VAL_DSSM_SOURCE_DIR)))

print("FORCE_RETRAIN_DSSM:", FORCE_RETRAIN_DSSM)
print("TRAIN_DSSM_IF_MISSING:", TRAIN_DSSM_IF_MISSING)
print("TRAINING_CHECKPOINT_PATH:", rel(TRAINING_CHECKPOINT_PATH))
print("checkpoint exists:", TRAINING_CHECKPOINT_PATH.exists())
print("val source exists:", quick_has_parquet(VAL_DSSM_SOURCE_DIR))
print("RUN_DSSM_TRAINING_STAGE:", RUN_DSSM_TRAINING_STAGE)
print("RUN_VAL_DSSM_SOURCE_STAGE:", RUN_VAL_DSSM_SOURCE_STAGE)

# Two-stage training controls. Это intentionally outside frozen dataclass.
DSSM_PRETRAIN_EPOCHS = 4
DSSM_FINETUNE_EPOCHS = 5
DSSM_PRETRAIN_LR = 5e-4
DSSM_FINETUNE_LR = 1.2e-4

# Hard negatives в fine-tune оставляем мягкими: в candidate pool могут встречаться неразмеченные positives.
DSSM_PRETRAIN_HARD_NEGATIVE_WEIGHT = 0.0
DSSM_FINETUNE_HARD_NEGATIVE_WEIGHT = 0.08
DSSM_PRETRAIN_MAX_TARGETS_PER_USER = 32
DSSM_FINETUNE_BATCH_SIZE = 4096

# Safe merge: базовый BM25/ALS/CLIP пул не вытесняется DSSM-кандидатами.
BASE_CANDIDATES_KEEP_PER_USER = 700
MERGED_WITH_DSSM_MAX_CANDIDATES_PER_USER = 900

# Candidate calibration после ANN oversample.
# Это не заменяет DSSM-score, а добавляет небольшие production-like priors:
# item popularity и совпадение с пользовательскими категориями из history.
DSSM_CANDIDATE_POP_ALPHA = 0.04
DSSM_CANDIDATE_CAT_L0_ALPHA = 0.08
DSSM_CANDIDATE_CATALOG_ALPHA = 0.03
DSSM_CANDIDATE_RANK_ALPHA = 0.0005

# Popularity correction in contrastive loss.
# Standard in-batch negatives are biased by item exposure/popularity; subtracting log(q(item))
# prevents the model from solving the task by popularity alone and improves long-tail retrieval.
DSSM_POPULARITY_CORRECTION_BETA = 0.05
DSSM_LOGQ_SMOOTHING = 1.0

# Hot/late experiments are intentionally not in the default final path:
# the previous hot+late version regressed validation quality. Keep them for separate ablation only.
USE_HOT_FEATURES_IN_FINAL_DSSM = False
USE_LATE_SCORER_IN_FINAL_DSSM = False

CFG = replace(
    CFG,
    run_dssm=True,
    run_sasrec=False,
    build_val_sources=True,
    build_all_split_sources_for_enabled_models=False,

    # Не переиспользуем старые weak DSSM model/source.
    rebuild_sequences=False,
    rebuild_dl_models=FORCE_RETRAIN_DSSM,
    rebuild_dl_sources=RUN_VAL_DSSM_SOURCE_STAGE,
    rebuild_ann_indexes=RUN_VAL_DSSM_SOURCE_STAGE,
    rebuild_merged_with_dl=True,
    rebuild_aux=False,

    # DSSM retrieval: rich item tower + history tower; user_id отключен для лучшего generalization.
    dssm_epochs=DSSM_FINETUNE_EPOCHS,
    dssm_batch_size=8192,
    dssm_embedding_dim=256,
    dssm_item_id_dim=256,
    dssm_catalog_dim=64,
    dssm_cat_l0_dim=32,
    dssm_user_id_dim=64,
    dssm_hidden_dims=(1024, 512, 256),
    dssm_dropout=0.12,
    dssm_lr=DSSM_FINETUNE_LR,
    dssm_weight_decay=2e-5,
    dssm_temperature_init=0.07,
    dssm_use_user_id_embedding=False,
    dssm_use_fclip_projection=True,
    dssm_light_history_tower=True,
    fclip_project_dim=64,

    # Hard negatives нужны только на fine-tune; дополнительно фильтруются от всех known positives пользователя.
    hard_negatives_per_example=8,
    dssm_hard_negative_weight=DSSM_FINETUNE_HARD_NEGATIVE_WEIGHT,
    hard_negative_pool_per_user=300,

    # Future order targets.
    max_targets_per_user=32,
    train_valid_fraction=0.03,
    max_valid_examples=100_000,

    # ANN/candidate generation. Увеличиваем oversample, затем калибруем candidates.
    dl_candidates_per_user=300,
    ann_oversample=2000,
    merged_max_candidates_per_user=MERGED_WITH_DSSM_MAX_CANDIDATES_PER_USER,
    hnsw_m=32,
    hnsw_ef_construction=180,
    hnsw_ef_search=500,
    hnsw_add_batch_size=50_000,
    retrieval_user_batch_size=4096,
    filter_seen_context_items=False,

    # Best checkpoint по validation loss.
    eval_every_epoch=True,
)

print("PROCESSED_DIR:", rel(PROCESSED_DIR))
print("MODELS_DIR:", rel(MODELS_DIR))
print("DL source:", DL_SUPERVISED_SOURCE)
print("DSSM_PRETRAIN_EPOCHS:", DSSM_PRETRAIN_EPOCHS)
print("DSSM_FINETUNE_EPOCHS:", DSSM_FINETUNE_EPOCHS)
print("CFG.dssm_use_user_id_embedding:", CFG.dssm_use_user_id_embedding)
print("CFG.rebuild_dl_models:", CFG.rebuild_dl_models)
print("CFG.rebuild_dl_sources:", CFG.rebuild_dl_sources)
print("CFG.ann_oversample:", CFG.ann_oversample)
print("DSSM calibration:", {
    "pop": DSSM_CANDIDATE_POP_ALPHA,
    "cat_l0": DSSM_CANDIDATE_CAT_L0_ALPHA,
    "catalog": DSSM_CANDIDATE_CATALOG_ALPHA,
    "rank": DSSM_CANDIDATE_RANK_ALPHA,
})

In [ ]:
def print_dl_source_sanity(source_name: str = "dl_dssm", split_name: str = "val") -> None:
    """Быстрая проверка, что оценивается свежий DSSM source, а не старый кеш."""
    out_dir = CANDIDATES_DIR / source_name / split_name
    meta_path = out_dir / "_meta.json"
    print(f"source dir: {rel(out_dir)}")
    print(f"exists: {out_dir.exists()}")
    print(f"meta exists: {meta_path.exists()}")
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        print("source_name:", meta.get("source_name"))
        print("split_name:", meta.get("split_name"))
        print("ann_backend:", meta.get("ann_backend"))
        print("rows:", meta.get("rows"))
        cfg = meta.get("config", {})
        print("meta.dl_candidates_per_user:", cfg.get("dl_candidates_per_user"))
        print("current.dl_candidates_per_user:", CFG.dl_candidates_per_user)
        print("meta.filter_seen_context_items:", cfg.get("filter_seen_context_items"))
        print("current.filter_seen_context_items:", CFG.filter_seen_context_items)
        if meta.get("ann_backend") == "reused":
            print("WARNING: source was reused; for честной оценки DSSM нужно rebuild_dl_sources=True.")
        if cfg.get("dl_candidates_per_user") != CFG.dl_candidates_per_user:
            print("WARNING: source config differs from current CFG.")
    else:
        print("No source meta yet. It will be created during candidate generation.")

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
    torch.backends.cudnn.benchmark = True


seed_everything(CFG.random_state)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def scan_parquet(path: Path | str, *, hive_partitioning: bool = True) -> pl.LazyFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_dir():
        if not path_has_parquet(path):
            raise FileNotFoundError(f"No parquet files found in {path}")
        return pl.scan_parquet(str(path / "**" / "*.parquet"), hive_partitioning=hive_partitioning)
    return pl.scan_parquet(str(path), hive_partitioning=hive_partitioning)


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return list(lf.collect_schema().names())
    except Exception:
        return list(lf.schema.keys())


def ensure_numeric_cols(lf: pl.LazyFrame, cols: Iterable[str], dtype=pl.Float32) -> pl.LazyFrame:
    existing = set(lazy_columns(lf))
    exprs = [pl.lit(0).cast(dtype).alias(c) for c in cols if c not in existing]
    return lf.with_columns(exprs) if exprs else lf


def clean_output_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return str(path)


def events_path(level: str) -> Path:
    return AGG_DIR / "events" / level


def orders_path(level: str) -> Path:
    return AGG_DIR / "orders" / level


def read_events_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(events_path(level), hive_partitioning=True)
    date_col = "event_date"
    if date_col not in lazy_columns(lf):
        raise ValueError(f"{events_path(level)} must contain {date_col}")
    return lf.filter((pl.col(date_col).cast(pl.Date) >= pl.lit(start)) & (pl.col(date_col).cast(pl.Date) <= pl.lit(end)))


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(orders_path(level), hive_partitioning=True)
    date_col = "created_date"
    if date_col not in lazy_columns(lf):
        raise ValueError(f"{orders_path(level)} must contain {date_col}")
    return lf.filter((pl.col(date_col).cast(pl.Date) >= pl.lit(start)) & (pl.col(date_col).cast(pl.Date) <= pl.lit(end)))


def split_spec(split_name: str) -> tuple[date, date, date, date]:
    if split_name == "ranker_train":
        return CFG.ranker_train_context_start, CFG.ranker_train_context_end, CFG.ranker_train_target_start, CFG.ranker_train_target_end
    if split_name == "val":
        return CFG.val_context_start, CFG.val_context_end, CFG.val_target_start, CFG.val_target_end
    if split_name == "test":
        return CFG.full_context_start, CFG.full_context_end, CFG.test_target_start, CFG.test_target_end
    raise ValueError(f"Unknown split_name={split_name!r}")


def weight_values() -> dict[str, float]:
    if CFG.weight_preset == "ecup_soft":
        return {"page_view": 2.0, "view_description": 3.0, "review_view": 3.0, "favorite": 5.0, "to_cart": 1.0, "order": 5.0}
    if CFG.weight_preset == "hybrid":
        return {"page_view": 1.0, "view_description": 2.0, "review_view": 3.0, "favorite": 10.0, "to_cart": 30.0, "order": 60.0}
    return {
        "page_view": CFG.event_weight_page_view,
        "view_description": CFG.event_weight_view_description,
        "review_view": CFG.event_weight_review_view,
        "favorite": CFG.event_weight_favorite,
        "to_cart": CFG.event_weight_to_cart,
        "order": CFG.order_weight,
    }


def time_decay_expr(date_col: str, context_end: date) -> pl.Expr:
    if not CFG.time_decay_enabled:
        return pl.lit(1.0, dtype=pl.Float32)
    age_days = (pl.lit(context_end) - pl.col(date_col).cast(pl.Date)).dt.total_days().cast(pl.Float32)
    return (-age_days / float(CFG.time_decay_tau_days)).exp().clip(lower_bound=CFG.time_decay_min_factor, upper_bound=1.0).cast(pl.Float32)


def build_interactions_lf(start: date, end: date) -> pl.LazyFrame:
    """Weighted user-item events for DL training. Uses only context window."""
    w = weight_values()

    event_cols = ["cnt_page_view", "cnt_view_description", "cnt_review_view", "cnt_to_cart", "cnt_favorite"]
    e = ensure_numeric_cols(read_events_lf("user_item", start, end), event_cols)
    event_weight = (
        pl.col("cnt_page_view").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["page_view"]
        + pl.col("cnt_view_description").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["view_description"]
        + pl.col("cnt_review_view").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["review_view"]
        + (pl.col("cnt_favorite").fill_null(0) > 0).cast(pl.Float32) * w["favorite"]
        + (pl.col("cnt_to_cart").fill_null(0) > 0).cast(pl.Float32) * w["to_cart"]
    ) * time_decay_expr("event_date", end)

    e = (
        e.with_columns(event_weight.alias("weight"), pl.col("event_date").cast(pl.Date).alias("dt"))
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), "dt", pl.col("weight").cast(pl.Float32))
        .filter(pl.col("weight") >= float(CFG.min_item_weight))
    )

    o = ensure_numeric_cols(read_orders_lf("user_item", start, end), ["cnt_orders"])
    order_weight = (pl.col("cnt_orders").fill_null(0).cast(pl.Float32) * w["order"] * time_decay_expr("created_date", end)).cast(pl.Float32)
    o = (
        o.with_columns(order_weight.alias("weight"), pl.col("created_date").cast(pl.Date).alias("dt"))
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), "dt", pl.col("weight").cast(pl.Float32))
        .filter(pl.col("weight") >= float(CFG.min_item_weight))
    )

    return (
        pl.concat([e, o], how="diagonal_relaxed")
        .group_by(["user_id", "item_id", "dt"])
        .agg(pl.sum("weight").cast(pl.Float32).alias("weight"))
        .filter(pl.col("weight") > 0)
    )


def build_target_gt(start: date, end: date) -> dict[int, list[int]]:
    df = collect_streaming(
        read_orders_lf("user_item", start, end)
        .filter(pl.col("cnt_orders").fill_null(0) > 0)
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64))
        .unique()
        .sort(["user_id", "item_id"])
    )
    result: dict[int, list[int]] = defaultdict(list)
    for u, i in zip(df["user_id"].to_numpy(), df["item_id"].to_numpy()):
        result[int(u)].append(int(i))
    return dict(result)

In [ ]:
def _as_unique_list(values) -> list[int]:
    if values is None:
        return []
    if isinstance(values, set):
        return [int(x) for x in values]
    return list(dict.fromkeys(int(x) for x in values))


def hit_rate_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    return float(any(int(x) in gt for x in pred_items[:k]))


def recall_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    return float(len(gt & set(int(x) for x in pred_items[:k])) / len(gt))


def ndcg_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            dcg += 1.0 / np.log2(rank + 1)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(len(gt), k) + 1))
    return float(dcg / idcg) if idcg else 0.0


def ap_at_k(gt_items, pred_items, k: int | None = None) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    if k is None:
        k = len(pred_items)
    hits = 0
    score = 0.0
    seen = set()
    for rank, item in enumerate(pred_items[:k], start=1):
        item = int(item)
        if item in gt and item not in seen:
            hits += 1
            score += hits / rank
            seen.add(item)
    return float(score / len(gt))


def rr_at_k(gt_items, pred_items, k: int | None = None) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    if k is None:
        k = len(pred_items)
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            return float(1.0 / rank)
    return 0.0


def candidate_files(path: Path) -> list[Path]:
    return sorted(path.rglob("*.parquet")) if path.exists() else []


def candidate_dir_looks_complete(path: Path, *, n_shards: int = CFG.n_shards) -> bool:
    files = candidate_files(path)
    if not files:
        return False
    shards = {
        int(p.parent.name.split("=", 1)[1])
        for p in files
        if p.parent.name.startswith("user_shard=")
        and p.parent.name.split("=", 1)[1].isdigit()
        and p.stat().st_size > 0
    }
    return len(shards) >= min(n_shards, 1)


def scan_candidate_shard_or_none(path: Path, shard_id: int) -> pl.LazyFrame | None:
    shard_dir = path / f"user_shard={shard_id}"
    if not shard_dir.exists() or not path_has_parquet(shard_dir):
        return None
    return scan_parquet(shard_dir, hive_partitioning=False)


def evaluate_candidate_dir(
    candidates_dir: Path,
    ground_truth: dict[int, list[int]],
    *,
    ks: Iterable[int] = CFG.metric_ks,
    rank_col: str = "rank",
    label: str = "model",
    n_shards: int = CFG.n_shards,
) -> pd.DataFrame:
    ks = tuple(int(k) for k in ks)
    sums = {k: defaultdict(float) for k in ks}
    counts = []
    users_eval = 0
    unique_items: set[int] = set()

    for shard_id in trange(n_shards, desc=f"Evaluate {label}"):
        lf = scan_candidate_shard_or_none(candidates_dir, shard_id)
        if lf is None:
            continue
        cols = lazy_columns(lf)
        actual_rank_col = rank_col if rank_col in cols else ("candidate_rank" if "candidate_rank" in cols else "rank")
        part = collect_streaming(
            lf.select(
                pl.col("user_id").cast(pl.Int64),
                pl.col("item_id").cast(pl.Int64),
                pl.col(actual_rank_col).cast(pl.Int32).alias("eval_rank"),
            ).sort(["user_id", "eval_rank", "item_id"])
        )
        if part.is_empty():
            continue
        pdf = part.to_pandas()
        for user_id, group in pdf.groupby("user_id", sort=False):
            user_id = int(user_id)
            gt = ground_truth.get(user_id)
            if not gt:
                continue
            pred = group["item_id"].astype(np.int64).tolist()
            users_eval += 1
            counts.append(len(pred))
            unique_items.update(pred[:max(ks)])
            for k in ks:
                sums[k]["hit_rate"] += hit_rate_at_k(gt, pred, k)
                sums[k]["recall"] += recall_at_k(gt, pred, k)
                sums[k]["ndcg"] += ndcg_at_k(gt, pred, k)
                sums[k]["map"] += ap_at_k(gt, pred, k)
                sums[k]["mrr"] += rr_at_k(gt, pred, k)
        del pdf, part
        gc.collect()

    rows = []
    total_gt = len(ground_truth)
    for k in ks:
        row = {
            "model": label,
            "k": k,
            "users_total_gt": total_gt,
            "users_evaluated": users_eval,
            "coverage_users": users_eval / total_gt if total_gt else np.nan,
            "avg_candidates_per_user": float(np.mean(counts)) if counts else 0.0,
            "p50_candidates_per_user": float(np.percentile(counts, 50)) if counts else 0.0,
            "p95_candidates_per_user": float(np.percentile(counts, 95)) if counts else 0.0,
            "users_with_less_than_k": int(sum(c < k for c in counts)),
            "unique_items": len(unique_items),
        }
        for metric in ["hit_rate", "recall", "ndcg", "map", "mrr"]:
            row[metric] = float(sums[k][metric] / users_eval) if users_eval else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def split_cache_dir(split_name: str) -> Path:
    return PROCESSED_DIR / "sequences" / split_name


def build_item_vocab(interactions: pl.DataFrame, max_items: int) -> tuple[dict[int, int], np.ndarray, np.ndarray]:
    pop = (
        interactions.group_by("item_id")
        .agg(pl.sum("weight").alias("item_weight"))
        .sort(["item_weight", "item_id"], descending=[True, False])
        .head(max_items)
    )
    item_ids = pop["item_id"].cast(pl.Int64).to_numpy()
    weights = pop["item_weight"].cast(pl.Float32).to_numpy()
    item_id_by_idx = np.concatenate([np.array([0], dtype=np.int64), item_ids.astype(np.int64)])
    item_weight_by_idx = np.concatenate([np.array([0.0], dtype=np.float32), weights.astype(np.float32)])
    item2idx = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}
    return item2idx, item_id_by_idx, item_weight_by_idx


def interactions_to_sequences(
    interactions: pl.DataFrame,
    item2idx: dict[int, int],
) -> tuple[np.ndarray, list[np.ndarray], dict[int, np.ndarray], dict[int, set[int]]]:
    item_keys = list(item2idx.keys())
    df = (
        interactions
        .filter(pl.col("item_id").is_in(item_keys))
        .sort(["user_id", "dt", "weight", "item_id"], descending=[False, False, True, False])
        .select("user_id", "item_id", "dt", "weight")
        .to_pandas()
    )
    df["item_idx"] = df["item_id"].map(item2idx).astype(np.int32)

    user_ids: list[int] = []
    seqs: list[np.ndarray] = []
    user_seq: dict[int, np.ndarray] = {}
    seen_item_ids: dict[int, set[int]] = {}

    for user_id, g in tqdm(df.groupby("user_id", sort=False), desc="Build user sequences"):
        # chronological sequence with repeated interactions collapsed to last occurrence per adjacent block
        seq = g["item_idx"].to_numpy(dtype=np.int32)
        item_ids = g["item_id"].to_numpy(dtype=np.int64)
        if len(seq) < CFG.min_sequence_len:
            continue
        user_id = int(user_id)
        user_ids.append(user_id)
        seqs.append(seq)
        user_seq[user_id] = seq
        seen_item_ids[user_id] = set(int(x) for x in item_ids)

    return np.asarray(user_ids, dtype=np.int64), seqs, user_seq, seen_item_ids


def make_examples_from_sequences(
    seqs: list[np.ndarray],
    *,
    max_examples: int | None,
    max_targets_per_user: int,
    seed: int,
) -> np.ndarray:
    examples = []
    for uidx, seq in enumerate(seqs):
        n = len(seq)
        if n < CFG.min_sequence_len:
            continue
        positions = np.arange(1, n, dtype=np.int32)
        if len(positions) > max_targets_per_user:
            # Берем последние позиции: они ближе к production next-item setup.
            positions = positions[-max_targets_per_user:]
        for pos in positions:
            examples.append((uidx, int(pos)))
    if not examples:
        raise RuntimeError("No training examples built from sequences")
    arr = np.asarray(examples, dtype=np.int32)
    rng = np.random.default_rng(seed)
    if max_examples is not None and len(arr) > max_examples:
        keep = rng.choice(len(arr), size=max_examples, replace=False)
        arr = arr[keep]
    rng.shuffle(arr)
    return arr


def load_split_sequences(split_name: str):
    start, end, _, _ = split_spec(split_name)
    cache = split_cache_dir(split_name)
    cache.mkdir(parents=True, exist_ok=True)
    meta_path = cache / "meta.json"
    item_map_path = cache / "item_id_by_idx.npy"
    item_weight_path = cache / "item_weight_by_idx.npy"
    user_ids_path = cache / "user_ids.npy"
    seqs_path = cache / "seqs.npy"
    seen_path = cache / "seen_by_user.json"

    if (
        item_map_path.exists()
        and item_weight_path.exists()
        and user_ids_path.exists()
        and seqs_path.exists()
        and seen_path.exists()
        and meta_path.exists()
        and not CFG.rebuild_sequences
    ):
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        item_id_by_idx = np.load(item_map_path)
        item_weight_by_idx = np.load(item_weight_path)
        seq_user_ids = np.load(user_ids_path)
        seqs_obj = np.load(seqs_path, allow_pickle=True)
        seqs = [np.asarray(x, dtype=np.int32) for x in seqs_obj.tolist()]
        item2idx = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}
        user_seq = {int(u): seqs[i] for i, u in enumerate(seq_user_ids)}
        seen_raw = json.loads(seen_path.read_text(encoding="utf-8"))
        seen_by_user = {int(k): set(int(x) for x in v) for k, v in seen_raw.items()}
        print(f"{split_name}: sequences loaded from cache: users={len(seq_user_ids):,}, items={len(item_id_by_idx)-1:,}")
        return item2idx, item_id_by_idx, item_weight_by_idx, seq_user_ids, seqs, user_seq, seen_by_user, meta

    print(f"{split_name}: building context interactions {start}..{end}")
    interactions = collect_streaming(build_interactions_lf(start, end))
    print(f"{split_name}: interactions rows={interactions.height:,}")

    item2idx, item_id_by_idx, item_weight_by_idx = build_item_vocab(interactions, CFG.max_item_vocab)
    seq_user_ids, seqs, user_seq, seen_by_user = interactions_to_sequences(interactions, item2idx)

    np.save(item_map_path, item_id_by_idx)
    np.save(item_weight_path, item_weight_by_idx)
    np.save(user_ids_path, seq_user_ids)
    np.save(seqs_path, np.asarray(seqs, dtype=object), allow_pickle=True)
    # JSON is large but convenient and portable.
    seen_path.write_text(json.dumps({str(k): list(v) for k, v in seen_by_user.items()}), encoding="utf-8")
    meta = {
        "split": split_name,
        "context_start": str(start),
        "context_end": str(end),
        "users": int(len(seq_user_ids)),
        "items": int(len(item_id_by_idx) - 1),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": asdict(CFG),
    }
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    del interactions
    gc.collect()
    print(f"{split_name}: sequences saved -> {rel(cache)}")
    return item2idx, item_id_by_idx, item_weight_by_idx, seq_user_ids, seqs, user_seq, seen_by_user, meta

In [ ]:
def raw_items_data_path() -> Path | None:
    candidates = [
        RAW_DIR / "ml_ozon_recsys_train_final_apparel_items_data",
        DATA_DIR / "ml_ozon_recsys_train_final_apparel_items_data",
        RAW_DIR / "items",
        DATA_DIR / "items",
    ]
    return next((p for p in candidates if p.exists()), None)


def iter_parquet_files(path: Path) -> list[Path]:
    if path.is_file():
        return [path]
    return sorted(path.rglob("*.parquet"))


def load_item_catalog_mapping() -> pd.DataFrame:
    path = DATAMART_DIR / "mappings" / "item_catalog.parquet"
    if path.exists():
        return pl.read_parquet(path).select(
            pl.col("item_id").cast(pl.Int64),
            pl.col("catalogid").cast(pl.Int64),
        ).unique().to_pandas()

    # Fallback from agg if catalogid is present there.
    for p in [events_path("user_item"), orders_path("user_item")]:
        if path_has_parquet(p):
            lf = scan_parquet(p, hive_partitioning=True)
            if "catalogid" in lazy_columns(lf):
                df = collect_streaming(lf.select(
                    pl.col("item_id").cast(pl.Int64),
                    pl.col("catalogid").cast(pl.Int64),
                ).drop_nulls().unique())
                return df.to_pandas()
    warnings.warn("catalogid mapping not found; item category features will be zero.")
    return pd.DataFrame({"item_id": [], "catalogid": []})


def build_item_side_arrays(item_id_by_idx: np.ndarray, out_dir: Path, split_name: str) -> dict:
    """Build arrays aligned with item_idx.

    Returns:
        catalog_idx_by_item_idx, cat_l0_idx_by_item_idx, optional fclip_reduced_path.
    """
    ensure_dir(out_dir)
    cat_path = out_dir / f"item_side_{split_name}.npz"
    meta_path = out_dir / f"item_side_{split_name}_meta.json"
    if cat_path.exists() and meta_path.exists() and not CFG.rebuild_aux:
        data = np.load(cat_path)
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        return {
            "catalog_idx_by_item_idx": data["catalog_idx_by_item_idx"].astype(np.int64),
            "cat_l0_idx_by_item_idx": data["cat_l0_idx_by_item_idx"].astype(np.int64),
            "num_catalogs": int(meta["num_catalogs"]),
            "num_cat_l0": int(meta["num_cat_l0"]),
        }

    mapping = load_item_catalog_mapping()
    item_to_pos = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}
    catalog_raw = np.zeros(len(item_id_by_idx), dtype=np.int64)
    if not mapping.empty:
        mapping = mapping.dropna()
        for item_id, catalogid in zip(mapping["item_id"].astype(np.int64), mapping["catalogid"].astype(np.int64)):
            pos = item_to_pos.get(int(item_id))
            if pos is not None:
                catalog_raw[pos] = int(catalogid)

    # cat_l0 as coarse category. If catalogid hierarchy differs, this still creates a stable coarse bucket.
    cat_l0_raw = np.where(catalog_raw > 0, catalog_raw // 1000, 0).astype(np.int64)

    unique_catalogs = sorted(int(x) for x in np.unique(catalog_raw) if int(x) > 0)
    unique_l0 = sorted(int(x) for x in np.unique(cat_l0_raw) if int(x) > 0)
    catalog2idx = {c: i + 1 for i, c in enumerate(unique_catalogs)}
    l02idx = {c: i + 1 for i, c in enumerate(unique_l0)}

    catalog_idx = np.zeros_like(catalog_raw, dtype=np.int64)
    cat_l0_idx = np.zeros_like(cat_l0_raw, dtype=np.int64)
    for i in range(1, len(item_id_by_idx)):
        if int(catalog_raw[i]) > 0:
            catalog_idx[i] = catalog2idx.get(int(catalog_raw[i]), 0)
        if int(cat_l0_raw[i]) > 0:
            cat_l0_idx[i] = l02idx.get(int(cat_l0_raw[i]), 0)

    np.savez_compressed(cat_path, catalog_idx_by_item_idx=catalog_idx, cat_l0_idx_by_item_idx=cat_l0_idx)
    meta = {"num_catalogs": len(unique_catalogs) + 1, "num_cat_l0": len(unique_l0) + 1}
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"{split_name}: item category side arrays saved -> {rel(cat_path)}")
    return {
        "catalog_idx_by_item_idx": catalog_idx,
        "cat_l0_idx_by_item_idx": cat_l0_idx,
        "num_catalogs": len(unique_catalogs) + 1,
        "num_cat_l0": len(unique_l0) + 1,
    }


def build_fclip_reduced_matrix(item_id_by_idx: np.ndarray, out_dir: Path, split_name: str) -> tuple[np.ndarray | None, int]:
    """Optional random-projected fclip side features aligned with item_idx.

    Full fclip can be large; random projection to CFG.fclip_project_dim keeps DSSM item tower practical.
    """
    if not CFG.dssm_use_fclip_projection:
        return None, 0

    items_path = raw_items_data_path()
    if items_path is None:
        warnings.warn("Raw item table with fclip_embed not found; DSSM will run without fclip side features.")
        return None, 0

    ensure_dir(out_dir)
    out_path = out_dir / f"fclip_reduced_{split_name}_dim={CFG.fclip_project_dim}.npy"
    meta_path = out_dir / f"fclip_reduced_{split_name}_dim={CFG.fclip_project_dim}.json"
    if out_path.exists() and meta_path.exists() and not CFG.rebuild_aux:
        mat = np.load(out_path, mmap_mode="r")
        return mat, int(mat.shape[1])

    item_to_idx = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}
    target_items = set(item_to_idx.keys())
    dim_in = None
    rng = np.random.default_rng(CFG.fclip_random_projection_seed)
    proj = None

    mat = np.zeros((len(item_id_by_idx), CFG.fclip_project_dim), dtype=np.float32)
    loaded = 0

    for f in tqdm(iter_parquet_files(items_path), desc="Load/project fclip"):
        try:
            df = pl.read_parquet(f, columns=["item_id", "fclip_embed"])
        except Exception:
            continue
        if df.is_empty() or "fclip_embed" not in df.columns:
            continue
        df = df.filter(pl.col("item_id").is_in(target_items))
        if df.is_empty():
            continue
        for item_id, emb in zip(df["item_id"].to_list(), df["fclip_embed"].to_list()):
            idx = item_to_idx.get(int(item_id))
            if idx is None or emb is None:
                continue
            vec = np.asarray(emb, dtype=np.float32)
            if vec.ndim != 1 or vec.size == 0 or not np.isfinite(vec).all():
                continue
            norm = float(np.linalg.norm(vec))
            if norm <= 0:
                continue
            vec = vec / norm
            if dim_in is None:
                dim_in = int(vec.size)
                # Achlioptas/Gaussian random projection; scale keeps variance approximately stable.
                proj = rng.normal(0.0, 1.0 / math.sqrt(CFG.fclip_project_dim), size=(dim_in, CFG.fclip_project_dim)).astype(np.float32)
            if int(vec.size) != int(dim_in):
                continue
            red = vec @ proj
            rn = float(np.linalg.norm(red))
            if rn > 0:
                red = red / rn
            mat[int(idx)] = red.astype(np.float32)
            loaded += 1
        del df
        gc.collect()

    if loaded == 0:
        warnings.warn("No fclip vectors loaded for DSSM; run without fclip side features.")
        return None, 0

    np.save(out_path, mat)
    meta = {"items_loaded": loaded, "dim_in": dim_in, "dim_out": CFG.fclip_project_dim, "built_at_utc": datetime.now(timezone.utc).isoformat()}
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"{split_name}: fclip reduced loaded={loaded:,}, dim={CFG.fclip_project_dim} -> {rel(out_path)}")
    return np.load(out_path, mmap_mode="r"), int(CFG.fclip_project_dim)

In [ ]:
def make_popular_negative_pool(item_weight_by_idx: np.ndarray, pool_size: int) -> np.ndarray:
    idx = np.arange(1, len(item_weight_by_idx), dtype=np.int64)
    weights = item_weight_by_idx[1:]
    if len(idx) == 0:
        return np.array([1], dtype=np.int64)
    order = np.argsort(-weights)[:min(pool_size, len(idx))]
    return idx[order].astype(np.int64)


def build_hard_negative_pool(
    split_name: str,
    item2idx: dict[int, int],
    seen_by_user: dict[int, set[int]],
) -> dict[int, np.ndarray]:
    """Hard negatives from existing non-DL enhanced candidate pool.

    These candidates were generated without target labels, so they are valid hard negatives for training
    inside the context window. If source dir is missing, fallback to popularity negatives is used.
    """
    existing_dir = CANDIDATES_DIR / CFG.existing_merged_dir_name / split_name
    if not candidate_dir_looks_complete(existing_dir):
        warnings.warn(f"Existing merged candidates not found for hard negatives: {existing_dir}")
        return {}

    out_dir = PROCESSED_DIR / "hard_negatives" / split_name
    ensure_dir(out_dir)
    out_path = out_dir / f"hard_negatives_top={CFG.hard_negative_pool_per_user}.npz"
    meta_path = out_dir / f"meta_top={CFG.hard_negative_pool_per_user}.json"
    if out_path.exists() and meta_path.exists() and not CFG.rebuild_aux:
        data = np.load(out_path, allow_pickle=True)
        user_ids = data["user_ids"].astype(np.int64)
        pools_obj = data["pools"]
        pools = {int(u): np.asarray(pools_obj[i], dtype=np.int64) for i, u in enumerate(user_ids)}
        print(f"{split_name}: hard negatives loaded: users={len(pools):,}")
        return pools

    user_ids = []
    pools = []
    for shard_id in trange(CFG.n_shards, desc=f"Build hard negatives {split_name}"):
        lf = scan_candidate_shard_or_none(existing_dir, shard_id)
        if lf is None:
            continue
        cols = lazy_columns(lf)
        rank_col = "candidate_rank" if "candidate_rank" in cols else "rank"
        part = collect_streaming(
            lf.select(
                pl.col("user_id").cast(pl.Int64),
                pl.col("item_id").cast(pl.Int64),
                pl.col(rank_col).cast(pl.Int32).alias("rank"),
            )
            .filter(pl.col("rank") <= int(CFG.hard_negative_pool_per_user))
            .sort(["user_id", "rank", "item_id"])
        )
        if part.is_empty():
            continue
        pdf = part.to_pandas()
        for u, g in pdf.groupby("user_id", sort=False):
            u = int(u)
            seen = seen_by_user.get(u, set())
            arr = []
            for item_id in g["item_id"].astype(np.int64).tolist():
                if int(item_id) in seen:
                    continue
                idx = item2idx.get(int(item_id))
                if idx is not None and idx > 0:
                    arr.append(int(idx))
                if len(arr) >= CFG.hard_negative_pool_per_user:
                    break
            if arr:
                user_ids.append(u)
                pools.append(np.asarray(arr, dtype=np.int64))
        del pdf, part
        gc.collect()

    np.savez_compressed(out_path, user_ids=np.asarray(user_ids, dtype=np.int64), pools=np.asarray(pools, dtype=object))
    meta_path.write_text(json.dumps({"users": len(user_ids), "built_at_utc": datetime.now(timezone.utc).isoformat()}, indent=2), encoding="utf-8")
    print(f"{split_name}: hard negatives saved: users={len(user_ids):,} -> {rel(out_path)}")
    return {int(u): pools[i] for i, u in enumerate(user_ids)}


def split_train_valid_examples(examples: np.ndarray, seed: int) -> tuple[np.ndarray, np.ndarray]:
    if CFG.train_valid_fraction <= 0 or len(examples) < 10_000:
        return examples, examples[:0]
    rng = np.random.default_rng(seed)
    n_val = int(min(max(1, len(examples) * CFG.train_valid_fraction), CFG.max_valid_examples))
    perm = rng.permutation(len(examples))
    val_idx = perm[:n_val]
    train_idx = perm[n_val:]
    return examples[train_idx], examples[val_idx]


class NextItemHardNegDataset(Dataset):
    def __init__(
        self,
        seq_user_ids: np.ndarray,
        seqs: list[np.ndarray],
        examples: np.ndarray,
        max_len: int,
        hard_negative_pool: dict[int, np.ndarray],
        popular_negative_pool: np.ndarray,
        hard_negatives_per_example: int,
        seed: int,
    ):
        self.seq_user_ids = seq_user_ids.astype(np.int64)
        self.seqs = seqs
        self.examples = examples.astype(np.int32)
        self.max_len = int(max_len)
        self.hard_negative_pool = hard_negative_pool
        self.popular_negative_pool = np.asarray(popular_negative_pool, dtype=np.int64)
        self.hard_negatives_per_example = int(hard_negatives_per_example)
        self.seed = int(seed)

    def __len__(self) -> int:
        return len(self.examples)

    def _sample_negatives(self, user_id: int, pos_item: int, idx: int) -> np.ndarray:
        # Production DSSM usually relies on large-batch in-batch negatives.
        # Local hard negatives are optional fine-tuning; keeping them at 0 makes training feasible.
        if self.hard_negatives_per_example <= 0:
            return np.empty(0, dtype=np.int64)
        rng = np.random.default_rng(self.seed + int(idx) * 9973)
        pool = self.hard_negative_pool.get(int(user_id))
        if pool is None or len(pool) == 0:
            pool = self.popular_negative_pool
        pool = np.asarray(pool, dtype=np.int64)
        pool = pool[pool != int(pos_item)]
        if len(pool) == 0:
            pool = self.popular_negative_pool
            pool = pool[pool != int(pos_item)]
        if len(pool) == 0:
            return np.ones(self.hard_negatives_per_example, dtype=np.int64)
        replace = len(pool) < self.hard_negatives_per_example
        return rng.choice(pool, size=self.hard_negatives_per_example, replace=replace).astype(np.int64)

    def __getitem__(self, idx: int):
        uidx, pos = self.examples[idx]
        uidx = int(uidx)
        pos = int(pos)
        seq = self.seqs[uidx]
        hist = seq[max(0, pos - self.max_len):pos]
        target = int(seq[pos])
        out = np.zeros(self.max_len, dtype=np.int64)
        if len(hist) > 0:
            out[-len(hist):] = hist.astype(np.int64)
        user_id = int(self.seq_user_ids[uidx])
        neg = self._sample_negatives(user_id, target, idx)
        return (
            torch.tensor(uidx, dtype=torch.long),
            torch.from_numpy(out),
            torch.tensor(target, dtype=torch.long),
            torch.from_numpy(neg.astype(np.int64)),
        )

In [ ]:
class MLP(nn.Module):
    def __init__(self, dims: list[int], dropout: float, *, final_norm: bool = True):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(nn.GELU())
                layers.append(nn.LayerNorm(dims[i + 1]))
                layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)
        self.final_norm = final_norm

    def forward(self, x):
        x = self.net(x)
        return F.normalize(x, dim=-1) if self.final_norm else x


class AttentionPooling(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        scores = self.scorer(x).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e4)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)


class DSSMMaxTwoTower(nn.Module):
    """Production-like DSSM / Two-Tower retrieval.

    User tower:
        sequence item representations -> mean pooling + attention pooling + last-item signal + optional user_id embedding.

    Item tower:
        item_id embedding + catalog embedding + coarse catalog embedding + optional reduced fclip side vector.
    """
    def __init__(
        self,
        num_items: int,
        num_users: int,
        catalog_idx_by_item_idx: np.ndarray,
        cat_l0_idx_by_item_idx: np.ndarray,
        num_catalogs: int,
        num_cat_l0: int,
        *,
        fclip_reduced: np.ndarray | None = None,
        output_dim: int = CFG.dssm_embedding_dim,
        dropout: float = CFG.dssm_dropout,
    ):
        super().__init__()
        self.num_items = int(num_items)
        self.output_dim = int(output_dim)
        self.use_user_id = bool(CFG.dssm_use_user_id_embedding)

        self.item_id_embedding = nn.Embedding(num_items, CFG.dssm_item_id_dim, padding_idx=0)
        self.catalog_embedding = nn.Embedding(max(1, num_catalogs), CFG.dssm_catalog_dim, padding_idx=0)
        self.cat_l0_embedding = nn.Embedding(max(1, num_cat_l0), CFG.dssm_cat_l0_dim, padding_idx=0)

        self.register_buffer("catalog_idx_by_item_idx", torch.as_tensor(catalog_idx_by_item_idx, dtype=torch.long), persistent=False)
        self.register_buffer("cat_l0_idx_by_item_idx", torch.as_tensor(cat_l0_idx_by_item_idx, dtype=torch.long), persistent=False)

        clip_dim = 0
        if fclip_reduced is not None:
            # Store fclip side table as a non-persistent buffer.
            # This avoids saving/loading a huge frozen table in model checkpoints and removes long silent torch.load/load_state_dict stalls.
            clip_arr = np.array(fclip_reduced, dtype=np.float32, copy=True)
            clip_dim = int(clip_arr.shape[1])
            self.register_buffer("fclip_table", torch.from_numpy(clip_arr), persistent=False)
        else:
            self.register_buffer("fclip_table", torch.empty(0, 0, dtype=torch.float32), persistent=False)

        item_input_dim = CFG.dssm_item_id_dim + CFG.dssm_catalog_dim + CFG.dssm_cat_l0_dim + clip_dim
        self.item_base = nn.Sequential(
            nn.Linear(item_input_dim, CFG.dssm_hidden_dims[0]),
            nn.GELU(),
            nn.LayerNorm(CFG.dssm_hidden_dims[0]),
            nn.Dropout(dropout),
            nn.Linear(CFG.dssm_hidden_dims[0], output_dim),
            nn.LayerNorm(output_dim),
        )
        self.item_tower = MLP([output_dim, *CFG.dssm_hidden_dims[1:], output_dim], dropout, final_norm=True)

        # A light history tower is the key speed fix. The item tower can stay rich for retrieval,
        # but user history pooling should not call the full item MLP for every token.
        history_input_dim = CFG.dssm_item_id_dim + CFG.dssm_catalog_dim + CFG.dssm_cat_l0_dim
        self.history_project = nn.Sequential(
            nn.Linear(history_input_dim, output_dim),
            nn.GELU(),
            nn.LayerNorm(output_dim),
            nn.Dropout(dropout),
        )

        user_input_dim = output_dim * 3
        if self.use_user_id:
            self.user_embedding = nn.Embedding(num_users, CFG.dssm_user_id_dim)
            user_input_dim += CFG.dssm_user_id_dim
        else:
            self.user_embedding = None

        self.attention_pool = AttentionPooling(output_dim, max(64, output_dim // 2), dropout)
        self.user_tower = MLP([user_input_dim, *CFG.dssm_hidden_dims, output_dim], dropout, final_norm=True)
        self.temperature = nn.Parameter(torch.tensor(float(CFG.dssm_temperature_init)))

    def _item_base_features(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_items - 1)
        cat = self.catalog_idx_by_item_idx[item_idx]
        l0 = self.cat_l0_idx_by_item_idx[item_idx]
        parts = [
            self.item_id_embedding(item_idx),
            self.catalog_embedding(cat.clamp(0, self.catalog_embedding.num_embeddings - 1)),
            self.cat_l0_embedding(l0.clamp(0, self.cat_l0_embedding.num_embeddings - 1)),
        ]
        if self.fclip_table.numel() > 0:
            parts.append(self.fclip_table[item_idx])
        x = torch.cat(parts, dim=-1)
        return self.item_base(x)

    def _history_token_features(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_items - 1)
        cat = self.catalog_idx_by_item_idx[item_idx]
        l0 = self.cat_l0_idx_by_item_idx[item_idx]
        x = torch.cat([
            self.item_id_embedding(item_idx),
            self.catalog_embedding(cat.clamp(0, self.catalog_embedding.num_embeddings - 1)),
            self.cat_l0_embedding(l0.clamp(0, self.cat_l0_embedding.num_embeddings - 1)),
        ], dim=-1)
        return self.history_project(x)

    def encode_item_idx(self, item_idx: torch.Tensor) -> torch.Tensor:
        z = self._item_base_features(item_idx)
        return self.item_tower(z)

    def encode_user(self, user_idx: torch.Tensor, hist: torch.Tensor) -> torch.Tensor:
        mask = hist.ne(0)
        flat = hist.reshape(-1)
        if CFG.dssm_light_history_tower:
            tok = self._history_token_features(flat).reshape(hist.size(0), hist.size(1), -1)
        else:
            tok = self._item_base_features(flat).reshape(hist.size(0), hist.size(1), -1)
        mask_f = mask.unsqueeze(-1).float()
        mean = (tok * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
        attn = self.attention_pool(tok, mask)

        lengths = mask.sum(dim=1).clamp_min(1)
        last_pos = (lengths - 1).view(-1, 1, 1).expand(-1, 1, tok.size(-1))
        last = tok.gather(1, last_pos).squeeze(1)

        parts = [mean, attn, last]
        if self.user_embedding is not None:
            parts.append(self.user_embedding(user_idx.clamp(0, self.user_embedding.num_embeddings - 1)))
        return self.user_tower(torch.cat(parts, dim=-1))

    def logits(self, user_idx: torch.Tensor, hist: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        u = self.encode_user(user_idx, hist)
        v = self.encode_item_idx(item_idx)
        temp = self.temperature.clamp(0.02, 0.20)
        return (u @ v.T) / temp


class SASRecRetrieval(nn.Module):
    def __init__(self, num_items: int, emb_dim: int, max_len: int, num_heads: int, num_layers: int, dropout: float):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, emb_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, emb_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.max_len = max_len
        self.temperature = nn.Parameter(torch.tensor(float(CFG.sasrec_temperature_init)))

    def encode_user(self, user_idx: torch.Tensor, hist: torch.Tensor) -> torch.Tensor:
        bsz, seq_len = hist.shape
        pos = torch.arange(seq_len, device=hist.device).unsqueeze(0).expand(bsz, -1)
        x = self.item_embedding(hist) + self.pos_embedding(pos)
        x = self.dropout(x)
        padding_mask = hist.eq(0)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=hist.device, dtype=torch.bool), diagonal=1)
        h = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask)
        h = self.norm(h)
        lengths = hist.ne(0).sum(dim=1).clamp_min(1)
        idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, h.size(-1))
        z = h.gather(1, idx).squeeze(1)
        return F.normalize(z, dim=-1)

    def encode_item_idx(self, item_idx: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.item_embedding(item_idx), dim=-1)

    def logits(self, user_idx: torch.Tensor, hist: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        u = self.encode_user(user_idx, hist)
        v = self.encode_item_idx(item_idx)
        temp = self.temperature.clamp(0.02, 0.20)
        return (u @ v.T) / temp

In [ ]:
def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=CFG.use_amp and torch.cuda.is_available())
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=CFG.use_amp and torch.cuda.is_available())


class autocast_context:
    def __enter__(self):
        try:
            self.ctx = torch.amp.autocast("cuda", enabled=CFG.use_amp and torch.cuda.is_available())
        except Exception:
            self.ctx = torch.cuda.amp.autocast(enabled=CFG.use_amp and torch.cuda.is_available())
        return self.ctx.__enter__()

    def __exit__(self, exc_type, exc_val, exc_tb):
        return self.ctx.__exit__(exc_type, exc_val, exc_tb)


def retrieval_losses(
    model: nn.Module,
    user_idx: torch.Tensor,
    hist: torch.Tensor,
    pos: torch.Tensor,
    neg: torch.Tensor,
    hard_negative_weight: float,
) -> tuple[torch.Tensor, dict]:
    u = model.encode_user(user_idx, hist)
    pos_v = model.encode_item_idx(pos)
    temp = model.temperature.clamp(0.02, 0.20)

    # In-batch softmax: positive item for each row, other positives in batch as negatives.
    logits_inbatch = (u @ pos_v.T) / temp
    labels = torch.arange(u.size(0), device=u.device)
    loss_inbatch = F.cross_entropy(logits_inbatch, labels)

    # Optional hard-negative local softmax. It is expensive because it encodes B * H negative items.
    # Default optimized DSSM uses only large-batch in-batch negatives; enable H=8..16 only for fine-tuning.
    loss_hard = torch.zeros((), dtype=loss_inbatch.dtype, device=loss_inbatch.device)
    if float(hard_negative_weight) > 0 and neg is not None and neg.numel() > 0 and neg.ndim == 2 and neg.size(1) > 0:
        bsz, hneg = neg.shape
        neg_flat = neg.reshape(-1)
        neg_v = model.encode_item_idx(neg_flat).reshape(bsz, hneg, -1)
        pos_score = (u * pos_v).sum(dim=-1, keepdim=True)
        neg_score = torch.einsum("bd,bhd->bh", u, neg_v)
        logits_hard = torch.cat([pos_score, neg_score], dim=1) / temp
        labels_hard = torch.zeros(bsz, dtype=torch.long, device=u.device)
        loss_hard = F.cross_entropy(logits_hard, labels_hard)

    loss = loss_inbatch + float(hard_negative_weight) * loss_hard
    metrics = {
        "loss": float(loss.detach().cpu()),
        "loss_inbatch": float(loss_inbatch.detach().cpu()),
        "loss_hard": float(loss_hard.detach().cpu()),
        "temp": float(temp.detach().cpu()),
    }
    return loss, metrics


@torch.no_grad()
def evaluate_loss(model: nn.Module, dataset: Dataset, *, batch_size: int, hard_negative_weight: float, max_batches: int = 50) -> dict:
    if len(dataset) == 0:
        return {}
    model.eval()
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=CFG.pin_memory and torch.cuda.is_available(),
        drop_last=False,
    )
    vals = defaultdict(list)
    for batch_no, (user_idx, hist, pos, neg) in enumerate(loader):
        if batch_no >= max_batches:
            break
        user_idx = user_idx.to(DEVICE, non_blocking=True)
        hist = hist.to(DEVICE, non_blocking=True)
        pos = pos.to(DEVICE, non_blocking=True)
        neg = neg.to(DEVICE, non_blocking=True)
        with autocast_context():
            _, m = retrieval_losses(model, user_idx, hist, pos, neg, hard_negative_weight)
        for k, v in m.items():
            vals[k].append(v)
    return {f"valid_{k}": float(np.mean(v)) for k, v in vals.items() if v}


def train_retrieval_model(
    model: nn.Module,
    train_dataset: Dataset,
    valid_dataset: Dataset,
    *,
    epochs: int,
    batch_size: int,
    lr: float,
    weight_decay: float,
    hard_negative_weight: float,
    model_name: str,
) -> dict:
    model = model.to(DEVICE)
    if CFG.compile_model and hasattr(torch, "compile"):
        try:
            model = torch.compile(model)
        except Exception as e:
            warnings.warn(f"torch.compile failed, continue without compile: {type(e).__name__}: {e}")

    loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=CFG.pin_memory and torch.cuda.is_available(),
        drop_last=True,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = max(1, epochs * len(loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=total_steps,
        pct_start=CFG.scheduler_pct_start,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=100.0,
    )
    scaler = make_grad_scaler()

    start_time = time.perf_counter()
    history = []
    global_step = 0

    for epoch in range(1, epochs + 1):
        model.train()
        rolling = defaultdict(list)
        pbar = tqdm(loader, desc=f"{model_name} epoch {epoch}/{epochs}")
        for user_idx, hist, pos, neg in pbar:
            user_idx = user_idx.to(DEVICE, non_blocking=True)
            hist = hist.to(DEVICE, non_blocking=True)
            pos = pos.to(DEVICE, non_blocking=True)
            neg = neg.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                loss, m = retrieval_losses(model, user_idx, hist, pos, neg, hard_negative_weight)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            global_step += 1
            for k, v in m.items():
                rolling[k].append(v)
            if global_step % CFG.log_every_steps == 0:
                pbar.set_postfix(
                    loss=np.mean(rolling["loss"][-CFG.log_every_steps:]),
                    hard=np.mean(rolling["loss_hard"][-CFG.log_every_steps:]),
                    temp=np.mean(rolling["temp"][-CFG.log_every_steps:]),
                    lr=scheduler.get_last_lr()[0],
                )

        epoch_row = {"epoch": epoch, **{k: float(np.mean(v)) for k, v in rolling.items() if v}, "lr": scheduler.get_last_lr()[0]}
        if CFG.eval_every_epoch and len(valid_dataset) > 0:
            epoch_row.update(evaluate_loss(model, valid_dataset, batch_size=batch_size, hard_negative_weight=hard_negative_weight))
        history.append(epoch_row)
        print(model_name, epoch_row)

    train_time = time.perf_counter() - start_time
    return {"train_time_sec": float(train_time), "history": history}

In [ ]:
@torch.no_grad()
def compute_item_vectors(model: nn.Module, num_items: int, batch_size: int = 65536) -> np.ndarray:
    model.eval()
    vecs = []
    for start in trange(1, num_items, batch_size, desc="Encode items"):
        end = min(num_items, start + batch_size)
        idx = torch.arange(start, end, device=DEVICE, dtype=torch.long)
        v = model.encode_item_idx(idx).detach().float().cpu().numpy()
        vecs.append(v.astype(np.float32))
    return np.vstack(vecs).astype(np.float32)


@torch.no_grad()
def encode_user_batch(model: nn.Module, user_indices: np.ndarray, hists: np.ndarray, batch_size: int = 8192) -> np.ndarray:
    model.eval()
    out = []
    for start in range(0, len(user_indices), batch_size):
        end = min(len(user_indices), start + batch_size)
        uidx = torch.from_numpy(user_indices[start:end].astype(np.int64)).to(DEVICE)
        hist = torch.from_numpy(hists[start:end].astype(np.int64)).to(DEVICE)
        v = model.encode_user(uidx, hist).detach().float().cpu().numpy()
        out.append(v.astype(np.float32))
    if not out:
        return np.empty((0, 1), dtype=np.float32)
    return np.vstack(out).astype(np.float32)


def resolve_ann_backend() -> str:
    if CFG.ann_backend != "auto":
        return CFG.ann_backend
    if HAS_HNSWLIB:
        return "hnswlib"
    if HAS_FAISS:
        return "faiss"
    if torch.cuda.is_available():
        return "torch"
    return "numpy"


@dataclass
class ANNIndex:
    backend: str
    index: object
    item_vecs: np.ndarray

    def query(self, user_vecs: np.ndarray, topk: int) -> tuple[np.ndarray, np.ndarray]:
        topk = int(min(topk, self.item_vecs.shape[0]))
        if self.backend == "hnswlib":
            labels, dist = self.index.knn_query(np.asarray(user_vecs, dtype=np.float32), k=topk)
            return labels.astype(np.int64), (1.0 - dist).astype(np.float32)
        if self.backend == "faiss":
            scores, labels = self.index.search(np.asarray(user_vecs, dtype=np.float32), topk)
            return labels.astype(np.int64), scores.astype(np.float32)
        if self.backend == "torch":
            return exact_topk_torch(user_vecs, self.item_vecs, topk)
        if self.backend == "numpy":
            return topk_numpy(user_vecs, self.item_vecs, topk)
        raise ValueError(self.backend)


def build_ann_index(item_vecs: np.ndarray, out_dir: Path, source_name: str, split_name: str) -> ANNIndex:
    ensure_dir(out_dir)
    backend = resolve_ann_backend()
    item_vecs = np.asarray(item_vecs, dtype=np.float32)
    # row id 0 corresponds to item_idx=1.
    if backend == "hnswlib":
        if not HAS_HNSWLIB:
            raise ImportError("hnswlib is not installed")
        dim = item_vecs.shape[1]
        index_path = out_dir / f"{source_name}_{split_name}_hnsw.bin"
        idx = hnswlib.Index(space="cosine", dim=dim)
        if index_path.exists() and not CFG.rebuild_ann_indexes:
            idx.load_index(str(index_path), max_elements=item_vecs.shape[0])
            idx.set_ef(max(CFG.hnsw_ef_search, CFG.ann_oversample))
            print(f"HNSW loaded: {rel(index_path)}")
        else:
            idx.init_index(max_elements=item_vecs.shape[0], ef_construction=CFG.hnsw_ef_construction, M=CFG.hnsw_m)
            add_bs = int(getattr(CFG, "hnsw_add_batch_size", 50_000))
            for s in trange(0, item_vecs.shape[0], add_bs, desc=f"Build HNSW {source_name}/{split_name}"):
                e = min(item_vecs.shape[0], s + add_bs)
                idx.add_items(
                    item_vecs[s:e],
                    np.arange(s, e, dtype=np.int64),
                    num_threads=CFG.hnsw_num_threads,
                )
            idx.set_ef(max(CFG.hnsw_ef_search, CFG.ann_oversample))
            idx.save_index(str(index_path))
            print(f"HNSW built: {rel(index_path)}")
        return ANNIndex("hnswlib", idx, item_vecs)

    if backend == "faiss":
        if not HAS_FAISS:
            raise ImportError("faiss is not installed")
        dim = item_vecs.shape[1]
        xb = np.ascontiguousarray(item_vecs.astype(np.float32))
        faiss.normalize_L2(xb)
        index_path = out_dir / f"{source_name}_{split_name}_faiss_{CFG.faiss_index_type}.index"
        if index_path.exists() and not CFG.rebuild_ann_indexes:
            idx = faiss.read_index(str(index_path))
            print(f"FAISS loaded: {rel(index_path)}")
        else:
            if CFG.faiss_index_type == "flat" or xb.shape[0] < max(10_000, CFG.faiss_nlist * 20):
                idx = faiss.IndexFlatIP(dim)
            elif CFG.faiss_index_type == "ivf_flat":
                nlist = int(min(CFG.faiss_nlist, max(1, math.sqrt(xb.shape[0]) * 4)))
                quantizer = faiss.IndexFlatIP(dim)
                idx = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
                sample_n = min(xb.shape[0], CFG.faiss_train_sample_size)
                rng = np.random.default_rng(CFG.random_state)
                train_idx = rng.choice(xb.shape[0], size=sample_n, replace=False) if sample_n < xb.shape[0] else np.arange(xb.shape[0])
                idx.train(np.ascontiguousarray(xb[train_idx]))
                idx.nprobe = int(min(CFG.faiss_nprobe, nlist))
            else:
                raise ValueError(CFG.faiss_index_type)
            idx.add(xb)
            faiss.write_index(idx, str(index_path))
            print(f"FAISS built: {rel(index_path)}")
        if CFG.faiss_use_gpu and hasattr(faiss, "get_num_gpus") and faiss.get_num_gpus() > 0:
            res = faiss.StandardGpuResources()
            idx = faiss.index_cpu_to_gpu(res, 0, idx)
        if hasattr(idx, "nprobe"):
            idx.nprobe = int(min(CFG.faiss_nprobe, getattr(idx, "nlist", CFG.faiss_nprobe)))
        return ANNIndex("faiss", idx, xb)

    if backend == "torch":
        return ANNIndex("torch", None, item_vecs)
    if backend == "numpy":
        return ANNIndex("numpy", None, item_vecs)
    raise ValueError(backend)


def exact_topk_torch(user_vecs: np.ndarray, item_vecs: np.ndarray, topk: int, batch_size: int = 512) -> tuple[np.ndarray, np.ndarray]:
    dev = DEVICE if torch.cuda.is_available() else torch.device("cpu")
    item_t = torch.from_numpy(item_vecs).to(dev).float()
    item_t = F.normalize(item_t, dim=1)
    all_idx, all_scores = [], []
    for start in trange(0, len(user_vecs), batch_size, desc="Torch exact retrieval"):
        u = torch.from_numpy(user_vecs[start:start + batch_size]).to(dev).float()
        u = F.normalize(u, dim=1)
        scores = u @ item_t.T
        vals, idx = torch.topk(scores, k=min(topk, item_t.shape[0]), dim=1)
        all_idx.append(idx.detach().cpu().numpy().astype(np.int64))
        all_scores.append(vals.detach().cpu().numpy().astype(np.float32))
    return np.vstack(all_idx), np.vstack(all_scores)


def topk_numpy(user_vecs: np.ndarray, item_vecs: np.ndarray, topk: int, batch_size: int = 256) -> tuple[np.ndarray, np.ndarray]:
    item = item_vecs / np.maximum(np.linalg.norm(item_vecs, axis=1, keepdims=True), 1e-12)
    out_idx, out_scores = [], []
    for start in trange(0, len(user_vecs), batch_size, desc="Numpy retrieval"):
        u = user_vecs[start:start + batch_size]
        u = u / np.maximum(np.linalg.norm(u, axis=1, keepdims=True), 1e-12)
        scores = u @ item.T
        kk = min(topk, scores.shape[1])
        idx = np.argpartition(-scores, kk - 1, axis=1)[:, :kk]
        vals = np.take_along_axis(scores, idx, axis=1)
        order = np.argsort(-vals, axis=1)
        idx = np.take_along_axis(idx, order, axis=1)
        vals = np.take_along_axis(vals, order, axis=1)
        out_idx.append(idx.astype(np.int64))
        out_scores.append(vals.astype(np.float32))
    return np.vstack(out_idx), np.vstack(out_scores)

In [ ]:
def write_buffer_to_shards(rows: list[tuple], out_dir: Path, source_name: str, part_id: int) -> int:
    if not rows:
        return 0
    df = pd.DataFrame(rows, columns=["user_id", "item_id", "score", "rank"])
    df["source"] = source_name
    df[f"{source_name}_score"] = df["score"].astype("float32")
    df[f"{source_name}_rank"] = df["rank"].astype("int32")
    df[f"in_{source_name}"] = np.int8(1)

    # Precompute f_num__candidate__ features so the existing reranker preserves even new DL columns.
    df[f"f_num__candidate__in_{source_name}"] = np.float32(1.0)
    df[f"f_num__candidate__{source_name}_score"] = df["score"].astype("float32")
    df[f"f_num__candidate__{source_name}_rank"] = df["rank"].astype("float32")
    df[f"f_num__candidate__{source_name}_rr"] = (1.0 / df["rank"].astype("float32")).astype("float32")

    df["user_shard"] = (df["user_id"].astype("int64") % CFG.n_shards).astype("int16")
    written = 0
    for shard_id, part in df.groupby("user_shard", sort=True):
        shard_dir = out_dir / f"user_shard={int(shard_id)}"
        shard_dir.mkdir(parents=True, exist_ok=True)
        out_file = shard_dir / f"{source_name}_part_{part_id:05d}.parquet"
        pl.from_pandas(part.drop(columns=["user_shard"])).write_parquet(out_file, compression="zstd")
        written += len(part)
    return written


def generate_candidates_streaming(
    model: nn.Module,
    seq_user_ids: np.ndarray,
    user_seq: dict[int, np.ndarray],
    seen_by_user: dict[int, set[int]],
    item_id_by_idx: np.ndarray,
    out_dir: Path,
    *,
    source_name: str,
    split_name: str,
) -> dict:
    if out_dir.exists() and CFG.rebuild_dl_sources:
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if candidate_dir_looks_complete(out_dir) and not CFG.rebuild_dl_sources:
        print(f"{source_name}/{split_name}: source reused -> {rel(out_dir)}")
        return {"candidate_inference_time_sec": 0.0, "ann_backend": "reused", "rows": None}

    clean_output_dir(out_dir)
    t0 = time.perf_counter()

    item_vecs = compute_item_vectors(model, len(item_id_by_idx))
    ann = build_ann_index(item_vecs, MODELS_DIR / source_name / split_name, source_name, split_name)

    target_users = seq_user_ids
    # For val/ranker_train evaluate and train only users with target GT, but source output for reranker should include all warm users.
    # Existing candidate pool covers cold users; DL source is a warm-user source.

    rows_buffer: list[tuple] = []
    part_id = 0
    total_rows = 0

    for start in trange(0, len(target_users), CFG.retrieval_user_batch_size, desc=f"Generate {source_name}/{split_name}"):
        users = target_users[start:start + CFG.retrieval_user_batch_size]
        hists = np.zeros((len(users), CFG.sequence_max_len), dtype=np.int64)
        user_indices = np.zeros(len(users), dtype=np.int64)
        keep = []
        for j, user_id in enumerate(users):
            seq = user_seq.get(int(user_id))
            if seq is None or len(seq) == 0:
                continue
            hist = seq[-CFG.sequence_max_len:]
            hists[j, -len(hist):] = hist.astype(np.int64)
            # user index is position in seq_user_ids array by construction.
            user_indices[j] = start + j
            keep.append(j)
        if not keep:
            continue
        keep = np.asarray(keep, dtype=np.int64)
        batch_users = users[keep].astype(np.int64)
        user_vecs = encode_user_batch(model, user_indices[keep], hists[keep], batch_size=CFG.retrieval_user_batch_size)
        top_rows, top_scores = ann.query(user_vecs, CFG.ann_oversample)

        for local_row, user_id in enumerate(batch_users):
            seen = seen_by_user.get(int(user_id), set()) if CFG.filter_seen_context_items else set()
            rank = 1
            for ridx, score in zip(top_rows[local_row], top_scores[local_row]):
                ridx = int(ridx)
                # ANN row 0 corresponds to item_idx 1.
                item_idx = ridx + 1
                if item_idx <= 0 or item_idx >= len(item_id_by_idx):
                    continue
                item_id = int(item_id_by_idx[item_idx])
                if CFG.filter_seen_context_items and item_id in seen:
                    continue
                rows_buffer.append((int(user_id), item_id, float(score), rank))
                rank += 1
                if rank > CFG.dl_candidates_per_user:
                    break

        if len(rows_buffer) >= CFG.retrieval_flush_rows:
            total_rows += write_buffer_to_shards(rows_buffer, out_dir, source_name, part_id)
            rows_buffer.clear()
            part_id += 1
            gc.collect()

    if rows_buffer:
        total_rows += write_buffer_to_shards(rows_buffer, out_dir, source_name, part_id)
        rows_buffer.clear()

    infer_time = time.perf_counter() - t0
    meta = {
        "source_name": source_name,
        "split_name": split_name,
        "rows": int(total_rows),
        "ann_backend": ann.backend,
        "candidate_inference_time_sec": float(infer_time),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": asdict(CFG),
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"{source_name}/{split_name}: rows={total_rows:,}, time={infer_time:.1f}s -> {rel(out_dir)}")
    del item_vecs, ann
    gc.collect()
    return {"candidate_inference_time_sec": float(infer_time), "ann_backend": meta["ann_backend"], "rows": int(total_rows)}

In [ ]:
def prepare_split_training_data(split_name: str):
    item2idx, item_id_by_idx, item_weight_by_idx, seq_user_ids, seqs, user_seq, seen_by_user, meta = load_split_sequences(split_name)
    examples = make_examples_from_sequences(
        seqs,
        max_examples=CFG.max_train_examples,
        max_targets_per_user=CFG.max_targets_per_user,
        seed=CFG.random_state,
    )
    train_examples, valid_examples = split_train_valid_examples(examples, CFG.random_state)
    hard_pool = build_hard_negative_pool(split_name, item2idx, seen_by_user)
    popular_pool = make_popular_negative_pool(item_weight_by_idx, CFG.popularity_negative_pool_size)

    train_ds = NextItemHardNegDataset(
        seq_user_ids, seqs, train_examples, CFG.sequence_max_len, hard_pool, popular_pool,
        CFG.hard_negatives_per_example, CFG.random_state,
    )
    valid_ds = NextItemHardNegDataset(
        seq_user_ids, seqs, valid_examples, CFG.sequence_max_len, hard_pool, popular_pool,
        CFG.hard_negatives_per_example, CFG.random_state + 13,
    )
    side = build_item_side_arrays(item_id_by_idx, PROCESSED_DIR / "item_side", split_name)
    fclip, fclip_dim = build_fclip_reduced_matrix(item_id_by_idx, PROCESSED_DIR / "item_side", split_name)
    side["fclip_reduced"] = fclip
    side["fclip_dim"] = fclip_dim
    return {
        "item2idx": item2idx,
        "item_id_by_idx": item_id_by_idx,
        "item_weight_by_idx": item_weight_by_idx,
        "seq_user_ids": seq_user_ids,
        "seqs": seqs,
        "user_seq": user_seq,
        "seen_by_user": seen_by_user,
        "meta": meta,
        "train_ds": train_ds,
        "valid_ds": valid_ds,
        "side": side,
    }


def init_model(model_kind: str, data: dict) -> tuple[nn.Module, dict]:
    num_items = len(data["item_id_by_idx"])
    num_users = len(data["seq_user_ids"])

    if model_kind == "dssm":
        side = data["side"]
        model = DSSMMaxTwoTower(
            num_items=num_items,
            num_users=num_users,
            catalog_idx_by_item_idx=side["catalog_idx_by_item_idx"],
            cat_l0_idx_by_item_idx=side["cat_l0_idx_by_item_idx"],
            num_catalogs=side["num_catalogs"],
            num_cat_l0=side["num_cat_l0"],
            fclip_reduced=side["fclip_reduced"],
            output_dim=CFG.dssm_embedding_dim,
            dropout=CFG.dssm_dropout,
        )
        hparams = {
            "embedding_dim": CFG.dssm_embedding_dim,
            "item_id_dim": CFG.dssm_item_id_dim,
            "catalog_dim": CFG.dssm_catalog_dim,
            "cat_l0_dim": CFG.dssm_cat_l0_dim,
            "user_id_dim": CFG.dssm_user_id_dim if CFG.dssm_use_user_id_embedding else 0,
            "hidden_dims": list(CFG.dssm_hidden_dims),
            "dropout": CFG.dssm_dropout,
            "epochs": CFG.dssm_epochs,
            "batch_size": CFG.dssm_batch_size,
            "lr": CFG.dssm_lr,
            "weight_decay": CFG.dssm_weight_decay,
            "hard_negatives_per_example": CFG.hard_negatives_per_example,
            "hard_negative_weight": CFG.dssm_hard_negative_weight,
            "fclip_dim": side["fclip_dim"],
            "light_history_tower": CFG.dssm_light_history_tower,
            "training_mode": "large_batch_inbatch_negatives",
        }
        return model, hparams

    if model_kind == "sasrec":
        model = SASRecRetrieval(
            num_items=num_items,
            emb_dim=CFG.sasrec_embedding_dim,
            max_len=CFG.sequence_max_len,
            num_heads=CFG.sasrec_num_heads,
            num_layers=CFG.sasrec_num_layers,
            dropout=CFG.sasrec_dropout,
        )
        hparams = {
            "embedding_dim": CFG.sasrec_embedding_dim,
            "layers": CFG.sasrec_num_layers,
            "heads": CFG.sasrec_num_heads,
            "dropout": CFG.sasrec_dropout,
            "epochs": CFG.sasrec_epochs,
            "batch_size": CFG.sasrec_batch_size,
            "lr": CFG.sasrec_lr,
            "weight_decay": CFG.sasrec_weight_decay,
            "hard_negatives_per_example": CFG.hard_negatives_per_example,
            "hard_negative_weight": CFG.sasrec_hard_negative_weight,
        }
        return model, hparams

    raise ValueError(model_kind)


def train_and_generate_source(split_name: str, source_name: str, model_kind: str, *, build_source: bool) -> dict:
    model_dir = MODELS_DIR / source_name / split_name
    model_path = model_dir / "model.pt"
    meta_path = model_dir / "run_meta.json"
    out_dir = CANDIDATES_DIR / source_name / split_name

    print(f"{source_name}/{split_name}: [stage] prepare split data")
    data = prepare_split_training_data(split_name)
    num_items = len(data["item_id_by_idx"])
    train_examples = len(data["train_ds"])
    print(f"{source_name}/{split_name}: users={len(data['seq_user_ids']):,}, items={num_items:,}, train_examples={train_examples:,}")

    if model_path.exists() and meta_path.exists() and not CFG.rebuild_dl_models:
        size_mb = model_path.stat().st_size / 1024 / 1024
        print(f"{source_name}/{split_name}: [stage] load checkpoint {rel(model_path)} ({size_mb:.1f} MB)")
        ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
        print(f"{source_name}/{split_name}: [stage] init model")
        model, hparams = init_model(model_kind, data)
        state = ckpt["model_state_dict"]
        # Old checkpoints may contain the huge frozen fclip_embedding.weight.
        # In this version fclip is a non-persistent side buffer rebuilt from item metadata, so skip stale/heavy key.
        if "fclip_embedding.weight" in state:
            state = {k: v for k, v in state.items() if k != "fclip_embedding.weight"}
            print(f"{source_name}/{split_name}: skipped stale fclip_embedding.weight from checkpoint")
        print(f"{source_name}/{split_name}: [stage] load state_dict")
        load_res = model.load_state_dict(state, strict=False)
        missing = [k for k in load_res.missing_keys if not k.startswith("fclip_table")]
        unexpected = [k for k in load_res.unexpected_keys if k != "fclip_embedding.weight"]
        if missing or unexpected:
            print(f"{source_name}/{split_name}: load_state_dict missing={missing[:5]}, unexpected={unexpected[:5]}")
        train_info = ckpt.get("train_info", {"train_time_sec": 0.0, "history": []})
        print(f"{source_name}/{split_name}: model reused -> {rel(model_path)}")
    else:
        print(f"{source_name}/{split_name}: [stage] init model for training")
        model, hparams = init_model(model_kind, data)
        if model_kind == "dssm":
            train_info = train_retrieval_model(
                model, data["train_ds"], data["valid_ds"],
                epochs=CFG.dssm_epochs,
                batch_size=CFG.dssm_batch_size,
                lr=CFG.dssm_lr,
                weight_decay=CFG.dssm_weight_decay,
                hard_negative_weight=CFG.dssm_hard_negative_weight,
                model_name=f"{source_name}/{split_name}",
            )
        elif model_kind == "sasrec":
            train_info = train_retrieval_model(
                model, data["train_ds"], data["valid_ds"],
                epochs=CFG.sasrec_epochs,
                batch_size=CFG.sasrec_batch_size,
                lr=CFG.sasrec_lr,
                weight_decay=CFG.sasrec_weight_decay,
                hard_negative_weight=CFG.sasrec_hard_negative_weight,
                model_name=f"{source_name}/{split_name}",
            )
        else:
            raise ValueError(model_kind)

        ensure_dir(model_dir)
        torch.save({
            "model_state_dict": model.state_dict(),
            "source_name": source_name,
            "model_kind": model_kind,
            "item_id_by_idx": data["item_id_by_idx"],
            "seq_user_ids": data["seq_user_ids"],
            "config": asdict(CFG),
            "hparams": hparams,
            "train_info": train_info,
        }, model_path)
        meta_path.write_text(json.dumps({
            "source_name": source_name,
            "model_kind": model_kind,
            "split_name": split_name,
            "num_items": int(num_items),
            "num_users": int(len(data["seq_user_ids"])),
            "train_examples": int(train_examples),
            "train_info": train_info,
            "hparams": hparams,
            "built_at_utc": datetime.now(timezone.utc).isoformat(),
        }, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        print(f"{source_name}/{split_name}: model saved -> {rel(model_path)}")

    infer_info = {"candidate_inference_time_sec": None, "ann_backend": None, "rows": None}
    if build_source:
        print(f"{source_name}/{split_name}: [stage] move model to {DEVICE} and generate candidates")
        model = model.to(DEVICE)
        infer_info = generate_candidates_streaming(
            model,
            data["seq_user_ids"],
            data["user_seq"],
            data["seen_by_user"],
            data["item_id_by_idx"],
            out_dir,
            source_name=source_name,
            split_name=split_name,
        )

    stats = {
        "model": source_name,
        "model_kind": model_kind,
        "split": split_name,
        "architecture": model.__class__.__name__,
        "num_items": int(num_items),
        "num_users": int(len(data["seq_user_ids"])),
        "train_examples": int(train_examples),
        "train_time_sec": float(train_info.get("train_time_sec", 0.0)),
        **infer_info,
        "hparams": json.dumps(hparams, ensure_ascii=False),
        "model_path": rel(model_path),
    }

    del model, data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return stats

## Supervised DSSM: order-target retrieval

In [ ]:

# =================================================================================================
# DSSM supervised by future order targets
# =================================================================================================
# Главный фикс относительно предыдущего ноутбука:
#   было: train examples = next interaction inside context sequence
#   стало: train examples = (user context before cutoff -> item ordered in next target window)
#
# Для честной validation:
#   train labels: ranker_train_target = 2025-06-04..2025-06-17
#   val context:  2025-04-18..2025-06-17
#   val labels:   2025-06-18..2025-07-01
#
# Модель учится на future orders до val target и генерирует candidates для val без утечки.

def supervised_context_cache_name(split_name: str, context_start: date, context_end: date) -> str:
    return f"{split_name}_context_{context_start.isoformat()}_{context_end.isoformat()}".replace("-", "")


def build_context_sequences_with_vocab(
    *,
    cache_name: str,
    context_start: date,
    context_end: date,
    item2idx: dict[int, int],
) -> tuple[np.ndarray, list[np.ndarray], dict[int, np.ndarray], dict[int, set[int]]]:
    cache = PROCESSED_DIR / "supervised_context_sequences" / cache_name
    ensure_dir(cache)

    user_ids_path = cache / "user_ids.npy"
    seqs_path = cache / "seqs.npy"
    seen_path = cache / "seen_by_user.json"
    meta_path = cache / "meta.json"

    if (
        user_ids_path.exists()
        and seqs_path.exists()
        and seen_path.exists()
        and meta_path.exists()
        and not CFG.rebuild_sequences
    ):
        user_ids = np.load(user_ids_path)
        seqs_obj = np.load(seqs_path, allow_pickle=True)
        seqs = [np.asarray(x, dtype=np.int32) for x in seqs_obj.tolist()]
        user_seq = {int(u): seqs[i] for i, u in enumerate(user_ids)}
        seen_raw = json.loads(seen_path.read_text(encoding="utf-8"))
        seen_by_user = {int(k): set(int(x) for x in v) for k, v in seen_raw.items()}
        print(f"supervised context reused: {cache_name}, users={len(user_ids):,}")
        return user_ids.astype(np.int64), seqs, user_seq, seen_by_user

    print(f"Build supervised context sequences: {context_start}..{context_end}")
    interactions = collect_streaming(build_interactions_lf(context_start, context_end))
    print(f"context interactions rows={interactions.height:,}")
    user_ids, seqs, user_seq, seen_by_user = interactions_to_sequences(interactions, item2idx)

    np.save(user_ids_path, user_ids)
    np.save(seqs_path, np.asarray(seqs, dtype=object), allow_pickle=True)
    seen_path.write_text(json.dumps({str(k): list(v) for k, v in seen_by_user.items()}), encoding="utf-8")
    meta_path.write_text(json.dumps({
        "cache_name": cache_name,
        "context_start": str(context_start),
        "context_end": str(context_end),
        "users": int(len(user_ids)),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    del interactions
    gc.collect()
    print(f"supervised context saved -> {rel(cache)}")
    return user_ids.astype(np.int64), seqs, user_seq, seen_by_user


def build_order_target_examples(
    *,
    seq_user_ids: np.ndarray,
    context_user_seq: dict[int, np.ndarray],
    target_start: date,
    target_end: date,
    item2idx: dict[int, int],
    max_targets_per_user: int,
    seed: int,
) -> tuple[np.ndarray, dict]:
    target_gt = build_target_gt(target_start, target_end)
    user_idx_by_id = {int(u): idx for idx, u in enumerate(seq_user_ids)}
    rng = np.random.default_rng(seed)

    examples: list[tuple[int, int]] = []
    users_with_target = 0
    skipped_no_user_idx = 0
    skipped_no_context = 0
    skipped_target_not_in_vocab = 0

    for user_id, item_ids in tqdm(target_gt.items(), desc=f"Build supervised order targets {target_start}..{target_end}"):
        user_id = int(user_id)
        uidx = user_idx_by_id.get(user_id)
        if uidx is None:
            skipped_no_user_idx += 1
            continue
        hist = context_user_seq.get(user_id)
        if hist is None or len(hist) == 0:
            skipped_no_context += 1
            continue

        pos = []
        for item_id in item_ids:
            idx = item2idx.get(int(item_id))
            if idx is None or idx <= 0:
                skipped_target_not_in_vocab += 1
                continue
            pos.append(int(idx))

        # Unique positives, preserve order.
        pos = list(dict.fromkeys(pos))
        if not pos:
            continue

        users_with_target += 1
        if len(pos) > max_targets_per_user:
            pos = rng.choice(np.asarray(pos, dtype=np.int32), size=max_targets_per_user, replace=False).astype(np.int32).tolist()

        for item_idx in pos:
            examples.append((int(uidx), int(item_idx)))

    if not examples:
        raise RuntimeError("No supervised order-target examples were built.")

    arr = np.asarray(examples, dtype=np.int32)
    rng.shuffle(arr)

    stats = {
        "target_users_raw": int(len(target_gt)),
        "users_with_train_target": int(users_with_target),
        "examples": int(len(arr)),
        "skipped_no_user_idx": int(skipped_no_user_idx),
        "skipped_no_context": int(skipped_no_context),
        "skipped_target_not_in_vocab": int(skipped_target_not_in_vocab),
        "target_start": str(target_start),
        "target_end": str(target_end),
    }
    print("supervised target stats:", json.dumps(stats, ensure_ascii=False, indent=2))
    return arr, stats


class SupervisedOrderTargetDataset(Dataset):
    def __init__(
        self,
        seq_user_ids: np.ndarray,
        context_user_seq: dict[int, np.ndarray],
        examples: np.ndarray,
        max_len: int,
        hard_negative_pool: dict[int, np.ndarray],
        popular_negative_pool: np.ndarray,
        hard_negatives_per_example: int,
        seed: int,
    ):
        self.seq_user_ids = seq_user_ids.astype(np.int64)
        self.context_user_seq = context_user_seq
        self.examples = examples.astype(np.int32)
        self.max_len = int(max_len)
        self.hard_negative_pool = hard_negative_pool
        self.popular_negative_pool = np.asarray(popular_negative_pool, dtype=np.int64)
        self.hard_negatives_per_example = int(hard_negatives_per_example)
        self.seed = int(seed)

    def __len__(self) -> int:
        return len(self.examples)

    def _sample_negatives(self, user_id: int, pos_item: int, idx: int) -> np.ndarray:
        if self.hard_negatives_per_example <= 0:
            return np.empty(0, dtype=np.int64)
        rng = np.random.default_rng(self.seed + int(idx) * 1000003)
        pool = self.hard_negative_pool.get(int(user_id))
        if pool is None or len(pool) == 0:
            pool = self.popular_negative_pool
        pool = np.asarray(pool, dtype=np.int64)
        pool = pool[pool != int(pos_item)]
        if len(pool) == 0:
            return np.ones(self.hard_negatives_per_example, dtype=np.int64)
        replace = len(pool) < self.hard_negatives_per_example
        return rng.choice(pool, size=self.hard_negatives_per_example, replace=replace).astype(np.int64)

    def __getitem__(self, idx: int):
        uidx, target = self.examples[idx]
        uidx = int(uidx)
        target = int(target)
        user_id = int(self.seq_user_ids[uidx])
        seq = self.context_user_seq.get(user_id)
        if seq is None:
            seq = np.empty(0, dtype=np.int32)
        hist = seq[-self.max_len:]

        out = np.zeros(self.max_len, dtype=np.int64)
        if len(hist) > 0:
            out[-len(hist):] = hist.astype(np.int64)

        neg = self._sample_negatives(user_id, target, idx)
        return (
            torch.tensor(uidx, dtype=torch.long),
            torch.from_numpy(out),
            torch.tensor(target, dtype=torch.long),
            torch.from_numpy(neg.astype(np.int64)),
        )


def supervised_training_window_for_inference_split(split_name: str) -> tuple[date, date, date, date, str]:
    """Return context/target window used to train model for candidate generation at split_name.

    For validation candidates:
        train on ranker_train target only, then evaluate on val target.
    For test candidates:
        train on val target using val context; this is allowed because val target is before test target.
    """
    if split_name in {"val", "ranker_train"}:
        return (
            CFG.ranker_train_context_start,
            CFG.ranker_train_context_end,
            CFG.ranker_train_target_start,
            CFG.ranker_train_target_end,
            "ranker_train",
        )
    if split_name == "test":
        return (
            CFG.val_context_start,
            CFG.val_context_end,
            CFG.val_target_start,
            CFG.val_target_end,
            "val",
        )
    raise ValueError(split_name)


def prepare_split_training_data(split_name: str):
    # Inference vocabulary/context is split-specific: val candidates use val context, test candidates use full context.
    item2idx, item_id_by_idx, item_weight_by_idx, seq_user_ids, seqs, user_seq, seen_by_user, meta = load_split_sequences(split_name)

    train_context_start, train_context_end, train_target_start, train_target_end, hard_negative_split = supervised_training_window_for_inference_split(split_name)
    cache_name = supervised_context_cache_name(split_name, train_context_start, train_context_end)
    train_user_ids, train_seqs, train_user_seq, train_seen_by_user = build_context_sequences_with_vocab(
        cache_name=cache_name,
        context_start=train_context_start,
        context_end=train_context_end,
        item2idx=item2idx,
    )

    examples, target_stats = build_order_target_examples(
        seq_user_ids=seq_user_ids,
        context_user_seq=train_user_seq,
        target_start=train_target_start,
        target_end=train_target_end,
        item2idx=item2idx,
        max_targets_per_user=CFG.max_targets_per_user,
        seed=CFG.random_state,
    )

    train_examples, valid_examples = split_train_valid_examples(examples, CFG.random_state)
    popular_pool = make_popular_negative_pool(item_weight_by_idx, CFG.popularity_negative_pool_size)
    hard_pool = build_hard_negative_pool(hard_negative_split, item2idx, train_seen_by_user)

    train_ds = SupervisedOrderTargetDataset(
        seq_user_ids,
        train_user_seq,
        train_examples,
        CFG.sequence_max_len,
        hard_pool,
        popular_pool,
        CFG.hard_negatives_per_example,
        CFG.random_state,
    )
    valid_ds = SupervisedOrderTargetDataset(
        seq_user_ids,
        train_user_seq,
        valid_examples,
        CFG.sequence_max_len,
        hard_pool,
        popular_pool,
        CFG.hard_negatives_per_example,
        CFG.random_state + 13,
    )

    side = build_item_side_arrays(item_id_by_idx, PROCESSED_DIR / "item_side", split_name)
    fclip, fclip_dim = build_fclip_reduced_matrix(item_id_by_idx, PROCESSED_DIR / "item_side", split_name)
    side["fclip_reduced"] = fclip
    side["fclip_dim"] = fclip_dim

    meta = dict(meta)
    meta["supervised_training"] = {
        "train_context_start": str(train_context_start),
        "train_context_end": str(train_context_end),
        "train_target_start": str(train_target_start),
        "train_target_end": str(train_target_end),
        "target_stats": target_stats,
    }

    return {
        "item2idx": item2idx,
        "item_id_by_idx": item_id_by_idx,
        "item_weight_by_idx": item_weight_by_idx,
        "seq_user_ids": seq_user_ids,
        "seqs": seqs,
        "user_seq": user_seq,                 # inference context for candidate generation
        "seen_by_user": seen_by_user,         # inference seen items for optional filtering
        "meta": meta,
        "train_ds": train_ds,
        "valid_ds": valid_ds,
        "side": side,
    }


def train_and_generate_source(split_name: str, source_name: str, model_kind: str, *, build_source: bool) -> dict:
    model_dir = MODELS_DIR / source_name / split_name
    model_path = model_dir / "model.pt"
    meta_path = model_dir / "run_meta.json"
    out_dir = CANDIDATES_DIR / source_name / split_name

    print(f"{source_name}/{split_name}: [stage] prepare supervised order-target data")
    data = prepare_split_training_data(split_name)
    num_items = len(data["item_id_by_idx"])
    train_examples = len(data["train_ds"])
    print(
        f"{source_name}/{split_name}: users={len(data['seq_user_ids']):,}, "
        f"items={num_items:,}, supervised_train_examples={train_examples:,}"
    )

    if model_path.exists() and meta_path.exists() and not CFG.rebuild_dl_models:
        print(f"{source_name}/{split_name}: [stage] load checkpoint {rel(model_path)}")
        ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
        model, hparams = init_model(model_kind, data)
        state = ckpt["model_state_dict"]
        if "fclip_embedding.weight" in state:
            state = {k: v for k, v in state.items() if k != "fclip_embedding.weight"}
        model.load_state_dict(state, strict=False)
        train_info = ckpt.get("train_info", {"train_time_sec": 0.0, "history": []})
        print(f"{source_name}/{split_name}: model reused -> {rel(model_path)}")
    else:
        print(f"{source_name}/{split_name}: [stage] init model for supervised training")
        model, hparams = init_model(model_kind, data)
        train_info = train_retrieval_model(
            model,
            data["train_ds"],
            data["valid_ds"],
            epochs=CFG.dssm_epochs,
            batch_size=CFG.dssm_batch_size,
            lr=CFG.dssm_lr,
            weight_decay=CFG.dssm_weight_decay,
            hard_negative_weight=CFG.dssm_hard_negative_weight,
            model_name=f"{source_name}/{split_name}",
        )

        ensure_dir(model_dir)
        torch.save({
            "model_state_dict": model.state_dict(),
            "source_name": source_name,
            "model_kind": model_kind,
            "item_id_by_idx": data["item_id_by_idx"],
            "seq_user_ids": data["seq_user_ids"],
            "config": asdict(CFG),
            "hparams": hparams,
            "train_info": train_info,
            "data_meta": data["meta"],
        }, model_path)
        meta_path.write_text(json.dumps({
            "source_name": source_name,
            "model_kind": model_kind,
            "split_name": split_name,
            "num_items": int(num_items),
            "num_users": int(len(data["seq_user_ids"])),
            "supervised_train_examples": int(train_examples),
            "hparams": hparams,
            "train_time_sec": float(train_info["train_time_sec"]),
            "data_meta": data["meta"],
            "built_at_utc": datetime.now(timezone.utc).isoformat(),
        }, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        print(f"{source_name}/{split_name}: model saved -> {rel(model_path)}")

    infer_info = {}
    if build_source:
        print(f"{source_name}/{split_name}: [stage] generate candidates")
        model = model.to(DEVICE)
        infer_info = generate_candidates_streaming(
            model,
            data["seq_user_ids"],
            data["user_seq"],
            data["seen_by_user"],
            data["item_id_by_idx"],
            out_dir,
            source_name=source_name,
            split_name=split_name,
        )

    result = {
        "model": source_name,
        "model_kind": model_kind,
        "split": split_name,
        "architecture": "DSSMTwoTowerSupervisedOrderTargets",
        "num_items": int(num_items),
        "num_users": int(len(data["seq_user_ids"])),
        "train_examples": int(train_examples),
        "train_time_sec": float(train_info.get("train_time_sec", 0.0)),
        "candidate_inference_time_sec": float(infer_info.get("candidate_inference_time_sec", 0.0)),
        "ann_backend": infer_info.get("ann_backend", None),
        "rows": infer_info.get("rows", None),
        "hparams": json.dumps(hparams, ensure_ascii=False),
        "model_path": str(model_path),
    }
    del model, data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


## Training configuration

In [ ]:

# =================================================================================================
# DSSM v4 overrides: fixed last-item pooling, two-stage training, best checkpoint selection.
# This cell intentionally overrides earlier class/function definitions.
# =================================================================================================

class DSSMMaxTwoTower(nn.Module):
    """DSSM / Two-Tower retrieval with fixed right-aligned history pooling.

    Important fix:
    histories are left-padded and right-aligned, so the latest token is always hist[:, -1]
    for non-empty histories. Previous code used lengths - 1 and therefore often selected padding.
    """
    def __init__(
        self,
        num_items: int,
        num_users: int,
        catalog_idx_by_item_idx: np.ndarray,
        cat_l0_idx_by_item_idx: np.ndarray,
        num_catalogs: int,
        num_cat_l0: int,
        *,
        fclip_reduced: np.ndarray | None = None,
        output_dim: int = CFG.dssm_embedding_dim,
        dropout: float = CFG.dssm_dropout,
    ):
        super().__init__()
        self.num_items = int(num_items)
        self.output_dim = int(output_dim)
        self.use_user_id = bool(CFG.dssm_use_user_id_embedding)

        self.item_id_embedding = nn.Embedding(num_items, CFG.dssm_item_id_dim, padding_idx=0)
        self.catalog_embedding = nn.Embedding(max(1, num_catalogs), CFG.dssm_catalog_dim, padding_idx=0)
        self.cat_l0_embedding = nn.Embedding(max(1, num_cat_l0), CFG.dssm_cat_l0_dim, padding_idx=0)

        self.register_buffer("catalog_idx_by_item_idx", torch.as_tensor(catalog_idx_by_item_idx, dtype=torch.long), persistent=False)
        self.register_buffer("cat_l0_idx_by_item_idx", torch.as_tensor(cat_l0_idx_by_item_idx, dtype=torch.long), persistent=False)

        clip_dim = 0
        if fclip_reduced is not None and CFG.dssm_use_fclip_projection:
            clip_arr = np.array(fclip_reduced, dtype=np.float32, copy=True)
            clip_dim = int(clip_arr.shape[1])
            self.register_buffer("fclip_table", torch.from_numpy(clip_arr), persistent=False)
        else:
            self.register_buffer("fclip_table", torch.empty(0, 0, dtype=torch.float32), persistent=False)

        item_input_dim = CFG.dssm_item_id_dim + CFG.dssm_catalog_dim + CFG.dssm_cat_l0_dim + clip_dim
        self.item_base = nn.Sequential(
            nn.Linear(item_input_dim, CFG.dssm_hidden_dims[0]),
            nn.GELU(),
            nn.LayerNorm(CFG.dssm_hidden_dims[0]),
            nn.Dropout(dropout),
            nn.Linear(CFG.dssm_hidden_dims[0], output_dim),
            nn.LayerNorm(output_dim),
        )
        self.item_tower = MLP([output_dim, *CFG.dssm_hidden_dims[1:], output_dim], dropout, final_norm=True)

        history_input_dim = CFG.dssm_item_id_dim + CFG.dssm_catalog_dim + CFG.dssm_cat_l0_dim
        self.history_project = nn.Sequential(
            nn.Linear(history_input_dim, output_dim),
            nn.GELU(),
            nn.LayerNorm(output_dim),
            nn.Dropout(dropout),
        )

        user_input_dim = output_dim * 3
        if self.use_user_id:
            self.user_embedding = nn.Embedding(num_users, CFG.dssm_user_id_dim)
            user_input_dim += CFG.dssm_user_id_dim
        else:
            self.user_embedding = None

        self.attention_pool = AttentionPooling(output_dim, max(64, output_dim // 2), dropout)
        self.user_tower = MLP([user_input_dim, *CFG.dssm_hidden_dims, output_dim], dropout, final_norm=True)
        self.temperature = nn.Parameter(torch.tensor(float(CFG.dssm_temperature_init)))

    def _item_base_features(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_items - 1)
        cat = self.catalog_idx_by_item_idx[item_idx]
        l0 = self.cat_l0_idx_by_item_idx[item_idx]
        parts = [
            self.item_id_embedding(item_idx),
            self.catalog_embedding(cat.clamp(0, self.catalog_embedding.num_embeddings - 1)),
            self.cat_l0_embedding(l0.clamp(0, self.cat_l0_embedding.num_embeddings - 1)),
        ]
        if self.fclip_table.numel() > 0:
            parts.append(self.fclip_table[item_idx])
        return self.item_base(torch.cat(parts, dim=-1))

    def _history_token_features(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_items - 1)
        cat = self.catalog_idx_by_item_idx[item_idx]
        l0 = self.cat_l0_idx_by_item_idx[item_idx]
        x = torch.cat([
            self.item_id_embedding(item_idx),
            self.catalog_embedding(cat.clamp(0, self.catalog_embedding.num_embeddings - 1)),
            self.cat_l0_embedding(l0.clamp(0, self.cat_l0_embedding.num_embeddings - 1)),
        ], dim=-1)
        return self.history_project(x)

    def encode_item_idx(self, item_idx: torch.Tensor) -> torch.Tensor:
        return self.item_tower(self._item_base_features(item_idx))

    def encode_user(self, user_idx: torch.Tensor, hist: torch.Tensor) -> torch.Tensor:
        mask = hist.ne(0)
        flat = hist.reshape(-1)
        if CFG.dssm_light_history_tower:
            tok = self._history_token_features(flat).reshape(hist.size(0), hist.size(1), -1)
        else:
            tok = self._item_base_features(flat).reshape(hist.size(0), hist.size(1), -1)

        mask_f = mask.unsqueeze(-1).float()
        mean = (tok * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
        attn = self.attention_pool(tok, mask)

        # Histories are right-aligned. The latest real token is at the last column for non-empty users.
        # For all-zero histories, mask ensures mean/attn are stable; last will be padding representation.
        last = tok[:, -1, :]

        parts = [mean, attn, last]
        if self.user_embedding is not None:
            parts.append(self.user_embedding(user_idx.clamp(0, self.user_embedding.num_embeddings - 1)))
        return self.user_tower(torch.cat(parts, dim=-1))

    def logits(self, user_idx: torch.Tensor, hist: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        u = self.encode_user(user_idx, hist)
        v = self.encode_item_idx(item_idx)
        temp = self.temperature.clamp(0.02, 0.20)
        return (u @ v.T) / temp


class SASRecRetrieval(nn.Module):
    """SASRec with fixed right-aligned latest-token selection."""
    def __init__(self, num_items: int, emb_dim: int, max_len: int, num_heads: int, num_layers: int, dropout: float):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, emb_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_len, emb_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)
        self.max_len = max_len
        self.temperature = nn.Parameter(torch.tensor(float(CFG.sasrec_temperature_init)))

    def encode_user(self, user_idx: torch.Tensor, hist: torch.Tensor) -> torch.Tensor:
        bsz, seq_len = hist.shape
        pos = torch.arange(seq_len, device=hist.device).unsqueeze(0).expand(bsz, -1)
        x = self.item_embedding(hist) + self.pos_embedding(pos)
        x = self.dropout(x)
        padding_mask = hist.eq(0)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=hist.device, dtype=torch.bool), diagonal=1)
        h = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask)
        h = self.norm(h)
        z = h[:, -1, :]
        return F.normalize(z, dim=-1)

    def encode_item_idx(self, item_idx: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.item_embedding(item_idx), dim=-1)

    def logits(self, user_idx: torch.Tensor, hist: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
        u = self.encode_user(user_idx, hist)
        v = self.encode_item_idx(item_idx)
        temp = self.temperature.clamp(0.02, 0.20)
        return (u @ v.T) / temp


def train_retrieval_model(
    model: nn.Module,
    train_dataset: Dataset,
    valid_dataset: Dataset,
    *,
    epochs: int,
    batch_size: int,
    lr: float,
    weight_decay: float,
    hard_negative_weight: float,
    model_name: str,
) -> dict:
    """Train retrieval model and restore best state by valid_loss when available."""
    model = model.to(DEVICE)
    if CFG.compile_model and hasattr(torch, "compile"):
        try:
            model = torch.compile(model)
        except Exception as e:
            warnings.warn(f"torch.compile failed, continue without compile: {type(e).__name__}: {e}")

    loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=CFG.pin_memory and torch.cuda.is_available(),
        drop_last=True,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = max(1, epochs * len(loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=total_steps,
        pct_start=CFG.scheduler_pct_start,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=100.0,
    )
    scaler = make_grad_scaler()

    start_time = time.perf_counter()
    history = []
    global_step = 0
    best_state = None
    best_metric = float("inf")
    best_epoch = None

    for epoch in range(1, epochs + 1):
        model.train()
        rolling = defaultdict(list)
        pbar = tqdm(loader, desc=f"{model_name} epoch {epoch}/{epochs}")
        for user_idx, hist, pos, neg in pbar:
            user_idx = user_idx.to(DEVICE, non_blocking=True)
            hist = hist.to(DEVICE, non_blocking=True)
            pos = pos.to(DEVICE, non_blocking=True)
            neg = neg.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                loss, m = retrieval_losses(model, user_idx, hist, pos, neg, hard_negative_weight)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            global_step += 1
            for k, v in m.items():
                rolling[k].append(v)
            if global_step % CFG.log_every_steps == 0:
                pbar.set_postfix(
                    loss=np.mean(rolling["loss"][-CFG.log_every_steps:]),
                    hard=np.mean(rolling["loss_hard"][-CFG.log_every_steps:]),
                    temp=np.mean(rolling["temp"][-CFG.log_every_steps:]),
                    lr=scheduler.get_last_lr()[0],
                )

        epoch_row = {"epoch": epoch, **{k: float(np.mean(v)) for k, v in rolling.items() if v}, "lr": scheduler.get_last_lr()[0]}
        if CFG.eval_every_epoch and len(valid_dataset) > 0:
            epoch_row.update(evaluate_loss(model, valid_dataset, batch_size=batch_size, hard_negative_weight=hard_negative_weight))
        history.append(epoch_row)
        print(model_name, epoch_row)

        valid_loss = epoch_row.get("valid_loss")
        selection_metric = float(valid_loss if valid_loss is not None and not np.isnan(valid_loss) else epoch_row.get("loss", float("inf")))
        if selection_metric < best_metric:
            best_metric = selection_metric
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state, strict=True)
        print(f"{model_name}: restored best epoch={best_epoch}, best_selection_loss={best_metric:.6f}")

    train_time = time.perf_counter() - start_time
    return {"train_time_sec": float(train_time), "history": history, "best_epoch": best_epoch, "best_selection_loss": best_metric}


def prepare_interaction_pretrain_data(split_name: str):
    """Stage 1 DSSM pretrain: context interactions -> next interacted item."""
    item2idx, item_id_by_idx, item_weight_by_idx, seq_user_ids, seqs, user_seq, seen_by_user, meta = load_split_sequences(split_name)
    examples = make_examples_from_sequences(
        seqs,
        max_examples=CFG.max_train_examples,
        max_targets_per_user=DSSM_PRETRAIN_MAX_TARGETS_PER_USER,
        seed=CFG.random_state,
    )
    train_examples, valid_examples = split_train_valid_examples(examples, CFG.random_state)
    popular_pool = make_popular_negative_pool(item_weight_by_idx, CFG.popularity_negative_pool_size)
    # For speed and stability, pretrain relies on large-batch in-batch negatives.
    hard_pool = {}

    train_ds = NextItemHardNegDataset(
        seq_user_ids, seqs, train_examples, CFG.sequence_max_len, hard_pool, popular_pool,
        0, CFG.random_state,
    )
    valid_ds = NextItemHardNegDataset(
        seq_user_ids, seqs, valid_examples, CFG.sequence_max_len, hard_pool, popular_pool,
        0, CFG.random_state + 13,
    )
    return {
        "item2idx": item2idx,
        "item_id_by_idx": item_id_by_idx,
        "item_weight_by_idx": item_weight_by_idx,
        "seq_user_ids": seq_user_ids,
        "seqs": seqs,
        "user_seq": user_seq,
        "seen_by_user": seen_by_user,
        "meta": dict(meta, pretrain_examples=int(len(examples))),
        "train_ds": train_ds,
        "valid_ds": valid_ds,
    }


def train_and_generate_source(split_name: str, source_name: str, model_kind: str, *, build_source: bool) -> dict:
    model_dir = MODELS_DIR / source_name / split_name
    model_path = model_dir / "model.pt"
    meta_path = model_dir / "run_meta.json"
    out_dir = CANDIDATES_DIR / source_name / split_name

    print(f"{source_name}/{split_name}: [stage] prepare supervised order-target data")
    finetune_data = prepare_split_training_data(split_name)
    pretrain_data = prepare_interaction_pretrain_data(split_name)
    num_items = len(finetune_data["item_id_by_idx"])
    print(
        f"{source_name}/{split_name}: users={len(finetune_data['seq_user_ids']):,}, "
        f"items={num_items:,}, pretrain_examples={len(pretrain_data['train_ds']):,}, "
        f"finetune_examples={len(finetune_data['train_ds']):,}"
    )

    if model_path.exists() and meta_path.exists() and not CFG.rebuild_dl_models:
        print(f"{source_name}/{split_name}: [stage] load checkpoint {rel(model_path)}")
        ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
        model, hparams = init_model(model_kind, finetune_data)
        state = ckpt["model_state_dict"]
        state = {k: v for k, v in state.items() if k != "fclip_embedding.weight"}
        load_res = model.load_state_dict(state, strict=False)
        if load_res.missing_keys or load_res.unexpected_keys:
            print(f"load_state_dict missing={load_res.missing_keys[:5]}, unexpected={load_res.unexpected_keys[:5]}")
        train_info = ckpt.get("train_info", {"train_time_sec": 0.0, "history": []})
        print(f"{source_name}/{split_name}: model reused -> {rel(model_path)}")
    else:
        print(f"{source_name}/{split_name}: [stage] init model")
        model, hparams = init_model(model_kind, finetune_data)

        print(f"{source_name}/{split_name}: [stage 1/2] interaction pretrain")
        pretrain_info = train_retrieval_model(
            model,
            pretrain_data["train_ds"],
            pretrain_data["valid_ds"],
            epochs=DSSM_PRETRAIN_EPOCHS,
            batch_size=CFG.dssm_batch_size,
            lr=DSSM_PRETRAIN_LR,
            weight_decay=CFG.dssm_weight_decay,
            hard_negative_weight=DSSM_PRETRAIN_HARD_NEGATIVE_WEIGHT,
            model_name=f"{source_name}/{split_name}/pretrain",
        )

        print(f"{source_name}/{split_name}: [stage 2/2] future-order fine-tune")
        finetune_info = train_retrieval_model(
            model,
            finetune_data["train_ds"],
            finetune_data["valid_ds"],
            epochs=DSSM_FINETUNE_EPOCHS,
            batch_size=DSSM_FINETUNE_BATCH_SIZE,
            lr=DSSM_FINETUNE_LR,
            weight_decay=CFG.dssm_weight_decay,
            hard_negative_weight=DSSM_FINETUNE_HARD_NEGATIVE_WEIGHT,
            model_name=f"{source_name}/{split_name}/finetune",
        )

        train_info = {
            "train_time_sec": float(pretrain_info["train_time_sec"] + finetune_info["train_time_sec"]),
            "pretrain": pretrain_info,
            "finetune": finetune_info,
            "history": [
                {"stage": "pretrain", **r} for r in pretrain_info.get("history", [])
            ] + [
                {"stage": "finetune", **r} for r in finetune_info.get("history", [])
            ],
        }

        ensure_dir(model_dir)
        torch.save({
            "model_state_dict": model.state_dict(),
            "source_name": source_name,
            "model_kind": model_kind,
            "item_id_by_idx": finetune_data["item_id_by_idx"],
            "seq_user_ids": finetune_data["seq_user_ids"],
            "config": asdict(CFG),
            "two_stage": {
                "pretrain_epochs": DSSM_PRETRAIN_EPOCHS,
                "finetune_epochs": DSSM_FINETUNE_EPOCHS,
                "pretrain_lr": DSSM_PRETRAIN_LR,
                "finetune_lr": DSSM_FINETUNE_LR,
                "user_id_embedding": CFG.dssm_use_user_id_embedding,
                "last_item_pooling_fix": True,
            },
            "hparams": hparams,
            "train_info": train_info,
            "data_meta": finetune_data["meta"],
        }, model_path)
        meta_path.write_text(json.dumps({
            "source_name": source_name,
            "model_kind": model_kind,
            "split_name": split_name,
            "num_items": int(num_items),
            "num_users": int(len(finetune_data["seq_user_ids"])),
            "pretrain_examples": int(len(pretrain_data["train_ds"])),
            "finetune_examples": int(len(finetune_data["train_ds"])),
            "train_time_sec": float(train_info["train_time_sec"]),
            "hparams": hparams,
            "two_stage": {
                "pretrain_epochs": DSSM_PRETRAIN_EPOCHS,
                "finetune_epochs": DSSM_FINETUNE_EPOCHS,
                "pretrain_lr": DSSM_PRETRAIN_LR,
                "finetune_lr": DSSM_FINETUNE_LR,
                "user_id_embedding": CFG.dssm_use_user_id_embedding,
                "last_item_pooling_fix": True,
                "safe_base_first_merge": True,
            },
            "data_meta": finetune_data["meta"],
            "built_at_utc": datetime.now(timezone.utc).isoformat(),
        }, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        print(f"{source_name}/{split_name}: model saved -> {rel(model_path)}")

    infer_info = {"candidate_inference_time_sec": None, "ann_backend": None, "rows": None}
    if build_source:
        print(f"{source_name}/{split_name}: [stage] generate candidates")
        model = model.to(DEVICE)
        infer_info = generate_candidates_streaming(
            model,
            finetune_data["seq_user_ids"],
            finetune_data["user_seq"],
            finetune_data["seen_by_user"],
            finetune_data["item_id_by_idx"],
            out_dir,
            source_name=source_name,
            split_name=split_name,
        )

    stats = {
        "model": source_name,
        "model_kind": model_kind,
        "split": split_name,
        "architecture": model.__class__.__name__,
        "training_scheme": "interaction_pretrain_then_future_order_finetune",
        "num_items": int(num_items),
        "num_users": int(len(finetune_data["seq_user_ids"])),
        "pretrain_examples": int(len(pretrain_data["train_ds"])),
        "finetune_examples": int(len(finetune_data["train_ds"])),
        "train_time_sec": float(train_info.get("train_time_sec", 0.0)),
        **infer_info,
        "hparams": json.dumps(hparams, ensure_ascii=False),
        "model_path": rel(model_path),
    }

    del model, finetune_data, pretrain_data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return stats


## Multi-positive objective and popularity correction

In [ ]:
# =================================================================================================
# DSSM v5 overrides: multi-positive contrastive loss + false-negative filtering + calibrated retrieval.
# This cell intentionally overrides earlier Dataset/loss/generation definitions.
# =================================================================================================

class SupervisedOrderTargetDataset(Dataset):
    """Future-order dataset with per-user positive filtering for hard negatives.

    The previous version excluded only the current positive item from hard negatives.
    But the existing candidate pool may contain another future order of the same user.
    Pushing such items down as negatives hurts DSSM heavily.
    """
    def __init__(
        self,
        seq_user_ids: np.ndarray,
        context_user_seq: dict[int, np.ndarray],
        examples: np.ndarray,
        max_len: int,
        hard_negative_pool: dict[int, np.ndarray],
        popular_negative_pool: np.ndarray,
        hard_negatives_per_example: int,
        seed: int,
    ):
        self.seq_user_ids = seq_user_ids.astype(np.int64)
        self.context_user_seq = context_user_seq
        self.examples = examples.astype(np.int32)
        self.max_len = int(max_len)
        self.hard_negative_pool = hard_negative_pool
        self.popular_negative_pool = np.asarray(popular_negative_pool, dtype=np.int64)
        self.hard_negatives_per_example = int(hard_negatives_per_example)
        self.seed = int(seed)

        # Known positives inside this dataset split, grouped by raw user_id.
        # Used only to avoid obvious false negatives in local hard-negative softmax.
        pos_by_user: dict[int, set[int]] = defaultdict(set)
        for uidx, target in self.examples:
            user_id = int(self.seq_user_ids[int(uidx)])
            pos_by_user[user_id].add(int(target))
        self.positive_items_by_user = pos_by_user

    def __len__(self) -> int:
        return len(self.examples)

    def _sample_negatives(self, user_id: int, pos_item: int, idx: int) -> np.ndarray:
        if self.hard_negatives_per_example <= 0:
            return np.empty(0, dtype=np.int64)
        rng = np.random.default_rng(self.seed + int(idx) * 1000003)
        pool = self.hard_negative_pool.get(int(user_id))
        if pool is None or len(pool) == 0:
            pool = self.popular_negative_pool
        pool = np.asarray(pool, dtype=np.int64)

        known_pos = self.positive_items_by_user.get(int(user_id), set())
        if known_pos:
            mask = np.fromiter((int(x) not in known_pos for x in pool), dtype=bool, count=len(pool))
            pool = pool[mask]
        else:
            pool = pool[pool != int(pos_item)]

        if len(pool) == 0:
            pool = self.popular_negative_pool
            known_pos = self.positive_items_by_user.get(int(user_id), {int(pos_item)})
            mask = np.fromiter((int(x) not in known_pos for x in pool), dtype=bool, count=len(pool))
            pool = pool[mask]
        if len(pool) == 0:
            return np.ones(self.hard_negatives_per_example, dtype=np.int64)
        replace = len(pool) < self.hard_negatives_per_example
        return rng.choice(pool, size=self.hard_negatives_per_example, replace=replace).astype(np.int64)

    def __getitem__(self, idx: int):
        uidx, target = self.examples[idx]
        uidx = int(uidx)
        target = int(target)
        user_id = int(self.seq_user_ids[uidx])
        seq = self.context_user_seq.get(user_id)
        if seq is None:
            seq = np.empty(0, dtype=np.int32)
        hist = seq[-self.max_len:]

        out = np.zeros(self.max_len, dtype=np.int64)
        if len(hist) > 0:
            out[-len(hist):] = hist.astype(np.int64)

        neg = self._sample_negatives(user_id, target, idx)
        return (
            torch.tensor(uidx, dtype=torch.long),
            torch.from_numpy(out),
            torch.tensor(target, dtype=torch.long),
            torch.from_numpy(neg.astype(np.int64)),
        )


def multipositive_inbatch_loss(
    user_vec: torch.Tensor,
    pos_vec: torch.Tensor,
    user_idx: torch.Tensor,
    pos: torch.Tensor,
    temp: torch.Tensor,
) -> torch.Tensor:
    """Multi-positive in-batch softmax.

    Standard in-batch CE assumes one positive per row and treats every other item in the batch as negative.
    In this dataset each user often has several order targets, so the same user's other positives become
    false negatives. This loss marks all items of the same user in the current batch as positives.

    Numerator: logsumexp over same-user positives in the batch.
    Denominator: logsumexp over all batch items.
    """
    logits = (user_vec @ pos_vec.T) / temp

    same_user = user_idx[:, None].eq(user_idx[None, :])
    # Always preserve diagonal; same_user already includes it, this is defensive.
    eye = torch.eye(logits.size(0), device=logits.device, dtype=torch.bool)
    positive_mask = same_user | eye

    neg_inf = torch.finfo(logits.dtype).min
    numerator = torch.logsumexp(logits.masked_fill(~positive_mask, neg_inf), dim=1)
    denominator = torch.logsumexp(logits, dim=1)
    return -(numerator - denominator).mean()


def retrieval_losses(
    model: nn.Module,
    user_idx: torch.Tensor,
    hist: torch.Tensor,
    pos: torch.Tensor,
    neg: torch.Tensor,
    hard_negative_weight: float,
) -> tuple[torch.Tensor, dict]:
    u = model.encode_user(user_idx, hist)
    pos_v = model.encode_item_idx(pos)
    temp = model.temperature.clamp(0.02, 0.20)

    # Multi-positive in-batch loss: fixes severe false-negative issue for users with multiple targets.
    loss_inbatch = multipositive_inbatch_loss(u, pos_v, user_idx, pos, temp)

    loss_hard = torch.zeros((), dtype=loss_inbatch.dtype, device=loss_inbatch.device)
    if float(hard_negative_weight) > 0 and neg is not None and neg.numel() > 0 and neg.ndim == 2 and neg.size(1) > 0:
        bsz, hneg = neg.shape
        neg_flat = neg.reshape(-1)
        neg_v = model.encode_item_idx(neg_flat).reshape(bsz, hneg, -1)
        pos_score = (u * pos_v).sum(dim=-1, keepdim=True)
        neg_score = torch.einsum("bd,bhd->bh", u, neg_v)
        logits_hard = torch.cat([pos_score, neg_score], dim=1) / temp
        labels_hard = torch.zeros(bsz, dtype=torch.long, device=u.device)
        loss_hard = F.cross_entropy(logits_hard, labels_hard)

    loss = loss_inbatch + float(hard_negative_weight) * loss_hard
    metrics = {
        "loss": float(loss.detach().cpu()),
        "loss_inbatch": float(loss_inbatch.detach().cpu()),
        "loss_hard": float(loss_hard.detach().cpu()),
        "temp": float(temp.detach().cpu()),
    }
    return loss, metrics


def _load_item_pop_prior_for_split(split_name: str, item_id_by_idx: np.ndarray) -> np.ndarray:
    path = split_cache_dir(split_name) / "item_weight_by_idx.npy"
    if path.exists():
        arr = np.load(path).astype(np.float32)
        if len(arr) == len(item_id_by_idx):
            pop = np.log1p(arr)
        else:
            pop = np.zeros(len(item_id_by_idx), dtype=np.float32)
    else:
        pop = np.zeros(len(item_id_by_idx), dtype=np.float32)
    if np.nanmax(pop) > 0:
        q = np.nanquantile(pop[1:], 0.995) if len(pop) > 1 else np.nanmax(pop)
        q = float(q) if np.isfinite(q) and q > 0 else float(np.nanmax(pop))
        pop = np.clip(pop / max(q, 1e-6), 0.0, 1.0)
    pop[0] = 0.0
    return pop.astype(np.float32)


def _safe_model_buffer_numpy(model: nn.Module, name: str, length: int) -> np.ndarray:
    if hasattr(model, name):
        arr = getattr(model, name)
        if torch.is_tensor(arr):
            out = arr.detach().cpu().numpy()
            if len(out) == length:
                return out.astype(np.int32)
    return np.zeros(length, dtype=np.int32)


def _history_pref_maps(hist_idx: np.ndarray, value_by_item_idx: np.ndarray) -> dict[int, float]:
    hist_idx = hist_idx[hist_idx > 0]
    if len(hist_idx) == 0:
        return {}
    vals = value_by_item_idx[np.clip(hist_idx, 0, len(value_by_item_idx) - 1)]
    vals = vals[vals > 0]
    if len(vals) == 0:
        return {}
    unique, cnt = np.unique(vals.astype(np.int32), return_counts=True)
    denom = float(cnt.sum())
    return {int(k): float(v) / denom for k, v in zip(unique, cnt)}


def write_buffer_to_shards(rows: list[tuple], out_dir: Path, source_name: str, part_id: int) -> int:
    """Extended writer with raw DSSM and calibrated scores.

    Expected row tuple:
        user_id, item_id, score, rank, raw_score, pop_prior, cat_l0_pref, catalog_pref
    """
    if not rows:
        return 0
    df = pd.DataFrame(
        rows,
        columns=["user_id", "item_id", "score", "rank", "raw_dssm_score", "pop_prior", "cat_l0_pref", "catalog_pref"],
    )
    df["source"] = source_name
    df[f"{source_name}_score"] = df["score"].astype("float32")
    df[f"{source_name}_rank"] = df["rank"].astype("int32")
    df[f"{source_name}_raw_score"] = df["raw_dssm_score"].astype("float32")
    df[f"{source_name}_pop_prior"] = df["pop_prior"].astype("float32")
    df[f"{source_name}_cat_l0_pref"] = df["cat_l0_pref"].astype("float32")
    df[f"{source_name}_catalog_pref"] = df["catalog_pref"].astype("float32")
    df[f"in_{source_name}"] = np.int8(1)

    # Precompute f_num__candidate__ features so the reranker preserves even new DL columns.
    df[f"f_num__candidate__in_{source_name}"] = np.float32(1.0)
    df[f"f_num__candidate__{source_name}_score"] = df["score"].astype("float32")
    df[f"f_num__candidate__{source_name}_rank"] = df["rank"].astype("float32")
    df[f"f_num__candidate__{source_name}_rr"] = (1.0 / df["rank"].astype("float32")).astype("float32")
    df[f"f_num__candidate__{source_name}_raw_score"] = df["raw_dssm_score"].astype("float32")
    df[f"f_num__candidate__{source_name}_pop_prior"] = df["pop_prior"].astype("float32")
    df[f"f_num__candidate__{source_name}_cat_l0_pref"] = df["cat_l0_pref"].astype("float32")
    df[f"f_num__candidate__{source_name}_catalog_pref"] = df["catalog_pref"].astype("float32")

    df["user_shard"] = (df["user_id"].astype("int64") % CFG.n_shards).astype("int16")
    written = 0
    for shard_id, part in df.groupby("user_shard", sort=True):
        shard_dir = out_dir / f"user_shard={int(shard_id)}"
        shard_dir.mkdir(parents=True, exist_ok=True)
        out_file = shard_dir / f"{source_name}_part_{part_id:05d}.parquet"
        pl.from_pandas(part.drop(columns=["user_shard"])).write_parquet(out_file, compression="zstd")
        written += len(part)
    return written


def generate_candidates_streaming(
    model: nn.Module,
    seq_user_ids: np.ndarray,
    user_seq: dict[int, np.ndarray],
    seen_by_user: dict[int, set[int]],
    item_id_by_idx: np.ndarray,
    out_dir: Path,
    *,
    source_name: str,
    split_name: str,
) -> dict:
    """Generate candidates from DSSM vectors.

    New vs previous version:
    1. query more items from ANN (`ann_oversample`);
    2. rerank this oversample by calibrated score:
       DSSM cosine + small popularity prior + small user-category preference prior;
    3. write both calibrated and raw DSSM features for the reranker.
    """
    if out_dir.exists() and CFG.rebuild_dl_sources:
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if candidate_dir_looks_complete(out_dir) and not CFG.rebuild_dl_sources:
        print(f"{source_name}/{split_name}: source reused -> {rel(out_dir)}")
        return {"candidate_inference_time_sec": 0.0, "ann_backend": "reused", "rows": None}

    clean_output_dir(out_dir)
    t0 = time.perf_counter()

    item_vecs = compute_item_vectors(model, len(item_id_by_idx))
    ann = build_ann_index(item_vecs, MODELS_DIR / source_name / split_name, source_name, split_name)

    pop_prior = _load_item_pop_prior_for_split(split_name, item_id_by_idx)
    catalog_arr = _safe_model_buffer_numpy(model, "catalog_idx_by_item_idx", len(item_id_by_idx))
    cat_l0_arr = _safe_model_buffer_numpy(model, "cat_l0_idx_by_item_idx", len(item_id_by_idx))

    target_users = seq_user_ids
    rows_buffer: list[tuple] = []
    part_id = 0
    total_rows = 0

    for start in trange(0, len(target_users), CFG.retrieval_user_batch_size, desc=f"Generate {source_name}/{split_name}"):
        users = target_users[start:start + CFG.retrieval_user_batch_size]
        hists = np.zeros((len(users), CFG.sequence_max_len), dtype=np.int64)
        user_indices = np.zeros(len(users), dtype=np.int64)
        keep = []
        for j, user_id in enumerate(users):
            seq = user_seq.get(int(user_id))
            if seq is None or len(seq) == 0:
                continue
            hist = seq[-CFG.sequence_max_len:]
            hists[j, -len(hist):] = hist.astype(np.int64)
            user_indices[j] = start + j
            keep.append(j)
        if not keep:
            continue

        keep = np.asarray(keep, dtype=np.int64)
        batch_users = users[keep].astype(np.int64)
        batch_hists = hists[keep]
        user_vecs = encode_user_batch(model, user_indices[keep], batch_hists, batch_size=CFG.retrieval_user_batch_size)
        top_rows, top_scores = ann.query(user_vecs, CFG.ann_oversample)

        for local_row, user_id in enumerate(batch_users):
            hist_idx = batch_hists[local_row]
            l0_pref_map = _history_pref_maps(hist_idx, cat_l0_arr)
            catalog_pref_map = _history_pref_maps(hist_idx, catalog_arr)

            seen = seen_by_user.get(int(user_id), set()) if CFG.filter_seen_context_items else set()
            cand = []
            for ann_rank, (ridx, raw_score) in enumerate(zip(top_rows[local_row], top_scores[local_row]), start=1):
                ridx = int(ridx)
                item_idx = ridx + 1  # ANN row 0 corresponds to item_idx 1.
                if item_idx <= 0 or item_idx >= len(item_id_by_idx):
                    continue
                item_id = int(item_id_by_idx[item_idx])
                if CFG.filter_seen_context_items and item_id in seen:
                    continue

                cat_l0_pref = float(l0_pref_map.get(int(cat_l0_arr[item_idx]), 0.0))
                catalog_pref = float(catalog_pref_map.get(int(catalog_arr[item_idx]), 0.0))
                pprior = float(pop_prior[item_idx])
                calibrated = (
                    float(raw_score)
                    + float(DSSM_CANDIDATE_POP_ALPHA) * pprior
                    + float(DSSM_CANDIDATE_CAT_L0_ALPHA) * cat_l0_pref
                    + float(DSSM_CANDIDATE_CATALOG_ALPHA) * catalog_pref
                    - float(DSSM_CANDIDATE_RANK_ALPHA) * math.log1p(float(ann_rank))
                )
                cand.append((calibrated, item_id, float(raw_score), pprior, cat_l0_pref, catalog_pref))

            if not cand:
                continue
            cand.sort(key=lambda x: (-x[0], x[1]))
            for rank, (score, item_id, raw_score, pprior, cat_l0_pref, catalog_pref) in enumerate(cand[:CFG.dl_candidates_per_user], start=1):
                rows_buffer.append((
                    int(user_id),
                    int(item_id),
                    float(score),
                    int(rank),
                    float(raw_score),
                    float(pprior),
                    float(cat_l0_pref),
                    float(catalog_pref),
                ))

        if len(rows_buffer) >= CFG.retrieval_flush_rows:
            total_rows += write_buffer_to_shards(rows_buffer, out_dir, source_name, part_id)
            rows_buffer.clear()
            part_id += 1
            gc.collect()

    if rows_buffer:
        total_rows += write_buffer_to_shards(rows_buffer, out_dir, source_name, part_id)
        rows_buffer.clear()

    infer_time = time.perf_counter() - t0
    meta = {
        "source_name": source_name,
        "split_name": split_name,
        "rows": int(total_rows),
        "ann_backend": ann.backend,
        "candidate_inference_time_sec": float(infer_time),
        "multi_positive_loss": True,
        "candidate_calibration": {
            "pop_alpha": float(DSSM_CANDIDATE_POP_ALPHA),
            "cat_l0_alpha": float(DSSM_CANDIDATE_CAT_L0_ALPHA),
            "catalog_alpha": float(DSSM_CANDIDATE_CATALOG_ALPHA),
            "rank_alpha": float(DSSM_CANDIDATE_RANK_ALPHA),
            "ann_oversample": int(CFG.ann_oversample),
        },
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": asdict(CFG),
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"{source_name}/{split_name}: rows={total_rows:,}, time={infer_time:.1f}s -> {rel(out_dir)}")
    del item_vecs, ann
    gc.collect()
    return {"candidate_inference_time_sec": float(infer_time), "ann_backend": meta["ann_backend"], "rows": int(total_rows)}

## Retrieval objective

In [ ]:

# =================================================================================================
# DSSM final overrides: popularity-corrected multi-positive loss.
# This cell intentionally overrides init_model(), multipositive_inbatch_loss(), retrieval_losses().
# =================================================================================================

def _item_log_q_from_weights(item_weight_by_idx: np.ndarray) -> np.ndarray:
    w = np.asarray(item_weight_by_idx, dtype=np.float64)
    if w.ndim != 1 or len(w) == 0:
        return np.zeros(1, dtype=np.float32)
    w = np.maximum(w, 0.0)
    if len(w) > 0:
        w[0] = 0.0
    q = w + float(DSSM_LOGQ_SMOOTHING)
    if len(q) > 0:
        q[0] = float(DSSM_LOGQ_SMOOTHING)
    q = q / max(q.sum(), 1e-12)
    return np.log(np.maximum(q, 1e-12)).astype(np.float32)


def _attach_item_log_q(model: nn.Module, data: dict) -> nn.Module:
    log_q = _item_log_q_from_weights(data.get("item_weight_by_idx", np.ones(1, dtype=np.float32)))
    model.register_buffer("item_log_q_by_idx", torch.from_numpy(log_q), persistent=False)
    return model


def init_model(model_kind: str, data: dict) -> tuple[nn.Module, dict]:
    """Final init_model override with popularity-correction buffer attached to DSSM."""
    num_items = len(data["item_id_by_idx"])
    num_users = len(data["seq_user_ids"])

    if model_kind == "dssm":
        side = data["side"]
        model = DSSMMaxTwoTower(
            num_items=num_items,
            num_users=num_users,
            catalog_idx_by_item_idx=side["catalog_idx_by_item_idx"],
            cat_l0_idx_by_item_idx=side["cat_l0_idx_by_item_idx"],
            num_catalogs=side["num_catalogs"],
            num_cat_l0=side["num_cat_l0"],
            fclip_reduced=side["fclip_reduced"],
            output_dim=CFG.dssm_embedding_dim,
            dropout=CFG.dssm_dropout,
        )
        model = _attach_item_log_q(model, data)
        hparams = {
            "embedding_dim": CFG.dssm_embedding_dim,
            "item_id_dim": CFG.dssm_item_id_dim,
            "catalog_dim": CFG.dssm_catalog_dim,
            "cat_l0_dim": CFG.dssm_cat_l0_dim,
            "user_id_dim": CFG.dssm_user_id_dim if CFG.dssm_use_user_id_embedding else 0,
            "hidden_dims": list(CFG.dssm_hidden_dims),
            "dropout": CFG.dssm_dropout,
            "pretrain_epochs": DSSM_PRETRAIN_EPOCHS,
            "finetune_epochs": DSSM_FINETUNE_EPOCHS,
            "pretrain_lr": DSSM_PRETRAIN_LR,
            "finetune_lr": DSSM_FINETUNE_LR,
            "batch_size": CFG.dssm_batch_size,
            "finetune_batch_size": DSSM_FINETUNE_BATCH_SIZE,
            "weight_decay": CFG.dssm_weight_decay,
            "hard_negatives_per_example": CFG.hard_negatives_per_example,
            "hard_negative_weight": CFG.dssm_hard_negative_weight,
            "popularity_correction_beta": DSSM_POPULARITY_CORRECTION_BETA,
            "logq_smoothing": DSSM_LOGQ_SMOOTHING,
            "fclip_dim": side["fclip_dim"],
            "light_history_tower": CFG.dssm_light_history_tower,
            "training_mode": "two_stage_multipos_popularity_corrected",
        }
        return model, hparams

    if model_kind == "sasrec":
        model = SASRecRetrieval(
            num_items=num_items,
            emb_dim=CFG.sasrec_embedding_dim,
            max_len=CFG.sequence_max_len,
            num_heads=CFG.sasrec_num_heads,
            num_layers=CFG.sasrec_num_layers,
            dropout=CFG.sasrec_dropout,
        )
        hparams = {
            "embedding_dim": CFG.sasrec_embedding_dim,
            "layers": CFG.sasrec_num_layers,
            "heads": CFG.sasrec_num_heads,
            "dropout": CFG.sasrec_dropout,
            "epochs": CFG.sasrec_epochs,
            "batch_size": CFG.sasrec_batch_size,
            "lr": CFG.sasrec_lr,
            "weight_decay": CFG.sasrec_weight_decay,
            "hard_negatives_per_example": CFG.hard_negatives_per_example,
            "hard_negative_weight": CFG.sasrec_hard_negative_weight,
        }
        return model, hparams

    raise ValueError(model_kind)


def _correct_logits_by_item_popularity(model: nn.Module, logits: torch.Tensor, item_idx: torch.Tensor) -> torch.Tensor:
    beta = float(DSSM_POPULARITY_CORRECTION_BETA)
    if beta <= 0 or not hasattr(model, "item_log_q_by_idx"):
        return logits
    log_q = model.item_log_q_by_idx.to(device=logits.device, dtype=logits.dtype)
    item_idx = item_idx.clamp(0, log_q.numel() - 1)
    return logits - beta * log_q[item_idx]


def multipositive_inbatch_loss(
    model: nn.Module,
    user_vec: torch.Tensor,
    pos_vec: torch.Tensor,
    user_idx: torch.Tensor,
    pos: torch.Tensor,
    temp: torch.Tensor,
) -> torch.Tensor:
    """Multi-positive in-batch softmax with item-popularity correction."""
    logits = (user_vec @ pos_vec.T) / temp
    if float(DSSM_POPULARITY_CORRECTION_BETA) > 0 and hasattr(model, "item_log_q_by_idx"):
        log_q = model.item_log_q_by_idx.to(device=logits.device, dtype=logits.dtype)
        pos_safe = pos.clamp(0, log_q.numel() - 1)
        logits = logits - float(DSSM_POPULARITY_CORRECTION_BETA) * log_q[pos_safe][None, :]

    same_user = user_idx[:, None].eq(user_idx[None, :])
    eye = torch.eye(logits.size(0), device=logits.device, dtype=torch.bool)
    positive_mask = same_user | eye

    neg_inf = torch.finfo(logits.dtype).min
    numerator = torch.logsumexp(logits.masked_fill(~positive_mask, neg_inf), dim=1)
    denominator = torch.logsumexp(logits, dim=1)
    return -(numerator - denominator).mean()


def retrieval_losses(
    model: nn.Module,
    user_idx: torch.Tensor,
    hist: torch.Tensor,
    pos: torch.Tensor,
    neg: torch.Tensor,
    hard_negative_weight: float,
) -> tuple[torch.Tensor, dict]:
    u = model.encode_user(user_idx, hist)
    pos_v = model.encode_item_idx(pos)
    temp = model.temperature.clamp(0.02, 0.20)

    loss_inbatch = multipositive_inbatch_loss(model, u, pos_v, user_idx, pos, temp)

    loss_hard = torch.zeros((), dtype=loss_inbatch.dtype, device=loss_inbatch.device)
    if float(hard_negative_weight) > 0 and neg is not None and neg.numel() > 0 and neg.ndim == 2 and neg.size(1) > 0:
        bsz, hneg = neg.shape
        neg_flat = neg.reshape(-1)
        neg_v = model.encode_item_idx(neg_flat).reshape(bsz, hneg, -1)
        pos_score = (u * pos_v).sum(dim=-1, keepdim=True)
        neg_score = torch.einsum("bd,bhd->bh", u, neg_v)
        logits_hard = torch.cat([pos_score, neg_score], dim=1) / temp
        item_idx_hard = torch.cat([pos[:, None], neg], dim=1)
        logits_hard = _correct_logits_by_item_popularity(model, logits_hard, item_idx_hard)
        labels_hard = torch.zeros(bsz, dtype=torch.long, device=u.device)
        loss_hard = F.cross_entropy(logits_hard, labels_hard)

    loss = loss_inbatch + float(hard_negative_weight) * loss_hard
    metrics = {
        "loss": float(loss.detach().cpu()),
        "loss_inbatch": float(loss_inbatch.detach().cpu()),
        "loss_hard": float(loss_hard.detach().cpu()),
        "temp": float(temp.detach().cpu()),
        "pop_beta": float(DSSM_POPULARITY_CORRECTION_BETA),
    }
    return loss, metrics

print("Final DSSM objective active:")
print("  multi-positive in-batch loss")
print("  popularity correction beta:", DSSM_POPULARITY_CORRECTION_BETA)
print("  filtered hard negatives:", CFG.hard_negatives_per_example, "per example")
print("  source:", DL_SUPERVISED_SOURCE)


## Validation experiment

In [ ]:

experiment_rows = []
val_metrics_frames = []

print("=" * 120)
print("DSSM checkpoint/source stage:", DL_SUPERVISED_SOURCE)
print("RUN_DSSM_TRAINING_STAGE:", RUN_DSSM_TRAINING_STAGE)
print("RUN_VAL_DSSM_SOURCE_STAGE:", RUN_VAL_DSSM_SOURCE_STAGE)

if RUN_DSSM_TRAINING_STAGE or RUN_VAL_DSSM_SOURCE_STAGE:
    # If the checkpoint exists and FORCE_RETRAIN_DSSM=False, train_and_generate_source loads the checkpoint
    # and only rebuilds the validation DSSM source if needed.
    stats = train_and_generate_source("val", DL_SUPERVISED_SOURCE, "dssm", build_source=True)
    experiment_rows.append(stats)
else:
    print("Skip DSSM training/source generation: checkpoint and val source already exist.")
    stats = {
        "model": DL_SUPERVISED_SOURCE,
        "model_kind": "dssm",
        "split": "val",
        "status": "reused_existing_checkpoint_and_val_source",
        "model_path": rel(TRAINING_CHECKPOINT_PATH),
    }
    experiment_rows.append(stats)

print_dl_source_sanity(DL_SUPERVISED_SOURCE, "val")

if candidate_dir_looks_complete(CANDIDATES_DIR / DL_SUPERVISED_SOURCE / "val"):
    gt_val = build_target_gt(CFG.val_target_start, CFG.val_target_end)
    metrics = evaluate_candidate_dir(
        CANDIDATES_DIR / DL_SUPERVISED_SOURCE / "val",
        gt_val,
        ks=CFG.metric_ks,
        rank_col="rank",
        label=DL_SUPERVISED_SOURCE,
    )
    val_metrics_frames.append(metrics)

    for _, row in metrics.iterrows():
        k = int(row["k"])
        stats[f"recall@{k}"] = float(row["recall"])
        stats[f"ndcg@{k}"] = float(row["ndcg"])
        stats[f"hitrate@{k}"] = float(row["hit_rate"])
else:
    print("Validation DSSM source is missing; metrics are skipped.")

experiments = pd.DataFrame(experiment_rows)
val_metrics = pd.concat(val_metrics_frames, ignore_index=True) if val_metrics_frames else pd.DataFrame()

pd.set_option("display.max_columns", 200)
display(experiments)
display(val_metrics)

experiments.to_parquet(REPORTS_DIR / "dl_dssm_full_pipeline_experiments.parquet", index=False)
experiments.to_csv(REPORTS_DIR / "dl_dssm_full_pipeline_experiments.csv", index=False)
if not val_metrics.empty:
    val_metrics.to_parquet(REPORTS_DIR / "dl_dssm_full_pipeline_val_metrics.parquet", index=False)
    val_metrics.to_csv(REPORTS_DIR / "dl_dssm_full_pipeline_val_metrics.csv", index=False)

print("saved:")
print(rel(REPORTS_DIR / "dl_dssm_full_pipeline_experiments.parquet"))
if not val_metrics.empty:
    print(rel(REPORTS_DIR / "dl_dssm_full_pipeline_val_metrics.parquet"))


## Build DL candidates for all splits and safe merge

Следующий блок работает как inference/merge stage:

- загружает обученный DSSM checkpoint из `models/dl_candidate_generators_dssm_final/dl_dssm_final/val/model.pt`;
- строит отсутствующие `dl_dssm_final/{ranker_train,val,test}` без переобучения;
- собирает `merged_with_dl/{ranker_train,val,test}` по схеме `base 700 + DSSM 300 = total 1000`;
- считает validation metrics, unique positives и quota ablation.


# DSSM retrieval and merge

In [ ]:

import gc
import json
import math
import shutil
import time
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm, trange

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import hnswlib
    HAS_HNSWLIB = True
except Exception:
    HAS_HNSWLIB = False

try:
    import faiss
    HAS_FAISS = True
except Exception:
    HAS_FAISS = False


def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "datamart").exists() or (p / "data" / "raw").exists() or (p / "data").exists():
            return p
    raise FileNotFoundError("Не найден project root с data/datamart или data/raw")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATAMART_DIR = DATA_DIR / "datamart"
AGG_DIR = DATAMART_DIR / "agg"
CANDIDATES_DIR = DATAMART_DIR / "candidates"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR = DATA_DIR / "processed" / "dl_candidate_generators_dssm_final"
MODELS_DIR = PROJECT_ROOT / "models" / "dl_candidate_generators_dssm_final"
SOURCE_NAME = "dl_dssm_final"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


@dataclass(frozen=True)
class DLInferenceMergeConfig:
    # Dates are aligned with candidate/reranker notebooks.
    ranker_train_context_start: date = date(2025, 4, 18)
    ranker_train_context_end: date = date(2025, 6, 3)
    ranker_train_target_start: date = date(2025, 6, 4)
    ranker_train_target_end: date = date(2025, 6, 17)

    val_context_start: date = date(2025, 4, 18)
    val_context_end: date = date(2025, 6, 17)
    val_target_start: date = date(2025, 6, 18)
    val_target_end: date = date(2025, 7, 1)

    full_context_start: date = date(2025, 5, 2)
    full_context_end: date = date(2025, 7, 1)
    test_target_start: date = date(2025, 7, 2)
    test_target_end: date = date(2025, 7, 15)

    n_shards: int = 128
    source_name: str = SOURCE_NAME
    output_merged_name: str = "merged_with_dl"

    # Use the already trained validation checkpoint. Do not retrain DSSM here.
    checkpoint_split: str = "val"
    checkpoint_path: str = ""  # empty -> models/.../dl_dssm_final/val/model.pt

    # Build only missing DL sources by default. val exists already; ranker_train/test are usually missing.
    splits_to_build_dl: tuple[str, ...] = ("ranker_train", "val", "test")
    splits_to_merge: tuple[str, ...] = ("ranker_train", "val", "test")

    # Rebuild controls.
    rebuild_dl_sources: bool = False       # False = do not overwrite existing val source
    rebuild_missing_dl_sources: bool = True
    rebuild_ann_index: bool = False        # load existing val HNSW if present
    rebuild_merged_with_dl: bool = True

    # DSSM source generation.
    sequence_max_len: int = 80
    dl_candidates_per_user: int = 300
    ann_oversample: int = 2000
    retrieval_user_batch_size: int = 4096
    retrieval_flush_rows: int = 2_000_000
    hnsw_ef_search: int = 500
    hnsw_m: int = 24
    hnsw_ef_construction: int = 120
    hnsw_num_threads: int = -1
    hnsw_add_batch_size: int = 50_000
    filter_seen_context_items: bool = False

    # Safe merge policy.
    base_keep_per_user: int = 700
    dl_keep_per_user: int = 300
    total_keep_per_user: int = 1000  # base 700 + up to all 300 DSSM candidates

    # Calibration used in final DSSM notebook.
    dssm_candidate_pop_alpha: float = 0.04
    dssm_candidate_cat_l0_alpha: float = 0.08
    dssm_candidate_catalog_alpha: float = 0.03
    dssm_candidate_rank_alpha: float = 0.0005

    # Model architecture. These are also checked against checkpoint config when available.
    dssm_embedding_dim: int = 256
    dssm_item_id_dim: int = 256
    dssm_catalog_dim: int = 64
    dssm_cat_l0_dim: int = 32
    dssm_user_id_dim: int = 64
    dssm_hidden_dims: tuple[int, ...] = (1024, 512, 256)
    dssm_dropout: float = 0.12
    dssm_temperature_init: float = 0.07
    dssm_use_user_id_embedding: bool = False
    dssm_use_fclip_projection: bool = True
    fclip_project_dim: int = 64

    # Interaction sequence construction.
    weight_preset: str = "hybrid"
    event_weight_page_view: float = 1.0
    event_weight_view_description: float = 2.0
    event_weight_review_view: float = 3.0
    event_weight_favorite: float = 10.0
    event_weight_to_cart: float = 30.0
    order_weight: float = 60.0
    weak_max_cap: float = 10.0
    min_item_weight: float = 0.1
    time_decay_enabled: bool = True
    time_decay_tau_days: float = 30.0
    time_decay_min_factor: float = 0.05

    metric_ks: tuple[int, ...] = (20, 50, 100, 200, 300, 700, 900, 1000)
    evaluate_val: bool = True
    run_unique_positive_report: bool = True
    run_quota_ablation: bool = True
    quota_values: tuple[int, ...] = (50, 100, 200, 300)


CFG = DLInferenceMergeConfig()


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def rel(path: Path | str) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)


def cfg_json() -> str:
    return json.dumps({k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()}, ensure_ascii=False, indent=2)


print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("HAS_HNSWLIB:", HAS_HNSWLIB, "HAS_FAISS:", HAS_FAISS)
print("Config:")
print(cfg_json())


In [ ]:

# -----------------------------
# I/O helpers
# -----------------------------

def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def candidate_files(path: Path) -> list[Path]:
    return sorted(path.rglob("*.parquet")) if path.exists() else []


def candidate_dir_looks_complete(path: Path, *, min_shards: int | None = None) -> bool:
    files = candidate_files(path)
    if not files:
        return False
    if min_shards is None:
        return True
    shard_dirs = {p.parent.name for p in files if p.parent.name.startswith("user_shard=") and p.stat().st_size > 0}
    return len(shard_dirs) >= min_shards


def scan_parquet(path: Path | str, *, hive_partitioning: bool = True) -> pl.LazyFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_dir():
        if not path_has_parquet(path):
            raise FileNotFoundError(f"No parquet files found in {path}")
        return pl.scan_parquet(str(path / "**" / "*.parquet"), hive_partitioning=hive_partitioning)
    return pl.scan_parquet(str(path), hive_partitioning=hive_partitioning)


def scan_candidate_shard_or_none(base_dir: Path, shard_id: int) -> pl.LazyFrame | None:
    shard_dir = base_dir / f"user_shard={shard_id}"
    if not path_has_parquet(shard_dir):
        return None
    return pl.scan_parquet(str(shard_dir / "*.parquet"), hive_partitioning=True)


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return list(lf.collect_schema().names())
    except Exception:
        return list(lf.schema.keys())


def clean_output_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def resolve_manifest_candidate_dir(split: str) -> Path | None:
    manifest_path = CANDIDATES_DIR / "manifest_candidate_generators_enhanced.json"
    if not manifest_path.exists():
        return None
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"WARNING: failed to read manifest {rel(manifest_path)}: {e}")
        return None
    key_by_split = {
        "ranker_train": "ranker_train_candidates",
        "val": "val_candidates",
        "test": "test_candidates",
    }
    raw = manifest.get("outputs", {}).get(key_by_split.get(split, ""))
    if not raw:
        return None
    p = Path(raw)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    return p if candidate_dir_looks_complete(p) else None


def resolve_base_candidate_dir(split: str) -> Path | None:
    p = resolve_manifest_candidate_dir(split)
    if p is not None:
        return p
    for p in [
        CANDIDATES_DIR / "merged" / split,
        CANDIDATES_DIR / "merged_enhanced" / split,
        CANDIDATES_DIR / "enhanced_merged" / split,
    ]:
        if candidate_dir_looks_complete(p):
            return p
    return None


def resolve_dl_candidate_dir(split: str) -> Path | None:
    p = CANDIDATES_DIR / CFG.source_name / split
    return p if candidate_dir_looks_complete(p) else None


def get_checkpoint_path() -> Path:
    if CFG.checkpoint_path:
        p = Path(CFG.checkpoint_path)
        return p if p.is_absolute() else PROJECT_ROOT / p
    return MODELS_DIR / CFG.source_name / CFG.checkpoint_split / "model.pt"


def read_base_users(split: str) -> np.ndarray:
    cache = PROCESSED_DIR / "inference_from_checkpoint" / "base_users"
    ensure_dir(cache)
    out = cache / f"base_users_{split}.npy"
    if out.exists():
        arr = np.load(out)
        print(f"{split}: base users loaded -> {rel(out)}, users={len(arr):,}")
        return arr.astype(np.int64)

    base_dir = resolve_base_candidate_dir(split)
    if base_dir is None:
        raise FileNotFoundError(f"Base candidates not found for split={split}")

    users = []
    for shard_id in trange(CFG.n_shards, desc=f"Read base users {split}"):
        lf = scan_candidate_shard_or_none(base_dir, shard_id)
        if lf is None:
            continue
        part = collect_streaming(lf.select(pl.col("user_id").cast(pl.Int64).unique()))
        users.append(part["user_id"].to_numpy())
    if not users:
        raise RuntimeError(f"No base users read for split={split}")
    arr = np.unique(np.concatenate(users).astype(np.int64))
    np.save(out, arr)
    print(f"{split}: base users saved -> {rel(out)}, users={len(arr):,}")
    return arr


In [ ]:

# -----------------------------
# Context sequences with checkpoint item vocabulary
# -----------------------------

def events_path(level: str) -> Path:
    return AGG_DIR / "events" / level


def orders_path(level: str) -> Path:
    return AGG_DIR / "orders" / level


def read_events_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(events_path(level), hive_partitioning=True)
    return lf.filter((pl.col("event_date").cast(pl.Date) >= pl.lit(start)) & (pl.col("event_date").cast(pl.Date) <= pl.lit(end)))


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    lf = scan_parquet(orders_path(level), hive_partitioning=True)
    return lf.filter((pl.col("created_date").cast(pl.Date) >= pl.lit(start)) & (pl.col("created_date").cast(pl.Date) <= pl.lit(end)))


def ensure_numeric_cols(lf: pl.LazyFrame, cols: Sequence[str]) -> pl.LazyFrame:
    existing = set(lazy_columns(lf))
    exprs = []
    for c in cols:
        if c not in existing:
            exprs.append(pl.lit(0, dtype=pl.Float32).alias(c))
        else:
            exprs.append(pl.col(c).fill_null(0).cast(pl.Float32).alias(c))
    return lf.with_columns(exprs)


def split_spec(split_name: str) -> tuple[date, date, date, date]:
    if split_name == "ranker_train":
        return CFG.ranker_train_context_start, CFG.ranker_train_context_end, CFG.ranker_train_target_start, CFG.ranker_train_target_end
    if split_name == "val":
        return CFG.val_context_start, CFG.val_context_end, CFG.val_target_start, CFG.val_target_end
    if split_name == "test":
        return CFG.full_context_start, CFG.full_context_end, CFG.test_target_start, CFG.test_target_end
    raise ValueError(split_name)


def weight_values() -> dict[str, float]:
    if CFG.weight_preset == "ecup_soft":
        return {"page_view": 2.0, "view_description": 3.0, "review_view": 3.0, "favorite": 5.0, "to_cart": 1.0, "order": 5.0}
    if CFG.weight_preset == "hybrid":
        return {"page_view": 1.0, "view_description": 2.0, "review_view": 3.0, "favorite": 10.0, "to_cart": 30.0, "order": 60.0}
    return {
        "page_view": CFG.event_weight_page_view,
        "view_description": CFG.event_weight_view_description,
        "review_view": CFG.event_weight_review_view,
        "favorite": CFG.event_weight_favorite,
        "to_cart": CFG.event_weight_to_cart,
        "order": CFG.order_weight,
    }


def time_decay_expr(date_col: str, context_end: date) -> pl.Expr:
    if not CFG.time_decay_enabled:
        return pl.lit(1.0, dtype=pl.Float32)
    age_days = (pl.lit(context_end) - pl.col(date_col).cast(pl.Date)).dt.total_days().cast(pl.Float32)
    return (-age_days / float(CFG.time_decay_tau_days)).exp().clip(lower_bound=CFG.time_decay_min_factor, upper_bound=1.0).cast(pl.Float32)


def build_interactions_lf(start: date, end: date, base_users: np.ndarray | None = None) -> pl.LazyFrame:
    w = weight_values()
    user_filter = None
    if base_users is not None:
        users_df = pl.DataFrame({"user_id": base_users.astype(np.int64)}).lazy()
        user_filter = users_df

    event_cols = ["cnt_page_view", "cnt_view_description", "cnt_review_view", "cnt_to_cart", "cnt_favorite"]
    e = ensure_numeric_cols(read_events_lf("user_item", start, end), event_cols)
    if user_filter is not None:
        e = e.join(user_filter, on="user_id", how="inner")
    event_weight = (
        pl.col("cnt_page_view").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["page_view"]
        + pl.col("cnt_view_description").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["view_description"]
        + pl.col("cnt_review_view").cast(pl.Float32).clip(lower_bound=0, upper_bound=CFG.weak_max_cap) * w["review_view"]
        + (pl.col("cnt_favorite").fill_null(0) > 0).cast(pl.Float32) * w["favorite"]
        + (pl.col("cnt_to_cart").fill_null(0) > 0).cast(pl.Float32) * w["to_cart"]
    ) * time_decay_expr("event_date", end)
    e = (
        e.with_columns(event_weight.alias("weight"), pl.col("event_date").cast(pl.Date).alias("dt"))
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), "dt", pl.col("weight").cast(pl.Float32))
        .filter(pl.col("weight") >= float(CFG.min_item_weight))
    )

    o = ensure_numeric_cols(read_orders_lf("user_item", start, end), ["cnt_orders"])
    if user_filter is not None:
        o = o.join(user_filter, on="user_id", how="inner")
    order_weight = (pl.col("cnt_orders").fill_null(0).cast(pl.Float32) * w["order"] * time_decay_expr("created_date", end)).cast(pl.Float32)
    o = (
        o.with_columns(order_weight.alias("weight"), pl.col("created_date").cast(pl.Date).alias("dt"))
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), "dt", pl.col("weight").cast(pl.Float32))
        .filter(pl.col("weight") >= float(CFG.min_item_weight))
    )

    return (
        pl.concat([e, o], how="diagonal_relaxed")
        .group_by(["user_id", "item_id", "dt"])
        .agg(pl.sum("weight").cast(pl.Float32).alias("weight"))
        .filter(pl.col("weight") > 0)
    )


def build_sequences_for_split_with_checkpoint_vocab(split_name: str, item2idx: dict[int, int]) -> tuple[np.ndarray, dict[int, np.ndarray], dict[int, set[int]]]:
    cache = PROCESSED_DIR / "inference_from_checkpoint" / "sequences" / split_name
    ensure_dir(cache)
    user_ids_path = cache / "user_ids.npy"
    seqs_path = cache / "seqs.npy"
    meta_path = cache / "meta.json"

    if user_ids_path.exists() and seqs_path.exists() and meta_path.exists():
        user_ids = np.load(user_ids_path).astype(np.int64)
        seqs_obj = np.load(seqs_path, allow_pickle=True)
        seqs = [np.asarray(x, dtype=np.int32) for x in seqs_obj.tolist()]
        user_seq = {int(u): seqs[i] for i, u in enumerate(user_ids)}
        seen_by_user = {int(u): set() for u in user_ids}  # not needed unless filtering seen items
        print(f"{split_name}: checkpoint-vocab sequences reused -> {rel(cache)}, users={len(user_ids):,}")
        return user_ids, user_seq, seen_by_user

    base_users = read_base_users(split_name)
    start, end, _, _ = split_spec(split_name)
    item_map = pl.DataFrame({
        "item_id": np.fromiter(item2idx.keys(), dtype=np.int64),
        "item_idx": np.fromiter(item2idx.values(), dtype=np.int32),
    }).lazy()

    print(f"{split_name}: build context sequences with checkpoint item vocab, context={start}..{end}, base_users={len(base_users):,}")
    lf = (
        build_interactions_lf(start, end, base_users=base_users)
        .join(item_map, on="item_id", how="inner")
        .select("user_id", "item_idx", "dt", "weight")
        .sort(["user_id", "dt", "weight", "item_idx"], descending=[False, False, False, False])
        .group_by("user_id", maintain_order=True)
        .agg(pl.col("item_idx").tail(CFG.sequence_max_len).alias("seq"))
        .sort("user_id")
    )
    seq_df = collect_streaming(lf)
    if seq_df.is_empty():
        raise RuntimeError(f"No sequences built for split={split_name}")

    user_ids = seq_df["user_id"].to_numpy().astype(np.int64)
    seqs = [np.asarray(x, dtype=np.int32) for x in seq_df["seq"].to_list()]
    np.save(user_ids_path, user_ids)
    np.save(seqs_path, np.asarray(seqs, dtype=object), allow_pickle=True)
    meta_path.write_text(json.dumps({
        "split": split_name,
        "context_start": str(start),
        "context_end": str(end),
        "base_users": int(len(base_users)),
        "users_with_checkpoint_vocab_history": int(len(user_ids)),
        "sequence_max_len": int(CFG.sequence_max_len),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{split_name}: checkpoint-vocab sequences saved -> {rel(cache)}, users={len(user_ids):,}")
    return user_ids, {int(u): seqs[i] for i, u in enumerate(user_ids)}, {int(u): set() for u in user_ids}


In [ ]:

# -----------------------------
# DSSM model definition and checkpoint loading
# -----------------------------

class MLP(nn.Module):
    def __init__(self, dims: Sequence[int], dropout: float, final_norm: bool = False):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(nn.GELU())
                layers.append(nn.LayerNorm(dims[i + 1]))
                layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)
        self.final_norm = final_norm

    def forward(self, x):
        x = self.net(x)
        return F.normalize(x, dim=-1) if self.final_norm else x


class AttentionPooling(nn.Module):
    def __init__(self, dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        scores = self.scorer(x).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e4)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)


class DSSMMaxTwoTower(nn.Module):
    def __init__(
        self,
        num_items: int,
        num_users: int,
        catalog_idx_by_item_idx: np.ndarray,
        cat_l0_idx_by_item_idx: np.ndarray,
        num_catalogs: int,
        num_cat_l0: int,
        *,
        fclip_reduced: np.ndarray | None = None,
        output_dim: int = CFG.dssm_embedding_dim,
        dropout: float = CFG.dssm_dropout,
    ):
        super().__init__()
        self.num_items = int(num_items)
        self.output_dim = int(output_dim)
        self.use_user_id = bool(CFG.dssm_use_user_id_embedding)

        self.item_id_embedding = nn.Embedding(num_items, CFG.dssm_item_id_dim, padding_idx=0)
        self.catalog_embedding = nn.Embedding(max(1, num_catalogs), CFG.dssm_catalog_dim, padding_idx=0)
        self.cat_l0_embedding = nn.Embedding(max(1, num_cat_l0), CFG.dssm_cat_l0_dim, padding_idx=0)

        self.register_buffer("catalog_idx_by_item_idx", torch.as_tensor(catalog_idx_by_item_idx, dtype=torch.long), persistent=False)
        self.register_buffer("cat_l0_idx_by_item_idx", torch.as_tensor(cat_l0_idx_by_item_idx, dtype=torch.long), persistent=False)

        clip_dim = 0
        if fclip_reduced is not None:
            clip_arr = np.asarray(fclip_reduced, dtype=np.float32)
            clip_dim = int(clip_arr.shape[1])
            self.register_buffer("fclip_table", torch.from_numpy(clip_arr), persistent=False)
        else:
            self.register_buffer("fclip_table", torch.empty(0, 0, dtype=torch.float32), persistent=False)

        item_input_dim = CFG.dssm_item_id_dim + CFG.dssm_catalog_dim + CFG.dssm_cat_l0_dim + clip_dim
        self.item_base = nn.Sequential(
            nn.Linear(item_input_dim, CFG.dssm_hidden_dims[0]),
            nn.GELU(),
            nn.LayerNorm(CFG.dssm_hidden_dims[0]),
            nn.Dropout(dropout),
            nn.Linear(CFG.dssm_hidden_dims[0], output_dim),
            nn.LayerNorm(output_dim),
        )
        self.item_tower = MLP([output_dim, *CFG.dssm_hidden_dims[1:], output_dim], dropout, final_norm=True)

        history_input_dim = CFG.dssm_item_id_dim + CFG.dssm_catalog_dim + CFG.dssm_cat_l0_dim
        self.history_project = nn.Sequential(
            nn.Linear(history_input_dim, output_dim),
            nn.GELU(),
            nn.LayerNorm(output_dim),
            nn.Dropout(dropout),
        )

        user_input_dim = output_dim * 3
        if self.use_user_id:
            self.user_embedding = nn.Embedding(num_users, CFG.dssm_user_id_dim)
            user_input_dim += CFG.dssm_user_id_dim
        else:
            self.user_embedding = None

        self.attention_pool = AttentionPooling(output_dim, max(64, output_dim // 2), dropout)
        self.user_tower = MLP([user_input_dim, *CFG.dssm_hidden_dims, output_dim], dropout, final_norm=True)
        self.temperature = nn.Parameter(torch.tensor(float(CFG.dssm_temperature_init)))

    def _item_base_features(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_items - 1)
        catalog = self.catalog_idx_by_item_idx[item_idx]
        l0 = self.cat_l0_idx_by_item_idx[item_idx]
        parts = [
            self.item_id_embedding(item_idx),
            self.catalog_embedding(catalog.clamp(0, self.catalog_embedding.num_embeddings - 1)),
            self.cat_l0_embedding(l0.clamp(0, self.cat_l0_embedding.num_embeddings - 1)),
        ]
        if self.fclip_table.numel() > 0:
            parts.append(self.fclip_table[item_idx].to(parts[0].device))
        return torch.cat(parts, dim=-1)

    def _history_features(self, item_idx: torch.Tensor) -> torch.Tensor:
        item_idx = item_idx.clamp(0, self.num_items - 1)
        catalog = self.catalog_idx_by_item_idx[item_idx]
        l0 = self.cat_l0_idx_by_item_idx[item_idx]
        return torch.cat([
            self.item_id_embedding(item_idx),
            self.catalog_embedding(catalog.clamp(0, self.catalog_embedding.num_embeddings - 1)),
            self.cat_l0_embedding(l0.clamp(0, self.cat_l0_embedding.num_embeddings - 1)),
        ], dim=-1)

    def encode_item_idx(self, item_idx: torch.Tensor) -> torch.Tensor:
        base = self.item_base(self._item_base_features(item_idx))
        return self.item_tower(base)

    def encode_user(self, user_idx: torch.Tensor, hist_item_idx: torch.Tensor) -> torch.Tensor:
        mask = hist_item_idx > 0
        tok = self.history_project(self._history_features(hist_item_idx))
        mask_f = mask.unsqueeze(-1).float()
        mean = (tok * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp_min(1.0)
        att = self.attention_pool(tok, mask)
        # Sequences are right-aligned, so the last real event is at -1 for users with non-empty history.
        last = tok[:, -1, :]
        parts = [mean, att, last]
        if self.use_user_id and self.user_embedding is not None:
            parts.append(self.user_embedding(user_idx.clamp(0, self.user_embedding.num_embeddings - 1)))
        return self.user_tower(torch.cat(parts, dim=-1))


def load_side_arrays_for_checkpoint(item_id_by_idx: np.ndarray) -> dict:
    side_dir = PROCESSED_DIR / "item_side"
    side_path = side_dir / f"item_side_{CFG.checkpoint_split}.npz"
    meta_path = side_dir / f"item_side_{CFG.checkpoint_split}_meta.json"
    if not side_path.exists() or not meta_path.exists():
        raise FileNotFoundError(
            f"Cached side arrays are required for inference-only run: {rel(side_path)} / {rel(meta_path)}. "
            "Do not rebuild DSSM; first restore these files or rerun only item_side preparation."
        )
    data = np.load(side_path)
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    catalog_idx = data["catalog_idx_by_item_idx"].astype(np.int64)
    cat_l0_idx = data["cat_l0_idx_by_item_idx"].astype(np.int64)
    if len(catalog_idx) != len(item_id_by_idx) or len(cat_l0_idx) != len(item_id_by_idx):
        raise ValueError(f"Side array length mismatch: side={len(catalog_idx)}, checkpoint items={len(item_id_by_idx)}")

    fclip = None
    fclip_path = side_dir / f"fclip_reduced_{CFG.checkpoint_split}_dim={CFG.fclip_project_dim}.npy"
    if CFG.dssm_use_fclip_projection:
        if not fclip_path.exists():
            raise FileNotFoundError(f"fclip reduced matrix not found: {rel(fclip_path)}")
        fclip = np.load(fclip_path, mmap_mode="r")
        if len(fclip) != len(item_id_by_idx):
            raise ValueError(f"fclip length mismatch: fclip={len(fclip)}, checkpoint items={len(item_id_by_idx)}")

    return {
        "catalog_idx_by_item_idx": catalog_idx,
        "cat_l0_idx_by_item_idx": cat_l0_idx,
        "num_catalogs": int(meta["num_catalogs"]),
        "num_cat_l0": int(meta["num_cat_l0"]),
        "fclip_reduced": fclip,
    }


def load_dssm_checkpoint() -> tuple[nn.Module, np.ndarray, dict[int, int]]:
    ckpt_path = get_checkpoint_path()
    if not ckpt_path.exists():
        raise FileNotFoundError(f"DSSM checkpoint not found: {rel(ckpt_path)}")
    print(f"Load checkpoint: {rel(ckpt_path)}")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    item_id_by_idx = np.asarray(ckpt["item_id_by_idx"], dtype=np.int64)
    item2idx = {int(item_id): idx for idx, item_id in enumerate(item_id_by_idx) if idx > 0}
    side = load_side_arrays_for_checkpoint(item_id_by_idx)
    model = DSSMMaxTwoTower(
        num_items=len(item_id_by_idx),
        num_users=1,  # user_id embedding is disabled in final DSSM
        catalog_idx_by_item_idx=side["catalog_idx_by_item_idx"],
        cat_l0_idx_by_item_idx=side["cat_l0_idx_by_item_idx"],
        num_catalogs=side["num_catalogs"],
        num_cat_l0=side["num_cat_l0"],
        fclip_reduced=side["fclip_reduced"],
        output_dim=CFG.dssm_embedding_dim,
        dropout=CFG.dssm_dropout,
    )
    state = ckpt["model_state_dict"]
    state = {k: v for k, v in state.items() if k != "fclip_embedding.weight"}
    res = model.load_state_dict(state, strict=False)
    missing = [k for k in res.missing_keys if not k.startswith("fclip_table")]
    unexpected = [k for k in res.unexpected_keys if k != "fclip_embedding.weight"]
    if missing or unexpected:
        print("load_state_dict missing:", missing[:10], "unexpected:", unexpected[:10])
    model.eval().to(DEVICE)
    print(f"Checkpoint loaded: items={len(item_id_by_idx)-1:,}, source={SOURCE_NAME}")
    return model, item_id_by_idx, item2idx


In [ ]:

# -----------------------------
# ANN retrieval and DL candidate generation
# -----------------------------

@torch.no_grad()
def compute_item_vectors(model: nn.Module, num_items: int, batch_size: int = 65536) -> np.ndarray:
    model.eval()
    vecs = []
    for start in trange(1, num_items, batch_size, desc="Encode items"):
        end = min(num_items, start + batch_size)
        idx = torch.arange(start, end, device=DEVICE, dtype=torch.long)
        v = model.encode_item_idx(idx).detach().float().cpu().numpy()
        vecs.append(v.astype(np.float32))
    return np.vstack(vecs).astype(np.float32)


@torch.no_grad()
def encode_user_batch(model: nn.Module, hists: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    model.eval()
    out = []
    for start in range(0, len(hists), batch_size):
        end = min(len(hists), start + batch_size)
        uidx = torch.zeros(end - start, device=DEVICE, dtype=torch.long)
        hist = torch.from_numpy(hists[start:end].astype(np.int64)).to(DEVICE)
        v = model.encode_user(uidx, hist).detach().float().cpu().numpy()
        out.append(v.astype(np.float32))
    return np.vstack(out).astype(np.float32) if out else np.empty((0, CFG.dssm_embedding_dim), dtype=np.float32)


@dataclass
class ANNIndex:
    backend: str
    index: object | None
    n_items: int
    item_vecs: np.ndarray | None = None

    def query(self, user_vecs: np.ndarray, topk: int) -> tuple[np.ndarray, np.ndarray]:
        topk = int(min(topk, self.n_items))
        user_vecs = np.asarray(user_vecs, dtype=np.float32)
        if self.backend == "hnswlib":
            labels, dist = self.index.knn_query(user_vecs, k=topk)
            return labels.astype(np.int64), (1.0 - dist).astype(np.float32)
        if self.backend == "faiss":
            scores, labels = self.index.search(user_vecs, topk)
            return labels.astype(np.int64), scores.astype(np.float32)
        if self.backend == "torch":
            item_t = torch.from_numpy(self.item_vecs).to(DEVICE).float()
            item_t = F.normalize(item_t, dim=1)
            all_idx, all_scores = [], []
            bs = 512
            for s in range(0, len(user_vecs), bs):
                u = torch.from_numpy(user_vecs[s:s+bs]).to(DEVICE).float()
                u = F.normalize(u, dim=1)
                scores = u @ item_t.T
                vals, idx = torch.topk(scores, k=topk, dim=1)
                all_idx.append(idx.detach().cpu().numpy().astype(np.int64))
                all_scores.append(vals.detach().cpu().numpy().astype(np.float32))
            return np.vstack(all_idx), np.vstack(all_scores)
        raise ValueError(self.backend)


def load_or_build_ann_index(model: nn.Module, item_id_by_idx: np.ndarray) -> ANNIndex:
    n_items = len(item_id_by_idx) - 1
    ann_dir = MODELS_DIR / CFG.source_name / CFG.checkpoint_split
    hnsw_path = ann_dir / f"{CFG.source_name}_{CFG.checkpoint_split}_hnsw.bin"
    if HAS_HNSWLIB and hnsw_path.exists() and not CFG.rebuild_ann_index:
        idx = hnswlib.Index(space="cosine", dim=CFG.dssm_embedding_dim)
        idx.load_index(str(hnsw_path), max_elements=n_items)
        idx.set_ef(max(CFG.hnsw_ef_search, CFG.ann_oversample))
        print(f"HNSW loaded: {rel(hnsw_path)}")
        return ANNIndex("hnswlib", idx, n_items)

    item_vecs = compute_item_vectors(model, len(item_id_by_idx))
    if HAS_HNSWLIB:
        ensure_dir(ann_dir)
        idx = hnswlib.Index(space="cosine", dim=item_vecs.shape[1])
        idx.init_index(max_elements=item_vecs.shape[0], ef_construction=CFG.hnsw_ef_construction, M=CFG.hnsw_m)
        for s in trange(0, item_vecs.shape[0], CFG.hnsw_add_batch_size, desc="Build HNSW"):
            e = min(item_vecs.shape[0], s + CFG.hnsw_add_batch_size)
            idx.add_items(item_vecs[s:e], np.arange(s, e, dtype=np.int64), num_threads=CFG.hnsw_num_threads)
        idx.set_ef(max(CFG.hnsw_ef_search, CFG.ann_oversample))
        idx.save_index(str(hnsw_path))
        print(f"HNSW built: {rel(hnsw_path)}")
        return ANNIndex("hnswlib", idx, item_vecs.shape[0])

    print("hnswlib is unavailable; using torch exact retrieval. This can be slow.")
    return ANNIndex("torch", None, item_vecs.shape[0], item_vecs=item_vecs)


def _safe_model_buffer_numpy(model: nn.Module, name: str, length: int) -> np.ndarray:
    arr = getattr(model, name, None)
    if torch.is_tensor(arr):
        out = arr.detach().cpu().numpy()
        if len(out) == length:
            return out.astype(np.int32)
    return np.zeros(length, dtype=np.int32)


def _history_pref_maps(hist_idx: np.ndarray, value_by_item_idx: np.ndarray) -> dict[int, float]:
    hist_idx = hist_idx[hist_idx > 0]
    if len(hist_idx) == 0:
        return {}
    vals = value_by_item_idx[np.clip(hist_idx, 0, len(value_by_item_idx) - 1)]
    vals = vals[vals > 0]
    if len(vals) == 0:
        return {}
    unique, cnt = np.unique(vals.astype(np.int32), return_counts=True)
    denom = float(cnt.sum())
    return {int(k): float(v) / denom for k, v in zip(unique, cnt)}


def load_item_pop_prior_for_checkpoint_vocab(split_name: str, item_id_by_idx: np.ndarray) -> np.ndarray:
    # Try checkpoint-specific sequence cache first; otherwise compute a cheap zero prior.
    path = PROCESSED_DIR / "sequences" / CFG.checkpoint_split / "item_weight_by_idx.npy"
    if path.exists():
        arr = np.load(path).astype(np.float32)
        pop = np.log1p(arr) if len(arr) == len(item_id_by_idx) else np.zeros(len(item_id_by_idx), dtype=np.float32)
    else:
        pop = np.zeros(len(item_id_by_idx), dtype=np.float32)
    if np.nanmax(pop) > 0:
        q = np.nanquantile(pop[1:], 0.995) if len(pop) > 1 else np.nanmax(pop)
        q = float(q) if np.isfinite(q) and q > 0 else float(np.nanmax(pop))
        pop = np.clip(pop / max(q, 1e-6), 0.0, 1.0)
    pop[0] = 0.0
    return pop.astype(np.float32)


def write_dl_buffer_to_shards(rows: list[tuple], out_dir: Path, source_name: str, part_id: int) -> int:
    if not rows:
        return 0
    df = pd.DataFrame(
        rows,
        columns=["user_id", "item_id", "score", "rank", "raw_dssm_score", "pop_prior", "cat_l0_pref", "catalog_pref"],
    )
    df["source"] = source_name
    df[f"{source_name}_score"] = df["score"].astype("float32")
    df[f"{source_name}_rank"] = df["rank"].astype("int32")
    df[f"{source_name}_raw_score"] = df["raw_dssm_score"].astype("float32")
    df[f"{source_name}_pop_prior"] = df["pop_prior"].astype("float32")
    df[f"{source_name}_cat_l0_pref"] = df["cat_l0_pref"].astype("float32")
    df[f"{source_name}_catalog_pref"] = df["catalog_pref"].astype("float32")
    df[f"in_{source_name}"] = np.int8(1)
    df[f"f_num__candidate__in_{source_name}"] = np.float32(1.0)
    df[f"f_num__candidate__{source_name}_score"] = df["score"].astype("float32")
    df[f"f_num__candidate__{source_name}_rank"] = df["rank"].astype("float32")
    df[f"f_num__candidate__{source_name}_rr"] = (1.0 / df["rank"].astype("float32")).astype("float32")
    df[f"f_num__candidate__{source_name}_raw_score"] = df["raw_dssm_score"].astype("float32")
    df[f"f_num__candidate__{source_name}_pop_prior"] = df["pop_prior"].astype("float32")
    df[f"f_num__candidate__{source_name}_cat_l0_pref"] = df["cat_l0_pref"].astype("float32")
    df[f"f_num__candidate__{source_name}_catalog_pref"] = df["catalog_pref"].astype("float32")

    df["user_shard"] = (df["user_id"].astype("int64") % CFG.n_shards).astype("int16")
    written = 0
    for shard_id, part in df.groupby("user_shard", sort=True):
        shard_dir = out_dir / f"user_shard={int(shard_id)}"
        shard_dir.mkdir(parents=True, exist_ok=True)
        out_file = shard_dir / f"{source_name}_part_{part_id:05d}.parquet"
        pl.from_pandas(part.drop(columns=["user_shard"])).write_parquet(out_file, compression="zstd")
        written += len(part)
    return written


def generate_dl_candidates_for_split(model: nn.Module, ann: ANNIndex, split_name: str, item_id_by_idx: np.ndarray, item2idx: dict[int, int]) -> dict:
    out_dir = CANDIDATES_DIR / CFG.source_name / split_name
    if candidate_dir_looks_complete(out_dir) and not CFG.rebuild_dl_sources:
        print(f"{CFG.source_name}/{split_name}: source exists, reused -> {rel(out_dir)}")
        return {"split": split_name, "status": "reused", "rows": None, "out_dir": rel(out_dir)}

    if out_dir.exists() and (CFG.rebuild_dl_sources or CFG.rebuild_missing_dl_sources):
        shutil.rmtree(out_dir)
    ensure_dir(out_dir)

    target_users, user_seq, seen_by_user = build_sequences_for_split_with_checkpoint_vocab(split_name, item2idx)
    pop_prior = load_item_pop_prior_for_checkpoint_vocab(split_name, item_id_by_idx)
    catalog_arr = _safe_model_buffer_numpy(model, "catalog_idx_by_item_idx", len(item_id_by_idx))
    cat_l0_arr = _safe_model_buffer_numpy(model, "cat_l0_idx_by_item_idx", len(item_id_by_idx))

    rows_buffer: list[tuple] = []
    part_id = 0
    total_rows = 0
    t0 = time.perf_counter()

    for start in trange(0, len(target_users), CFG.retrieval_user_batch_size, desc=f"Generate {CFG.source_name}/{split_name}"):
        users = target_users[start:start + CFG.retrieval_user_batch_size]
        hists = np.zeros((len(users), CFG.sequence_max_len), dtype=np.int64)
        keep = []
        for j, user_id in enumerate(users):
            seq = user_seq.get(int(user_id))
            if seq is None or len(seq) == 0:
                continue
            hist = seq[-CFG.sequence_max_len:]
            hists[j, -len(hist):] = hist.astype(np.int64)
            keep.append(j)
        if not keep:
            continue

        keep = np.asarray(keep, dtype=np.int64)
        batch_users = users[keep].astype(np.int64)
        batch_hists = hists[keep]
        user_vecs = encode_user_batch(model, batch_hists, batch_size=CFG.retrieval_user_batch_size)
        top_rows, top_scores = ann.query(user_vecs, CFG.ann_oversample)

        for local_row, user_id in enumerate(batch_users):
            hist_idx = batch_hists[local_row]
            l0_pref_map = _history_pref_maps(hist_idx, cat_l0_arr)
            catalog_pref_map = _history_pref_maps(hist_idx, catalog_arr)
            seen = seen_by_user.get(int(user_id), set()) if CFG.filter_seen_context_items else set()
            cand = []
            for ann_rank, (ridx, raw_score) in enumerate(zip(top_rows[local_row], top_scores[local_row]), start=1):
                ridx = int(ridx)
                item_idx = ridx + 1
                if item_idx <= 0 or item_idx >= len(item_id_by_idx):
                    continue
                item_id = int(item_id_by_idx[item_idx])
                if CFG.filter_seen_context_items and item_id in seen:
                    continue
                cat_l0_pref = float(l0_pref_map.get(int(cat_l0_arr[item_idx]), 0.0))
                catalog_pref = float(catalog_pref_map.get(int(catalog_arr[item_idx]), 0.0))
                pprior = float(pop_prior[item_idx])
                calibrated = (
                    float(raw_score)
                    + CFG.dssm_candidate_pop_alpha * pprior
                    + CFG.dssm_candidate_cat_l0_alpha * cat_l0_pref
                    + CFG.dssm_candidate_catalog_alpha * catalog_pref
                    - CFG.dssm_candidate_rank_alpha * math.log1p(float(ann_rank))
                )
                cand.append((calibrated, item_id, float(raw_score), pprior, cat_l0_pref, catalog_pref))
            cand.sort(key=lambda x: (-x[0], x[1]))
            for rank, (score, item_id, raw_score, pprior, cat_l0_pref, catalog_pref) in enumerate(cand[:CFG.dl_candidates_per_user], start=1):
                rows_buffer.append((int(user_id), int(item_id), float(score), int(rank), float(raw_score), float(pprior), float(cat_l0_pref), float(catalog_pref)))

        if len(rows_buffer) >= CFG.retrieval_flush_rows:
            total_rows += write_dl_buffer_to_shards(rows_buffer, out_dir, CFG.source_name, part_id)
            rows_buffer.clear()
            part_id += 1
            gc.collect()

    if rows_buffer:
        total_rows += write_dl_buffer_to_shards(rows_buffer, out_dir, CFG.source_name, part_id)
        rows_buffer.clear()

    meta = {
        "source_name": CFG.source_name,
        "split_name": split_name,
        "mode": "inference_only_from_checkpoint",
        "checkpoint_path": rel(get_checkpoint_path()),
        "checkpoint_split": CFG.checkpoint_split,
        "rows": int(total_rows),
        "users_with_history": int(len(target_users)),
        "ann_backend": ann.backend,
        "candidate_inference_time_sec": float(time.perf_counter() - t0),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
        "config": {k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()},
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"{CFG.source_name}/{split_name}: rows={total_rows:,} -> {rel(out_dir)}")
    return {"split": split_name, "status": "built", "rows": int(total_rows), "out_dir": rel(out_dir)}


In [ ]:

# -----------------------------
# Safe merge with base-user filtering
# -----------------------------

def normalize_base_lf(lf: pl.LazyFrame) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    if "candidate_rank" not in cols:
        if "rank" in cols:
            lf = lf.with_columns(pl.col("rank").cast(pl.Int32).alias("candidate_rank"))
        else:
            raise ValueError("Base candidate shard has neither candidate_rank nor rank")
    return lf


def normalize_dl_lf(lf: pl.LazyFrame, source: str) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))
    exprs = []
    if "rank" not in cols and "candidate_rank" in cols:
        exprs.append(pl.col("candidate_rank").cast(pl.Int32).alias("rank"))
    if "score" not in cols and f"{source}_score" in cols:
        exprs.append(pl.col(f"{source}_score").cast(pl.Float32).alias("score"))
    if "source" not in cols:
        exprs.append(pl.lit(source).alias("source"))
    if f"{source}_score" not in cols and "score" in cols:
        exprs.append(pl.col("score").cast(pl.Float32).alias(f"{source}_score"))
    if f"{source}_rank" not in cols and "rank" in cols:
        exprs.append(pl.col("rank").cast(pl.Int32).alias(f"{source}_rank"))
    if f"in_{source}" not in cols:
        exprs.append(pl.lit(1, dtype=pl.Int8).alias(f"in_{source}"))
    if f"f_num__candidate__in_{source}" not in cols:
        exprs.append(pl.lit(1.0, dtype=pl.Float32).alias(f"f_num__candidate__in_{source}"))
    if f"f_num__candidate__{source}_score" not in cols:
        score_expr = pl.col(f"{source}_score") if f"{source}_score" in cols else pl.col("score")
        exprs.append(score_expr.cast(pl.Float32).alias(f"f_num__candidate__{source}_score"))
    if f"f_num__candidate__{source}_rank" not in cols:
        rank_expr = pl.col(f"{source}_rank") if f"{source}_rank" in cols else pl.col("rank")
        exprs.append(rank_expr.cast(pl.Float32).alias(f"f_num__candidate__{source}_rank"))
    if f"f_num__candidate__{source}_rr" not in cols:
        rank_expr = pl.col(f"{source}_rank") if f"{source}_rank" in cols else pl.col("rank")
        exprs.append((1.0 / rank_expr.cast(pl.Float32).clip(lower_bound=1.0)).cast(pl.Float32).alias(f"f_num__candidate__{source}_rr"))
    return lf.with_columns(exprs) if exprs else lf


def aggregate_merged_frame(df: pl.DataFrame, source: str) -> pl.DataFrame:
    agg_exprs = []
    for c in df.columns:
        if c in {"user_id", "item_id"}:
            continue
        if c in {"_base_present", "_dl_present"} or c.startswith("in_") or c.startswith("f_num__candidate__in_"):
            agg_exprs.append(pl.col(c).fill_null(0).max().alias(c))
        elif c in {"_base_rank", "_dl_rank", "candidate_rank", "rank", "min_source_rank"} or c.endswith("_rank"):
            agg_exprs.append(pl.col(c).min().alias(c))
        elif c in {"score", "max_source_score"} or c.endswith("_score") or c.endswith("_rr") or c.endswith("_prior") or c.endswith("_pref"):
            agg_exprs.append(pl.col(c).max().alias(c))
        elif c == "source":
            agg_exprs.append(pl.col(c).drop_nulls().first().alias(c))
        else:
            agg_exprs.append(pl.col(c).drop_nulls().first().alias(c))

    out = df.group_by(["user_id", "item_id"]).agg(agg_exprs)
    flag_cols = [c for c in out.columns if c.startswith("in_")]
    score_cols = [c for c in out.columns if c.endswith("_score") and not c.startswith("f_num__")]
    rank_cols = [c for c in out.columns if c.endswith("_rank") and not c.startswith("f_num__")]
    exprs = []
    if flag_cols:
        exprs.append(pl.sum_horizontal(*[pl.col(c).fill_null(0).cast(pl.Int16) for c in flag_cols]).cast(pl.Int16).alias("sources_count"))
    if score_cols:
        exprs.append(pl.max_horizontal(*[pl.col(c).fill_null(-1e9).cast(pl.Float32) for c in score_cols]).cast(pl.Float32).alias("max_source_score"))
    if rank_cols:
        exprs.append(pl.min_horizontal(*[pl.col(c).fill_null(1_000_000_000).cast(pl.Int32) for c in rank_cols]).cast(pl.Int32).alias("min_source_rank"))
    return out.with_columns(exprs) if exprs else out


def safe_merge_split(split: str) -> dict:
    base_dir = resolve_base_candidate_dir(split)
    dl_dir = resolve_dl_candidate_dir(split)
    if base_dir is None:
        return {"split": split, "status": "skipped_no_base", "rows": 0, "users": 0}
    if dl_dir is None:
        return {"split": split, "status": "skipped_no_dl", "base_dir": rel(base_dir), "rows": 0, "users": 0}

    out_dir = CANDIDATES_DIR / CFG.output_merged_name / split
    if out_dir.exists() and CFG.rebuild_merged_with_dl:
        shutil.rmtree(out_dir)
    ensure_dir(out_dir)

    source = CFG.source_name
    rows_total = 0
    users_total = set()

    for shard_id in trange(CFG.n_shards, desc=f"Safe merge {split}"):
        frames = []
        base_lf = scan_candidate_shard_or_none(base_dir, shard_id)
        if base_lf is None:
            continue
        base_lf = normalize_base_lf(base_lf)
        base = collect_streaming(
            base_lf
            .filter(pl.col("candidate_rank").cast(pl.Int32) <= int(CFG.base_keep_per_user))
            .with_columns([
                pl.lit(1, dtype=pl.Int8).alias("_base_present"),
                pl.col("candidate_rank").fill_null(1_000_000).cast(pl.Int32).alias("_base_rank"),
                pl.lit(0, dtype=pl.Int8).alias("_dl_present"),
                pl.lit(1_000_000, dtype=pl.Int32).alias("_dl_rank"),
            ])
        )
        if base.is_empty():
            continue
        frames.append(base)
        base_users = base.select(pl.col("user_id").cast(pl.Int64).unique()).lazy()

        dl_lf = scan_candidate_shard_or_none(dl_dir, shard_id)
        if dl_lf is not None:
            dl_lf = normalize_dl_lf(dl_lf, source)
            rank_col = "rank" if "rank" in lazy_columns(dl_lf) else "candidate_rank"
            # Critical fix: keep only users from base split/shard. This prevents 880k-user val explosions.
            dl = collect_streaming(
                dl_lf
                .join(base_users, on="user_id", how="inner")
                .filter(pl.col(rank_col).cast(pl.Int32) <= int(CFG.dl_keep_per_user))
                .with_columns([
                    pl.lit(0, dtype=pl.Int8).alias("_base_present"),
                    pl.lit(1_000_000, dtype=pl.Int32).alias("_base_rank"),
                    pl.lit(1, dtype=pl.Int8).alias("_dl_present"),
                    pl.col(rank_col).fill_null(1_000_000).cast(pl.Int32).alias("_dl_rank"),
                ])
            )
            if not dl.is_empty():
                frames.append(dl)

        all_cols = sorted(set().union(*(set(f.columns) for f in frames)))
        aligned = []
        for f in frames:
            missing = [c for c in all_cols if c not in f.columns]
            if missing:
                f = f.with_columns([pl.lit(None).alias(c) for c in missing])
            aligned.append(f.select(all_cols))

        merged = aggregate_merged_frame(pl.concat(aligned, how="vertical_relaxed"), source)
        score_for_sort = "max_source_score" if "max_source_score" in merged.columns else (f"{source}_score" if f"{source}_score" in merged.columns else "score")
        merged = (
            merged
            .with_columns([
                pl.when(pl.col("_base_present").fill_null(0) > 0).then(0).otherwise(1).cast(pl.Int8).alias("_source_priority"),
                pl.col(score_for_sort).fill_null(-1e9).cast(pl.Float32).alias("_tmp_score"),
            ])
            .sort(
                ["user_id", "_source_priority", "_base_rank", "_dl_rank", "_tmp_score", "item_id"],
                descending=[False, False, False, False, True, False],
            )
            .with_columns(pl.arange(1, pl.len() + 1).over("user_id").cast(pl.Int32).alias("candidate_rank"))
            .filter(pl.col("candidate_rank") <= int(CFG.total_keep_per_user))
            .drop(["_source_priority", "_tmp_score"], strict=False)
        )
        shard_out = out_dir / f"user_shard={shard_id}"
        ensure_dir(shard_out)
        merged.write_parquet(shard_out / "part.parquet", compression="zstd")
        rows_total += merged.height
        users_total.update(int(u) for u in merged.select("user_id").unique().to_series().to_list())
        del frames, aligned, merged, base
        gc.collect()

    meta = {
        "split": split,
        "status": "merged",
        "base_dir": rel(base_dir),
        "dl_dir": rel(dl_dir),
        "out_dir": rel(out_dir),
        "rows": int(rows_total),
        "users": int(len(users_total)),
        "base_keep_per_user": int(CFG.base_keep_per_user),
        "dl_keep_per_user": int(CFG.dl_keep_per_user),
        "total_keep_per_user": int(CFG.total_keep_per_user),
        "built_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (out_dir / "_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return meta


In [ ]:

# -----------------------------
# Validation metrics and reports
# -----------------------------

def build_ground_truth(start: date, end: date) -> dict[int, list[int]]:
    df = collect_streaming(
        read_orders_lf("user_item", start, end)
        .filter(pl.col("cnt_orders").fill_null(0) > 0)
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64))
        .unique()
        .sort(["user_id", "item_id"])
    )
    result: dict[int, list[int]] = defaultdict(list)
    for u, i in zip(df["user_id"].to_numpy(), df["item_id"].to_numpy()):
        result[int(u)].append(int(i))
    return dict(result)


def _unique_int_list(values) -> list[int]:
    if values is None:
        return []
    return list(dict.fromkeys(int(x) for x in values))


def hit_rate_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    return float(any(int(x) in gt for x in pred_items[:k])) if gt else np.nan


def recall_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    return float(len(gt & set(int(x) for x in pred_items[:k])) / len(gt)) if gt else np.nan


def ndcg_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            dcg += 1.0 / np.log2(rank + 1)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(len(gt), k) + 1))
    return float(dcg / idcg) if idcg else 0.0


def ap_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    hits = 0
    score = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            hits += 1
            score += hits / rank
    return float(score / len(gt))


def rr_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_unique_int_list(gt_items))
    if not gt:
        return np.nan
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            return float(1.0 / rank)
    return 0.0


def evaluate_candidate_dir(candidates_dir: Path, ground_truth: dict[int, list[int]], *, ks: Iterable[int], rank_col: str, label: str) -> pd.DataFrame:
    if not candidate_dir_looks_complete(candidates_dir):
        return pd.DataFrame()
    ks = tuple(int(k) for k in ks)
    sums = {k: defaultdict(float) for k in ks}
    candidate_counts = []
    users_eval = 0
    unique_items: set[int] = set()
    for shard_id in trange(CFG.n_shards, desc=f"Evaluate {label}"):
        lf = scan_candidate_shard_or_none(candidates_dir, shard_id)
        if lf is None:
            continue
        cols = lazy_columns(lf)
        actual_rank_col = rank_col if rank_col in cols else ("rank" if "rank" in cols else "candidate_rank")
        part = collect_streaming(
            lf.select(
                pl.col("user_id").cast(pl.Int64),
                pl.col("item_id").cast(pl.Int64),
                pl.col(actual_rank_col).cast(pl.Int32).alias("eval_rank"),
            ).sort(["user_id", "eval_rank", "item_id"])
        )
        if part.is_empty():
            continue
        pdf = part.to_pandas()
        for user_id, group in pdf.groupby("user_id", sort=False):
            user_id = int(user_id)
            gt = ground_truth.get(user_id)
            if not gt:
                continue
            pred = group["item_id"].astype(np.int64).tolist()
            users_eval += 1
            candidate_counts.append(len(pred))
            unique_items.update(pred[:max(ks)])
            for k in ks:
                sums[k]["hit_rate"] += hit_rate_at_k(gt, pred, k)
                sums[k]["recall"] += recall_at_k(gt, pred, k)
                sums[k]["ndcg"] += ndcg_at_k(gt, pred, k)
                sums[k]["map"] += ap_at_k(gt, pred, k)
                sums[k]["mrr"] += rr_at_k(gt, pred, k)
        del pdf, part
        gc.collect()
    rows = []
    denom = max(users_eval, 1)
    for k in ks:
        rows.append({
            "model": label,
            "k": k,
            "users_eval": int(users_eval),
            "hit_rate": sums[k]["hit_rate"] / denom,
            "recall": sums[k]["recall"] / denom,
            "ndcg": sums[k]["ndcg"] / denom,
            "map": sums[k]["map"] / denom,
            "mrr": sums[k]["mrr"] / denom,
            "avg_candidates_per_user": float(np.mean(candidate_counts)) if candidate_counts else 0.0,
            "p50_candidates_per_user": float(np.percentile(candidate_counts, 50)) if candidate_counts else 0.0,
            "p95_candidates_per_user": float(np.percentile(candidate_counts, 95)) if candidate_counts else 0.0,
            "unique_items": int(len(unique_items)),
        })
    return pd.DataFrame(rows)


def build_positive_pairs_df(gt: dict[int, list[int]]) -> pl.DataFrame:
    rows = [(int(u), int(i)) for u, items in gt.items() for i in items]
    return pl.DataFrame(rows, schema={"user_id": pl.Int64, "item_id": pl.Int64}, orient="row").unique() if rows else pl.DataFrame({"user_id": [], "item_id": []}, schema={"user_id": pl.Int64, "item_id": pl.Int64})


def collect_found_positive_pairs_at_k(path: Path, gt_pairs: pl.DataFrame, k: int, *, rank_col: str) -> set[tuple[int, int]]:
    found: set[tuple[int, int]] = set()
    if not candidate_dir_looks_complete(path) or gt_pairs.is_empty():
        return found
    gt_lf = gt_pairs.lazy()
    for shard_id in trange(CFG.n_shards, desc=f"positive pairs {path.parent.name}/{path.name}@{k}"):
        lf = scan_candidate_shard_or_none(path, shard_id)
        if lf is None:
            continue
        cols = lazy_columns(lf)
        actual_rank_col = rank_col if rank_col in cols else ("rank" if "rank" in cols else "candidate_rank")
        hits = collect_streaming(
            lf.select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), pl.col(actual_rank_col).cast(pl.Int32).alias("r"))
            .filter(pl.col("r") <= int(k))
            .join(gt_lf, on=["user_id", "item_id"], how="inner")
            .select("user_id", "item_id")
            .unique()
        )
        if not hits.is_empty():
            found.update((int(u), int(i)) for u, i in zip(hits["user_id"].to_numpy(), hits["item_id"].to_numpy()))
    return found


In [ ]:

# -----------------------------
# Run: build missing DL sources, safe merge, metrics
# -----------------------------

# 1) Resolve current state.
state_rows = []
for split in CFG.splits_to_merge:
    base = resolve_base_candidate_dir(split)
    dl = resolve_dl_candidate_dir(split)
    out = CANDIDATES_DIR / CFG.output_merged_name / split
    state_rows.append({
        "split": split,
        "base_dir": rel(base) if base else None,
        "base_exists": bool(base),
        "dl_dir": rel(dl) if dl else None,
        "dl_exists": bool(dl),
        "merged_with_dl_dir": rel(out),
        "merged_with_dl_exists": candidate_dir_looks_complete(out),
    })
state = pd.DataFrame(state_rows)
display(state)
state.to_parquet(REPORTS_DIR / "dl_candidates_inference_initial_state.parquet", index=False)

# 2) Load model/checkpoint once.
model, item_id_by_idx, item2idx = load_dssm_checkpoint()
ann = load_or_build_ann_index(model, item_id_by_idx)

# 3) Generate missing DL candidates for configured splits.
build_rows = []
for split in CFG.splits_to_build_dl:
    out_dir = CANDIDATES_DIR / CFG.source_name / split
    if candidate_dir_looks_complete(out_dir) and not CFG.rebuild_dl_sources:
        print(f"{split}: DL source already exists, skip generation -> {rel(out_dir)}")
        build_rows.append({"split": split, "status": "reused", "out_dir": rel(out_dir)})
        continue
    build_rows.append(generate_dl_candidates_for_split(model, ann, split, item_id_by_idx, item2idx))

build_report = pd.DataFrame(build_rows)
display(build_report)
build_report.to_parquet(REPORTS_DIR / "dl_candidates_inference_build_report.parquet", index=False)
build_report.to_csv(REPORTS_DIR / "dl_candidates_inference_build_report.csv", index=False)
print("saved:", rel(REPORTS_DIR / "dl_candidates_inference_build_report.parquet"))

# 4) Safe merge all splits. Existing val DL source is filtered to base users during merge.
merge_rows = []
for split in CFG.splits_to_merge:
    meta = safe_merge_split(split)
    merge_rows.append(meta)
merge_report = pd.DataFrame(merge_rows)
display(merge_report)
merge_report.to_parquet(REPORTS_DIR / "dl_candidates_merge_report.parquet", index=False)
merge_report.to_csv(REPORTS_DIR / "dl_candidates_merge_report.csv", index=False)
print("saved:", rel(REPORTS_DIR / "dl_candidates_merge_report.parquet"))

# 5) Validation metrics and reports.
if CFG.evaluate_val:
    gt_val = build_ground_truth(CFG.val_target_start, CFG.val_target_end)
    base_val = resolve_base_candidate_dir("val")
    dl_val = resolve_dl_candidate_dir("val")
    merged_val = CANDIDATES_DIR / CFG.output_merged_name / "val"
    frames = []
    if base_val is not None:
        frames.append(evaluate_candidate_dir(base_val, gt_val, ks=CFG.metric_ks, rank_col="candidate_rank", label="base_non_dl"))
    if dl_val is not None:
        frames.append(evaluate_candidate_dir(dl_val, gt_val, ks=CFG.metric_ks, rank_col="rank", label=CFG.source_name))
    if candidate_dir_looks_complete(merged_val):
        frames.append(evaluate_candidate_dir(merged_val, gt_val, ks=CFG.metric_ks, rank_col="candidate_rank", label=CFG.output_merged_name))
    val_metrics = pd.concat([f for f in frames if f is not None and not f.empty], ignore_index=True) if frames else pd.DataFrame()
    display(val_metrics)
    val_metrics.to_parquet(REPORTS_DIR / "dl_candidates_val_metrics.parquet", index=False)
    val_metrics.to_csv(REPORTS_DIR / "dl_candidates_val_metrics.csv", index=False)
    print("saved:", rel(REPORTS_DIR / "dl_candidates_val_metrics.parquet"))

    if CFG.run_unique_positive_report and base_val is not None and dl_val is not None:
        gt_pairs = build_positive_pairs_df(gt_val)
        total_pos = gt_pairs.height
        base_pos = collect_found_positive_pairs_at_k(base_val, gt_pairs, CFG.base_keep_per_user, rank_col="candidate_rank")
        dl_pos = collect_found_positive_pairs_at_k(dl_val, gt_pairs, CFG.dl_keep_per_user, rank_col="rank")
        merged_pos = collect_found_positive_pairs_at_k(merged_val, gt_pairs, CFG.total_keep_per_user, rank_col="candidate_rank") if candidate_dir_looks_complete(merged_val) else set()
        unique_report = pd.DataFrame([{
            "base_k": int(CFG.base_keep_per_user),
            "dl_k": int(CFG.dl_keep_per_user),
            "merged_k": int(CFG.total_keep_per_user),
            "target_positive_pairs": int(total_pos),
            "base_found_pairs": int(len(base_pos)),
            "dssm_found_pairs": int(len(dl_pos)),
            "dssm_unique_vs_base_pairs": int(len(dl_pos - base_pos)),
            "overlap_positive_pairs": int(len(base_pos & dl_pos)),
            "union_found_pairs": int(len(base_pos | dl_pos)),
            "merged_found_pairs": int(len(merged_pos)),
            "base_pair_recall": float(len(base_pos) / total_pos) if total_pos else np.nan,
            "dssm_pair_recall": float(len(dl_pos) / total_pos) if total_pos else np.nan,
            "incremental_pair_recall": float(len(dl_pos - base_pos) / total_pos) if total_pos else np.nan,
            "union_pair_recall": float(len(base_pos | dl_pos) / total_pos) if total_pos else np.nan,
            "merged_pair_recall": float(len(merged_pos) / total_pos) if total_pos else np.nan,
        }])
        display(unique_report)
        unique_report.to_parquet(REPORTS_DIR / "dl_candidates_unique_positives.parquet", index=False)
        unique_report.to_csv(REPORTS_DIR / "dl_candidates_unique_positives.csv", index=False)
        print("saved:", rel(REPORTS_DIR / "dl_candidates_unique_positives.parquet"))

        if CFG.run_quota_ablation:
            quota_rows = []
            for quota in CFG.quota_values:
                dl_q = min(int(quota), int(CFG.dl_keep_per_user))
                dl_pos_q = collect_found_positive_pairs_at_k(dl_val, gt_pairs, dl_q, rank_col="rank")
                union_q = base_pos | dl_pos_q
                quota_rows.append({
                    "base_k": int(CFG.base_keep_per_user),
                    "dssm_quota": int(dl_q),
                    "total_candidate_cap": int(CFG.base_keep_per_user + dl_q),
                    "dssm_found_pairs": int(len(dl_pos_q)),
                    "dssm_unique_vs_base_pairs": int(len(dl_pos_q - base_pos)),
                    "union_found_pairs": int(len(union_q)),
                    "incremental_pair_recall": float(len(dl_pos_q - base_pos) / total_pos) if total_pos else np.nan,
                    "union_pair_recall": float(len(union_q) / total_pos) if total_pos else np.nan,
                })
            quota_ablation = pd.DataFrame(quota_rows)
            display(quota_ablation)
            quota_ablation.to_parquet(REPORTS_DIR / "dl_candidates_quota_ablation.parquet", index=False)
            quota_ablation.to_csv(REPORTS_DIR / "dl_candidates_quota_ablation.csv", index=False)
            print("saved:", rel(REPORTS_DIR / "dl_candidates_quota_ablation.parquet"))

# 6) Manifest.
manifest = {
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "inference_only_from_existing_dssm_checkpoint",
    "checkpoint_path": rel(get_checkpoint_path()),
    "config": {k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()},
    "outputs": {
        split: rel(CANDIDATES_DIR / CFG.output_merged_name / split)
        for split in CFG.splits_to_merge
        if candidate_dir_looks_complete(CANDIDATES_DIR / CFG.output_merged_name / split)
    },
    "dl_sources": {
        split: rel(CANDIDATES_DIR / CFG.source_name / split)
        for split in CFG.splits_to_merge
        if candidate_dir_looks_complete(CANDIDATES_DIR / CFG.source_name / split)
    },
    "reports": {
        "build_report": rel(REPORTS_DIR / "dl_candidates_inference_build_report.parquet"),
        "merge_report": rel(REPORTS_DIR / "dl_candidates_merge_report.parquet"),
        "val_metrics": rel(REPORTS_DIR / "dl_candidates_val_metrics.parquet"),
        "unique_positives": rel(REPORTS_DIR / "dl_candidates_unique_positives.parquet"),
        "quota_ablation": rel(REPORTS_DIR / "dl_candidates_quota_ablation.parquet"),
    },
}
manifest_path = CANDIDATES_DIR / "manifest_dl_candidates_with_dssm_final.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", rel(manifest_path))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
